# 73 · The species comparison by label, on the package units

**What this does.** Notebooks 42, 43 and 49 compared human and mouse PT by S1/S2/S3 label, with notebooks 46, 47,
48 part B, 64 and 44 part B as sensitivities. All of them used notebook 13's structures and reviewed labels. Under
coordinator decision D1 the paper uses one unit set, so this notebook reruns them on the package's units, counts
and automatic labels.
- Their code is copied with only the inputs swapped. Each part is a function of one input bundle (section 11).
- No coordinate is used anywhere.
- Part 42's reference transfer is the first independent test of the package's human labels.

**The frozen protocol.** `docs/results/pt-package-label-comparison.md`, fixed before any fit on the package units.
**Code equivalence.** Before the package run, an untracked scratch script fed the old inputs through these same
functions. They reproduced the published tables of every part.

**Two library arms.**
- **Primary: the mt-encoded genes leave the library and the tested genes** (notebook 69's paper-wide rule).
  Every part runs on it, and the material-change rules apply to it.
- **Secondary: mt reads stay in** (notebook 43's rule, the pure input swap). It runs part 42's transfer and
  spillover, part 43 and part 49's part A.

**Inputs** (read only):
- `results/package_pseudospace/hvm/pt_units.h5ad` (notebook 70's export) and `all_units_positions.csv`;
- the package run folder `runs/hvm` (QC units, stage-01 matrices, Harmony features, fingerprints), and the aki
  export for 44 B;
- the GeoJSONs, the ortholog map, notebook 38's probe and Lake tables, and notebook 62's orthogroups;
- the external atlases (Lake/KPMP, Census, microdissection, rat);
- the old published tables, for the comparison only.

**Outputs** (`results/pt_package_label_comparison/`):
- `tables/{42,43,46,47,48B,49,64,44B}/`, with notebook 42–64's table names, and `tables/mt_in_library/` for the
  secondary arm;
- `tables/43/gene_gradients.csv` and `gene_classes_symmetric.csv`, the exports for notebooks 74 and 77;
- `tables/compare/` (`compare_old.csv`, `material_changes.csv` and the descriptive comparisons);
- `figures/` and `run_record.json`.

**Cohort.** Two male control mice and two cortex sections from one male human donor. Specimens, not units, are the
replicates. Everything is descriptive.

## 1 · Folders, the confirmed labels and the inherited constants

Every constant below is inherited unchanged from the notebook named beside it. The confirmed pass-1 label
fingerprint is the coordinator's sign-off (`docs/results/pt-package-pseudospace.md`, "Label checkpoint").

In [ ]:
import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
output = results_root / 'pt_package_label_comparison'
cache = output / 'stage_cache'
external = data_root / 'external'
upstream13 = results_root / 'minimal_pt_scfates'
ORTHOLOG_FILE = upstream13 / 'ortholog_map_used.csv'
LAKE_FILE = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
PROBE_FILE = results_root / 'pt_literature_deep' / 'probe_counts.csv'                 # notebook 38, per gene pair
LAKE_MEANS_FILE = results_root / 'pt_literature_deep' / 'lake_cortex_segment_means.csv'  # notebook 38, external only
CENSUS_DIR, CENSUS_EXTRA = external / 'census_pt_segments', external / 'census_pt_segments_paralogs'
MOUSE_PANEL = external / 'visium_probe_sets' / 'Visium_Mouse_Transcriptome_Probe_Set_v2.0_mm10-2020-A.csv'
MOLECULE_INFO = [data_root / s / 'outs' / 'molecule_info.h5' for s in ('HUK_COR_1', 'HUK_MED_1')]
UPSTREAM62 = results_root / 'pt_non_one_to_one_orthologs' / 'tables'                   # orthogroups, probe status
GENE_SET_DIR = data_root / 'mouse_vs_human' / 'pathway_gene_sets'
REQUIRED = [ORTHOLOG_FILE, LAKE_FILE, PROBE_FILE, LAKE_MEANS_FILE, MOUSE_PANEL, *MOLECULE_INFO,
            UPSTREAM62 / 'gene_inventory_mouse.csv', UPSTREAM62 / 'gene_inventory_human.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            *[CENSUS_DIR / f'{s}_pt_segment_pseudobulk_{k}' for s in ('mouse', 'human') for k in ('counts.csv.gz', 'meta.csv')],
            *[CENSUS_EXTRA / f'{s}_pt_segment_pseudobulk_counts.csv.gz' for s in ('mouse', 'human')],
            *[GENE_SET_DIR / f'{lib}.json' for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '73.package_labels.1'   # Bump when a cached calculation changes.
# The coordinator's label sign-off (2026-10-12): pass 1 of the human vs mouse run, confirmed as assigned.
CONFIRMED_PASS1_FINGERPRINT = {
    'n_cells': 27395, 'n_clusters': 18, 'membership_sha1': 'efafc9ac2e99',
    'labels_pass1': {'0': 'PT-S1', '1': 'PT-S2', '2': 'PT-S1', '3': 'PT-S1', '4': 'CNT', '5': 'CCD', '6': 'cTAL',
                     '7': 'mTAL', '8': 'mTAL', '9': 'DCT1', '10': 'PT-S3', '11': 'Podocyte', '12': 'mTAL',
                     '13': 'PT-S3', '14': 'Unassigned', '15': 'cTAL', '16': 'SmoothMuscle', '17': 'PT-S1'}}
CONFIRMED_AKI_PASS1 = {'n_cells': 36445, 'n_clusters': 20, 'membership_sha1': 'bb4a80c1b484'}   # 44 B only

expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
# Notebook 42.
CENTROID_TOLERANCE_PX = 2.0
ATTACH_UM, ATTACH_UM_SENSITIVITY = 2.0, 10.0
MIN_DETECTION, MIN_REFERENCE_LEVEL, MIN_REFERENCE_DIFFERENCE = .10, 1.0, 1.0
MARKERS = {'AL': ['Umod', 'Slc12a1'], 'DCT': ['Slc12a3', 'Pvalb'], 'CNT/CD': ['Aqp2', 'Calb1', 'Scnn1g'],
           'podocyte': ['Nphs1', 'Nphs2', 'Podxl'], 'endothelium': ['Pecam1', 'Emcn', 'Kdr'], 'immune': ['Ptprc'],
           'stroma': ['Pdgfrb', 'Col1a1', 'Col3a1'], 'smooth muscle': ['Acta2', 'Myh11']}
MOUSE_BIN_UM = 2.0
# Notebook 43.
SEED, N_BOOT = 43, 1000
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA = .05, .5, .3, .05
INDEX_STRONG, INDEX_PARTIAL = .5, .8
RAT_RELIABILITY, RAT_MAPPED = .5, .7
# Notebook 49.
SEED43, SEED49 = 43, 49
Q_PUBLISHED, Q_GRID = .01, (0., .01, .05)
EDGES = (0., .5, .8, .9, .95, .99, 1.)
MIN_RELIABILITY, CEILING_MIN, CONCENTRATION_MIN, MAX_MISSING = .2, .3, .25, .1
N_COMPONENTS, N_MODULES, N_MODULES_SENSITIVITY = 50, 100, (50, 200)
# Notebooks 46 and 47.
INDEX_MAX, INDEX_UPPER_MAX, SKEW_RATIO = .5, .6, 2.
TRANSPORTER_REGEX = r'^(Slc|Abc|Aqp)'
# Notebook 48 part B (notebook 40's enrichment).
SEED40, N_NULL40, ENRICH_Q = 40, 9999, .10
# Notebook 64.
INDEX_TOLERANCE, COUNT_TOLERANCE, SMALL_COUNT, SMALL_TOLERANCE = .03, .10, 20, 2
PT_DETECTION, PSEUDOCOUNT62 = .02, .05          # notebook 62: PT-expressed rule and the log2 CP10k pseudocount
# Shared.
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}
PARTS = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}
ZONATED_CLASSES = ['conserved', 'reversal', 'mouse-only', 'human-only']
UNION = 'union S2-S1 + S3c-early'
HUMAN, MOUSE, INK, NULL = '#D55E00', '#0072B2', '#333333', '#9A9A9A'

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

from pseudospace.stage_cache import cached_frame, cached_payload, digest

pd.set_option('display.width', 220, 'display.max_columns', 40, 'display.max_colwidth', 80)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
orthologs = pd.read_csv(ORTHOLOG_FILE)
MT_GENES = sorted(orthologs.mouse_symbol[orthologs.mouse_symbol.astype(str).str.startswith('mt-')])   # notebook 69
ARMS = {'primary': 'mt-free library (notebook 69 rule)', 'secondary': 'mt genes in the library (pure input swap)'}
print('Results folder:', output, '· mt genes:', len(MT_GENES))


def folders(out, part):
    """(tables, figures) of one part, created on first use."""
    tables, figures = out / 'tables' / part, out / 'figures' / part
    for folder in (tables, figures):
        folder.mkdir(parents=True, exist_ok=True)
    return tables, figures


def save_figure(fig, figures, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)

## 2 · External references (independent of our units)

- **Lake/KPMP cortex.** It is summed once over every gene, by donor, disease, sex and segment (notebook 64's
  route). It is then mapped to each gene space through the ortholog map and the human gene ids, exactly as notebook
  43 mapped it; notebook 64's guard showed the two routes agree.
- **Census pseudobulks.** The published tables plus notebook 64's paralog extension; human columns use human
  symbols until a pair table maps them.
- **Microdissection and rat.** GSE150338, GSE212213 and GSE56743.
- **Probes.** Notebook 38's probe-balance table (43, 46, 47) and notebook 38's probe counts per pair table (64).

In [ ]:
import h5py

from pseudospace import ortholog_pairing as pairing
from pseudospace.anchor_coordinate import bulk_segment_profiles, census_segment_profiles
from pseudospace.replication_rules import csr_group_sums


def lake_all_genes():
    """Notebook 43's Lake cortex PT grouping (donor | disease | sex | segment), summed over every gene (notebook 64)."""
    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(LAKE_FILE, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': sums,
            'var_ids': var_ids, 'var_names': var_names.astype(str)}


lake_raw = cached_payload('lake_all_genes', lake_all_genes, root=cache,
                          params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                          inputs={'lake': [LAKE_FILE.name, LAKE_FILE.stat().st_size]}, code=[lake_all_genes, csr_group_sums])
lake_meta = pd.DataFrame([g.split('|') for g in lake_raw['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake_raw['nuclei'], lake_raw['totals']
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']

census_tables = {}
for label in ('mouse', 'human'):
    published = pd.read_csv(CENSUS_DIR / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    extra = pd.read_csv(CENSUS_EXTRA / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    if set(extra.index) != set(published.index):
        raise ValueError(f'Census {label}: the extension has different groups.')
    extra = extra.reindex(published.index)
    if label == 'human':  # published human columns carry the paper's mouse symbols; restore human symbols
        published = published.rename(columns=dict(zip(orthologs.mouse_symbol, orthologs.human_symbol)))
    controls = [c for c in extra.columns if c in published.columns]
    if not controls or not all(np.array_equal(extra[c].to_numpy(), published[c].to_numpy()) for c in controls):
        raise ValueError(f'Census {label}: control genes are not reproduced by the extension (guard G7).')
    census_tables[label] = pd.concat([published, extra.drop(columns=controls)], axis=1)
census_meta = {label: pd.read_csv(CENSUS_DIR / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0) for label in ('mouse', 'human')}
micro_table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
micro_table = micro_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_sex_table = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
micro_sex_table = micro_sex_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
rat_table = pd.read_csv(external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz', sep='\t')
rat_table = rat_table.drop_duplicates('Symbol')
rat_table.index = rat_table.Symbol.astype(str).str.upper()
rat_table = rat_table[~rat_table.index.duplicated()]
probe = pd.read_csv(PROBE_FILE).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = set(probe.index[probe.probe_balance.eq('balanced')])
lake_segment_means = pd.read_csv(LAKE_MEANS_FILE, index_col='gene').rename(columns=lambda c: c.replace('lake_log2cpm_', ''))
micro_segment_means = bulk_segment_profiles(micro_table, '_PT{seg}_')
sn_profiles = census_segment_profiles(
    census_tables['mouse'], census_meta['mouse'].loc[census_tables['mouse'].index],
    {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
     'epithelial cell of proximal tubule segment 3': 'S3'})
sn_profiles = sn_profiles.loc[:, sn_profiles.columns.get_level_values('sex') == 'male']
snrna_male_means = pd.DataFrame({s: sn_profiles.xs(s, axis=1, level='segment').mean(axis=1) for s in ('S1', 'S2', 'S3')})

# Probe counts per gene id (notebook 38's definition), for any pair table (notebook 64).
mouse_panel = pd.read_csv(MOUSE_PANEL, comment='#')
mouse_probe_count = mouse_panel[mouse_panel.included.astype(str).str.upper().eq('TRUE')].groupby('gene_id').size()
human_counts_ = []
for path in MOLECULE_INFO:
    with h5py.File(path, 'r') as handle:
        probes = handle['probes']
        table_ = pd.DataFrame({'feature_id': probes['feature_id'][:].astype(str), 'included': probes['included_probes'][:].astype(bool)})
    human_counts_.append(table_[table_.included].groupby('feature_id').size())
pd.testing.assert_series_equal(human_counts_[0], human_counts_[1])
human_probe_count = human_counts_[0]
print(f'Lake cortex donors with ≥ {MIN_NUCLEI} nuclei per segment: {len(lake_donors)} ({len(healthy_lake)} healthy)')


def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def reference_frames(gene_space, pairs, human_ids):
    """Notebook 43's external references (section 4) on one gene space and pair table (notebook 64's route).

    ``human_ids`` maps human symbol -> Ensembl id (our human panel), used for the Lake mapping as in notebook 43.
    """
    per_mouse = {}
    for column in micro_table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    micro_male = pd.DataFrame({key: np.log2(micro_table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
                               for key, cols in per_mouse.items()}).reindex(gene_space)
    micro_sex = {sex: pd.DataFrame({f'S{k}': np.log2(pd.to_numeric(micro_sex_table[f'{prefix}PTS{k}'], errors='coerce').clip(lower=0) + 1)
                                    for k in (1, 2, 3)}).reindex(gene_space)
                 for sex, prefix in (('male', ''), ('female', 'f'))}

    def census_segments(label, segment_map):
        table = census_tables[label]
        if label == 'human':
            table = table.rename(columns=dict(zip(pairs.human_symbol, pairs.mouse_symbol)))
            table = table.loc[:, table.columns.isin(gene_space)]
            if table.columns.duplicated().any():
                raise ValueError('Census human columns collide after renaming to mouse symbols.')
        meta = census_meta[label]
        meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
        meta['segment'] = meta.cell_type.map(segment_map)
        meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
        needed = set(segment_map.values())
        complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
        meta = meta[meta.donor_id.isin(complete[complete].index)]
        counts_out, totals, info = {}, {}, {}
        for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
            for seg_name in needed:
                index = block.index[block.segment.eq(seg_name)]
                counts_out[(donor, seg_name)] = table.loc[index].sum()
                totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
            info[donor] = sex
        return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)

    sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                             'epithelial cell of proximal tubule segment 2': 'S2',
                                                             'epithelial cell of proximal tubule segment 3': 'S3'})
    census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
        'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})
    by_id = dict(zip(pairs.human_symbol.map(human_ids), pairs.mouse_symbol))
    by_symbol = dict(zip(pairs.human_symbol, pairs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake_raw['var_ids'], lake_raw['var_names'])])
    mapped = target != ''
    lake_sums = (pd.DataFrame(lake_raw['sums'][:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
                 .reindex(columns=gene_space, fill_value=0.))
    lake_counts, lake_totals = {}, {}
    for donor in lake_donors:
        for seg_name in ('S1', 'S2', 'S3'):
            rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
            lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
            lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
    upper_to_gene = pd.Series(gene_space, index=gene_space.str.upper())
    upper_to_gene = upper_to_gene[~upper_to_gene.index.duplicated()]
    rat_mapped = rat_table.index.intersection(upper_to_gene.index)
    rat_segments = {}
    for seg_name in ('S1', 'S2', 'S3'):
        columns = [c for c in rat_table.columns if re.fullmatch(rf'{seg_name}_\d+', str(c))]
        values = np.log2(rat_table.loc[rat_mapped, columns].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1)
        values.index = upper_to_gene.loc[rat_mapped].to_numpy()
        rat_segments[seg_name] = values.T.reset_index(drop=True).reindex(columns=gene_space)
    return {'micro_male': micro_male, 'micro_sex': micro_sex, 'sn_counts': sn_counts, 'sn_totals': sn_totals,
            'sn_sex': sn_sex, 'census_h_counts': census_h_counts, 'census_h_totals': census_h_totals,
            'census_h_sex': census_h_sex, 'lake_counts': pd.DataFrame(lake_counts), 'lake_totals': pd.Series(lake_totals),
            'rat_segments': rat_segments, 'rat_mapped_fraction': len(rat_mapped) / len(gene_space)}


def donor_contrasts(count_frame, totals, donors, contrast):
    """Per-donor log2 CPM contrast and per-segment level (notebook 43); early pools S1 + S2 counts."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def segment_regions(specimen, segment, depth):
    """Notebook 43's cortical-like mouse S3: depth ≤ the 95th percentile of the same specimen's S1 depth."""
    cortical_s3 = np.zeros(len(specimen), bool)
    for name in expected['mouse']:
        limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
        cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
    return np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment), cortical_s3

## 3 · Notebook 42: are the S1/S2/S3 labels anatomical in both species?

Notebook 42's code, with only its inputs swapped. `part42` takes one input bundle (section 9 builds it).
- **Geometry.** Every unit polygon is joined through `feature_index`, and its centroid must lie within 2 px.
  Glomeruli are glomerulus-sized when their median area is ≥ 3 × the median PT area.
- **(a) Distance and (b) attachment.** The distance runs from each PT polygon to the nearest morphological
  glomerulus; expression glomeruli are the sensitivity. A PT polygon within 2 µm (10 µm as sensitivity) is
  attached.
- **(c) Reference transfer.** This is the first independent test of the package's human labels.
  - Genes: measured in both species, detected in ≥ 10% of the species' PT units, a reference maximum ≥ 1 log2, and
    a largest segment difference ≥ 1 log2.
  - Primary score: Spearman argmax. Sensitivity: the centred score.
  - Reading: the labels are supported if human agreement is ≥ 0.70 and within 0.10 of the mouse microdissection
    agreement.
- **D7.** Non-PT marker spillover, and human polygons without bins.

Geometry, attachment and registration do not depend on the library, so only the primary bundle runs them. The
transfer and the spillover run in both arms.

In [ ]:
from scipy.stats import rankdata, spearmanr
from shapely import MultiPoint, STRtree
from shapely.geometry import shape
from sklearn.metrics import cohen_kappa_score


def part42(I, out, *, geometry=True, tag='42'):
    """Notebook 42 sections 2-7 on one input bundle; returns its tables."""
    tables, figures = folders(out, tag)
    cohort, SEGMENTATION, GLOMERULUS_GEOMETRY = I['cohort'], I['segmentation'], I['glomerulus_geometry']
    pt_obs = I['obs']
    result = {}
    if geometry:
        geometry_, join_rows, um_per_px = {}, [], {}
        for name, file in SEGMENTATION.items():
            features = json.loads((data_root / file).read_text())['features']
            polys = [shape(f['geometry']) for f in features]
            ratio = [np.sqrt(next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area um^2')
                             / next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area px')) for f in features]
            um_per_px[name] = float(np.median(ratio))
            rows = cohort[cohort['sample'].eq(name)]
            index = rows.feature_index.to_numpy(int)
            if index.max() >= len(polys):
                raise ValueError(f'{name}: feature_index exceeds the segmentation.')
            centroids = np.array([polys[i].centroid.coords[0] for i in index])
            offset = np.hypot(*(centroids - rows[['x_centroid', 'y_centroid']].to_numpy(float)).T)
            join_rows.append({'specimen': name, 'structures': len(rows), 'polygons': len(polys), 'um per px': um_per_px[name],
                              'max centroid offset px': float(offset.max()), 'join safe': bool(offset.max() <= CENTROID_TOLERANCE_PX)})
            geometry_[name] = pd.Series([polys[i] for i in index], index=rows.index)
        JOIN = pd.DataFrame(join_rows)
        JOIN.to_csv(tables / 'geometry_join_check.csv', index=False)
        display(JOIN)
        if not JOIN['join safe'].all():
            raise ValueError('Polygon join failed the centroid check (guard G4); geometry-based checks would be unsafe.')

        glomeruli, glom_rows = {}, []
        for name, file in GLOMERULUS_GEOMETRY.items():
            features = json.loads((data_root / file).read_text())['features']
            polys = [shape(f['geometry']) for f in features if f['properties'].get('classification', {}).get('name') == 'glomerulus']
            if file != SEGMENTATION[name]:
                from scipy.spatial import cKDTree
                frame = np.array([p.centroid.coords[0] for p in (shape(f['geometry']) for f in features)])
                fine = np.array([g.centroid.coords[0] for g in geometry_[name].iloc[:2000]])
                frame_offset = float(np.median(cKDTree(frame).query(fine)[0]))
            else:
                frame_offset = 0.0
            glomeruli[name] = polys
            pt_area = np.median([geometry_[name][i].area for i in cohort.index[cohort['sample'].eq(name) & cohort.label.str.startswith('PT-')]])
            glom_rows.append({'specimen': name, 'morphological glomeruli': len(polys),
                              'median glomerulus area um2': float(np.median([p.area for p in polys]) * um_per_px[name] ** 2),
                              'median PT area um2': float(pt_area * um_per_px[name] ** 2),
                              'area ratio': float(np.median([p.area for p in polys]) / pt_area),
                              'expression Glomerulus structures': int((cohort['sample'].eq(name) & cohort.label.eq('Glomerulus')).sum()),
                              'frame offset px (median)': frame_offset})
        GLOM = pd.DataFrame(glom_rows)
        GLOM['glomerulus-sized (ratio ≥ 3)'] = GLOM['area ratio'] >= 3
        GLOM.to_csv(tables / 'glomerulus_polygons.csv', index=False)
        display(GLOM.round(2))

        # (a) Distance to the nearest glomerulus and (b) glomerulus-attached PT.
        pt = cohort.loc[pt_obs.index].copy()
        pt['species'] = pt.comparison_species.astype(str)
        pt['segment'] = pt.label.str.replace('PT-', '', regex=False)
        for source in ('morphological', 'expression'):
            distance = np.full(len(pt), np.nan)
            for name in pt['sample'].unique():
                rows = np.flatnonzero(pt['sample'].eq(name).to_numpy())
                if source == 'morphological':
                    gpolys = glomeruli[name]
                else:
                    gpolys = list(geometry_[name][cohort.index[cohort['sample'].eq(name) & cohort.label.eq('Glomerulus')]])
                tree = STRtree(gpolys)
                polys = list(geometry_[name].loc[pt.index[rows]])
                _, d = tree.query_nearest(polys, return_distance=True, all_matches=False)
                distance[rows] = d * um_per_px[name]
            pt[f'glomerulus_distance_um_{source}'] = distance
        DIST_ROWS = []
        for source in ('morphological', 'expression'):
            col = f'glomerulus_distance_um_{source}'
            for (sp, seg), block in pt.groupby(['species', 'segment']):
                DIST_ROWS.append({'glomeruli': source, 'species': sp, 'segment': seg, 'n': len(block),
                                  'median um': float(block[col].median()), 'q25': float(block[col].quantile(.25)),
                                  'q75': float(block[col].quantile(.75))})
            for sp in expected:
                block = pt[pt.species.eq(sp)]
                DIST_ROWS.append({'glomeruli': source, 'species': sp, 'segment': 'Spearman(label order, distance)',
                                  'n': len(block), 'median um': spearmanr(block.segment.map({'S1': 0, 'S2': 1, 'S3': 2}), block[col]).statistic})
        DISTANCE = pd.DataFrame(DIST_ROWS)
        DISTANCE.to_csv(tables / 'glomerulus_distance_by_label.csv', index=False)
        display(DISTANCE.round(3))
        ATTACH_ROWS = []
        for source in ('morphological', 'expression'):
            col = f'glomerulus_distance_um_{source}'
            for threshold in (ATTACH_UM, ATTACH_UM_SENSITIVITY):
                for group, block in [(sp, pt[pt.species.eq(sp)]) for sp in expected] + [(n, pt[pt['sample'].eq(n)]) for n in names]:
                    attached = block[col] <= threshold
                    s1 = block.segment.eq('S1')
                    a, b_, c, d = (attached & s1).sum(), (attached & ~s1).sum(), (~attached & s1).sum(), (~attached & ~s1).sum()
                    ATTACH_ROWS.append({'glomeruli': source, 'threshold um': threshold, 'group': group, 'attached PT': int(attached.sum()),
                                        'S1 among attached': float(a / max(attached.sum(), 1)), 'S1 among all PT': float(s1.mean()),
                                        'S2 among attached': float((attached & block.segment.eq('S2')).sum() / max(attached.sum(), 1)),
                                        'S3 among attached': float((attached & block.segment.eq('S3')).sum() / max(attached.sum(), 1)),
                                        'odds ratio S1': float((a + .5) * (d + .5) / ((b_ + .5) * (c + .5)))})
        ATTACHED = pd.DataFrame(ATTACH_ROWS)
        ATTACHED.to_csv(tables / 'glomerulus_attached_pt.csv', index=False)
        display(ATTACHED[ATTACHED.glomeruli.eq('morphological')].round(3))
        result.update(JOIN=JOIN, GLOM=GLOM, DISTANCE=DISTANCE, ATTACHED=ATTACHED, pt_distances=pt)

    # (c) Expression-only reference transfer, species by species.
    X = sparse.csr_matrix(I['lognorm'], dtype=float)
    genes_all = pd.Index(I['gene_space'])
    measured_both = np.ones(len(genes_all), bool)
    species_pt = pt_obs.species.astype(str).to_numpy()
    specimen_pt = pt_obs['sample'].astype(str).to_numpy()
    detected = {sp: np.asarray((X[species_pt == sp] > 0).mean(axis=0)).ravel() for sp in expected}
    lake = lake_segment_means.reindex(genes_all)
    micro = micro_segment_means.reindex(genes_all)[['S1', 'S2', 'S3']]
    snrna = snrna_male_means.reindex(genes_all)
    REFERENCES = {'human': {'Lake/KPMP cortex snRNA': lake}, 'mouse': {'microdissection (primary)': micro, 'male snRNA (Census)': snrna}}

    def reference_genes(reference, sp):
        diffs = pd.concat([(reference[a] - reference[b]).abs() for a, b in (('S2', 'S1'), ('S3', 'S2'), ('S3', 'S1'))], axis=1).max(axis=1)
        keep = (measured_both & (detected[sp] >= MIN_DETECTION) & reference.notna().all(axis=1).to_numpy()
                & (reference.max(axis=1) >= MIN_REFERENCE_LEVEL).to_numpy() & (diffs >= MIN_REFERENCE_DIFFERENCE).to_numpy())
        return np.flatnonzero(keep)

    def transfer(rows, reference, cols):
        """Primary (Spearman, uncentred) and sensitivity (centred Pearson) segment assignments."""
        dense = X[rows][:, cols].toarray()
        ref = reference.iloc[cols][['S1', 'S2', 'S3']].to_numpy(float)
        ranks = rankdata(dense, axis=1)
        ranks = (ranks - ranks.mean(axis=1, keepdims=True))
        ranks /= np.linalg.norm(ranks, axis=1, keepdims=True).clip(1e-12)
        ref_ranks = rankdata(ref, axis=0)
        ref_ranks = (ref_ranks - ref_ranks.mean(axis=0)) / np.linalg.norm(ref_ranks - ref_ranks.mean(axis=0), axis=0)
        primary = ranks @ ref_ranks
        centred = dense.copy()
        for n in np.unique(specimen_pt[rows]):
            m = specimen_pt[rows] == n
            centred[m] -= centred[m].mean(axis=0)
        centred /= np.linalg.norm(centred, axis=1, keepdims=True).clip(1e-12)
        ref_c = ref - ref.mean(axis=1, keepdims=True)
        ref_c /= np.linalg.norm(ref_c, axis=0, keepdims=True).clip(1e-12)
        sensitivity = centred @ ref_c
        order = np.sort(primary, axis=1)
        return (np.array(['S1', 'S2', 'S3'])[primary.argmax(axis=1)], order[:, -1] - order[:, -2],
                np.array(['S1', 'S2', 'S3'])[sensitivity.argmax(axis=1)])

    current = pt_obs.segment.astype(str).to_numpy()
    labels_table = pd.DataFrame({'structure_id': pt_obs.index, 'specimen': specimen_pt, 'species': species_pt,
                                 'current_label': current})
    transfer_rows, fraction_rows = [], []
    for sp, refs in REFERENCES.items():
        rows = np.flatnonzero(species_pt == sp)
        for ref_name, reference in refs.items():
            cols = reference_genes(reference, sp)
            primary, margin, centred = transfer(rows, reference, cols)
            tag = ref_name.split(' ')[0].split('/')[0].lower()
            labels_table.loc[rows, f'transfer_{tag}'] = primary
            labels_table.loc[rows, f'transfer_{tag}_margin'] = margin
            labels_table.loc[rows, f'transfer_{tag}_centred'] = centred
            for score, assigned in (('primary (Spearman)', primary), ('sensitivity (centred)', centred)):
                transfer_rows.append({'species': sp, 'reference': ref_name, 'score': score, 'genes': len(cols),
                                      'agreement with current': float((assigned == current[rows]).mean()),
                                      'Cohen kappa': float(cohen_kappa_score(current[rows], assigned))})
                confusion = pd.crosstab(pd.Series(current[rows], name='current'), pd.Series(assigned, name='transfer'))
                confusion.to_csv(tables / f'transfer_confusion_{sp}_{tag}_{score.split(" ")[0]}.csv')
                for n in expected[sp]:
                    m = specimen_pt[rows] == n
                    fraction_rows.append({'species': sp, 'specimen': n, 'reference': ref_name, 'score': score,
                                          **{f'{s} fraction (transfer)': float((assigned[m] == s).mean()) for s in ('S1', 'S2', 'S3')},
                                          **{f'{s} fraction (current)': float((current[rows][m] == s).mean()) for s in ('S1', 'S2', 'S3')}})
    TRANSFER = pd.DataFrame(transfer_rows)
    FRACTIONS = pd.DataFrame(fraction_rows)
    TRANSFER.to_csv(tables / 'transfer_agreement.csv', index=False)
    FRACTIONS.to_csv(tables / 'segment_fractions.csv', index=False)
    display(TRANSFER.round(3))
    display(FRACTIONS.round(3))
    for sp, refs in REFERENCES.items():
        tag = list(refs)[0].split(' ')[0].split('/')[0].lower()
        display(pd.read_csv(tables / f'transfer_confusion_{sp}_{tag}_primary.csv', index_col=0).rename_axis(f'{sp}: current ↓ / transfer →'))
    human_primary = TRANSFER[(TRANSFER.species == 'human') & TRANSFER.score.str.startswith('primary')]['agreement with current'].iloc[0]
    mouse_primary = TRANSFER[(TRANSFER.species == 'mouse') & TRANSFER.score.str.startswith('primary')
                             & TRANSFER.reference.str.startswith('micro')]['agreement with current'].iloc[0]
    SUPPORT = bool(human_primary >= .70 and abs(human_primary - mouse_primary) <= .10)
    print(f'Pre-specified reading: human agreement {human_primary:.3f}, mouse {mouse_primary:.3f} -> '
          f'{"supports" if SUPPORT else "weak support for"} the current human labels')
    if geometry:
        pt_dist = result['pt_distances'][[c for c in result['pt_distances'].columns if c.startswith('glomerulus_distance')]]
        labels_table = pd.concat([labels_table, pt_dist.loc[labels_table.structure_id].reset_index(drop=True)], axis=1)
        labels_table['glomerulus_attached_2um'] = labels_table.glomerulus_distance_um_morphological <= ATTACH_UM
    labels_table.to_csv(tables / 'alternative_labels.csv', index=False)

    # D7 · Spillover from neighbouring cell types.
    C = sparse.csr_matrix(I['cohort_counts'], dtype=float)
    cohort_genes = pd.Index(I['cohort_genes'])
    library = np.asarray(I['cohort_library'], float)
    all_markers = [g for genes in MARKERS.values() for g in genes]
    present = [g for g in all_markers if g in set(cohort_genes[I['cohort_measured']])]
    nonpt = ~cohort.label.str.startswith('PT-').to_numpy()
    idx = cohort_genes.get_indexer(present)
    detection_nonpt = np.asarray((C[nonpt][:, idx] > 0).mean(axis=0)).ravel()
    used = [g for g, d in zip(present, detection_nonpt) if d >= .01]
    print('Markers used:', used, '| dropped:', sorted(set(all_markers) - set(used)))
    spill = {}
    for category, genes in MARKERS.items():
        g = [x for x in genes if x in used]
        spill[category] = (np.asarray(C[:, cohort_genes.get_indexer(g)].sum(axis=1)).ravel() / library) if g else np.zeros(len(library))
    spill = pd.DataFrame(spill, index=cohort.index)
    spill['total'] = spill.sum(axis=1)
    frame = spill.join(cohort[['sample', 'comparison_species', 'label']])
    SPILL = (frame.groupby(['comparison_species', 'label'])
             .agg(n=('total', 'size'), median=('total', 'median'), mean=('total', 'mean'),
                  **{f'mean {c}': (c, 'mean') for c in MARKERS}).reset_index())
    SPILL.to_csv(tables / 'spillover_index.csv', index=False)
    SPILL_SPECIMEN = frame[frame.label.str.startswith('PT-')].groupby(['sample', 'label']).total.agg(['median', 'mean']).reset_index()
    SPILL_SPECIMEN.to_csv(tables / 'spillover_index_by_specimen.csv', index=False)
    display(SPILL[SPILL.label.str.startswith('PT-')].round(4))
    display(SPILL[~SPILL.label.str.startswith('PT-')][['comparison_species', 'label', 'n', 'median', 'mean']].round(4))
    result.update(TRANSFER=TRANSFER, FRACTIONS=FRACTIONS, SUPPORT=SUPPORT, labels_table=labels_table, SPILL=SPILL,
                  SPILL_SPECIMEN=SPILL_SPECIMEN)

    if geometry:
        # D7 · Polygons without bins (registration QC).
        qc_rows = []
        for name, file in SEGMENTATION.items():
            features = json.loads((data_root / file).read_text())['features']
            obs = I['registration'][name]
            area_um2 = np.array([next(m['value'] for m in f['properties']['measurements'] if m['name'] == 'Area um^2') for f in features])
            polys = [shape(f['geometry']) for f in features]
            spots = np.zeros(len(features))
            spots[obs.feature_index.to_numpy(int)] = obs.n_spots.to_numpy(float)
            covered = spots > 0
            hull = MultiPoint([p.centroid for p, c in zip(polys, covered) if c]).convex_hull
            inside = np.array([hull.contains(p.centroid) for p in polys])
            coverage = spots[covered] * MOUSE_BIN_UM ** 2 / area_um2[covered]
            quintile = pd.qcut(area_um2, 5, labels=False)
            in_cohort = cohort['sample'].eq(name).sum()
            qc_rows.append({'specimen': name, 'polygons': len(features), 'with zero bins': int((~covered).sum()),
                            'zero-bin fraction': float((~covered).mean()),
                            'zero-bin polygons inside bin-covered hull': int((~covered & inside).sum()),
                            'zero-bin fraction inside hull': float((~covered & inside).sum() / max(inside.sum(), 1)),
                            'in notebook 13 cohort': int(in_cohort),
                            'median coverage (bins x 4 um2 / area)': float(np.median(coverage)),
                            'coverage IQR': f'{np.percentile(coverage, 25):.2f}–{np.percentile(coverage, 75):.2f}',
                            **{f'zero-bin fraction, area quintile {q + 1}': float((~covered[quintile == q]).mean()) for q in range(5)}})
        QC = pd.DataFrame(qc_rows).rename(columns={'in notebook 13 cohort': 'in the cohort (QC units)'})
        QC.to_csv(tables / 'registration_qc.csv', index=False)
        display(QC.T)

        fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout='constrained')
        DISTANCE, ATTACHED = result['DISTANCE'], result['ATTACHED']
        morph = DISTANCE[DISTANCE.glomeruli.eq('morphological') & DISTANCE.segment.isin(['S1', 'S2', 'S3'])]
        for i, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
            block = morph[morph.species.eq(sp)]
            x = np.arange(3) + (i - .5) * .36
            axes[0].bar(x, block['median um'], width=.34, color=color, label=sp)
            axes[0].errorbar(x, block['median um'], yerr=[block['median um'] - block.q25, block.q75 - block['median um']],
                             fmt='none', ecolor=INK, lw=1)
        axes[0].set(xticks=range(3), xticklabels=['S1', 'S2', 'S3'], ylabel='distance to nearest glomerulus (µm)',
                    title='(a) Distance to glomerulus by label')
        axes[0].legend(frameon=False)
        att = ATTACHED[ATTACHED.glomeruli.eq('morphological') & ATTACHED['threshold um'].eq(ATTACH_UM)
                       & ATTACHED.group.isin(list(expected))]
        x = np.arange(2)
        axes[1].bar(x - .18, att['S1 among all PT'], width=.34, color='#BBBBBB', label='all PT')
        axes[1].bar(x + .18, att['S1 among attached'], width=.34, color=[MOUSE if g == 'mouse' else HUMAN for g in att.group],
                    label='touching a glomerulus')
        axes[1].set(xticks=x, xticklabels=list(att.group), ylabel='fraction labelled S1', ylim=(0, 1),
                    title='(b) S1 among glomerulus-attached PT')
        axes[1].legend(frameon=False)
        prim = FRACTIONS[FRACTIONS.score.str.startswith('primary') & ~FRACTIONS.reference.str.startswith('male')]
        axes[2].bar(np.arange(len(prim)) - .18, prim['S1 fraction (current)'], width=.34, color='#BBBBBB', label='package labels')
        axes[2].bar(np.arange(len(prim)) + .18, prim['S1 fraction (transfer)'], width=.34,
                    color=[MOUSE if s == 'mouse' else HUMAN for s in prim.species], label='reference transfer')
        axes[2].set(xticks=range(len(prim)), xticklabels=list(prim.specimen), ylabel='S1 fraction', ylim=(0, 1),
                    title='(c) S1 fraction: package vs transferred')
        axes[2].legend(frameon=False)
        save_figure(fig, figures, 'label_validation')
        plt.show()
        plt.close(fig)
        result['QC'] = QC
    return result

## 4 · Notebook 43: reliability ceilings, conservation index and symmetric classes

Notebook 43's sections 2–6 and 8–9, with only the inputs swapped. Part 3 (M7, the robust-pathway categories) rests
on the retired 22-pathway list and is not carried over.
- **Gradients.** Pseudobulk log2 ratios of summed counts per specimen and segment; early is S1 + S2. The mouse S3c
  keeps the cortical-like S3: depth to the three nearest glomerulus units ≤ the 95th percentile of the specimen's
  S1 depth.
- **D2 · Correlations and index.** Noise-corrected correlations against every external dataset, and the
  conservation index with 1,000 gene bootstraps.
  - Wording: an upper bound < 0.5 in S2 − S1 and S3 − early means "largely different genes"; a lower bound ≥ 0.8
    means "largely conserved".
  - Rat is used only if its reliability is ≥ 0.5 and ≥ 70% of genes map.
- **D3 · Classes.** Eight settings; the symmetric setting (A3b + A8) is primary. Counts use probe-balanced genes.
  The asymmetry rule is a mouse-only ÷ human-only ratio ≥ 2.

In [ ]:
from pseudospace.zonation_amplitude import group_sums, log_ratio
from pseudospace.zonation_classes import confirm, cross_species_class, moderated_contrast, union_class, zonation_calls
from pseudospace.zonation_reliability import (conservation_index, confirm_flat, disattenuated, paired_gradient,
                                              unpaired_gradient)


def part43(I, out, *, panel_genes=(), tag='43'):
    """Notebook 43 sections 2-6 and 8-9 on one input bundle; returns every object later parts read."""
    tables, figures = folders(out, tag)
    gene_space, counts, lognorm, library = pd.Index(I['gene_space']), I['counts'], I['lognorm'], np.asarray(I['library'])
    specimen = I['obs']['sample'].astype(str).to_numpy()
    segment = I['obs'].segment.astype(str).to_numpy()
    depth = np.asarray(I['depth'], float)
    unit_ids = pd.Index(I['obs'].index)
    display(pd.crosstab(specimen, segment))
    structure_depth = pd.DataFrame({'specimen': specimen, 'library': library}).groupby('specimen').library.describe()[['count', '50%', 'mean']]
    display(structure_depth.rename(columns={'50%': 'median counts per structure'}))
    segment_region, cortical_s3 = segment_regions(specimen, segment, depth)
    print('Cortical-like mouse S3 structures:', {n: int((cortical_s3 & (specimen == n)).sum()) for n in expected['mouse']})

    def specimen_pseudobulk(count_matrix, specimen_labels, segment_labels, genes):
        """Summed counts per specimen and segment (S1, S2, S3, cortical-S3 and early = S1 + S2)."""
        names_, sums = group_sums(count_matrix, np.char.add(np.char.add(specimen_labels.astype(str), '|'), segment_labels.astype(str)))
        frame = pd.DataFrame(sums, index=names_, columns=genes)
        out_ = {}
        for name in np.unique(specimen_labels):
            rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
            out_[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
            out_[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
            out_[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
            out_[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
            out_[(name, 'early')] = out_[(name, 'S1')] + out_[(name, 'S2')]
        return pd.DataFrame(out_)

    ours_counts = specimen_pseudobulk(counts, specimen, segment_region, gene_space)
    ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(segment_region, PARTS[key[1]])].sum())
                              for key in ours_counts.columns})

    def gradients(pb, pb_library, specimens):
        out_ = {}
        for contrast, (late, early) in CONTRASTS.items():
            for name in specimens:
                out_[(contrast, name)] = log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
        return pd.DataFrame(out_, index=pb.index)

    ours = gradients(ours_counts, ours_library, names)
    log1p_means = {}
    for name in names:
        for label, parts in PARTS.items():
            rows = (specimen == name) & np.isin(segment_region, parts)
            log1p_means[(name, label)] = np.asarray(lognorm[rows].mean(axis=0)).ravel()
    log1p_means = pd.DataFrame(log1p_means, index=gene_space)
    ours_log1p = pd.DataFrame({(contrast, name): log1p_means[(name, late)] - log1p_means[(name, early)]
                               for contrast, (late, early) in CONTRASTS.items() for name in names})

    def count_filter(pb, specimens, parts):
        return pb[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)

    our_genes = {contrast: gene_space[count_filter(ours_counts, names, list(parts)).to_numpy()]
                 for contrast, parts in CONTRASTS.items()}
    print({c: len(g) for c, g in our_genes.items()}, 'genes pass the count filter (cross-species)')

    # Section 4 · Independent references.
    ref = reference_frames(gene_space, orthologs, I['human_gene_ids'])
    micro_male, micro_sex = ref['micro_male'], ref['micro_sex']
    sn_counts, sn_totals, sn_sex = ref['sn_counts'], ref['sn_totals'], ref['sn_sex']
    census_h_counts, census_h_totals, census_h_sex = ref['census_h_counts'], ref['census_h_totals'], ref['census_h_sex']
    lake_counts, lake_totals = ref['lake_counts'], ref['lake_totals']
    rat_segments, rat_mapped_fraction = ref['rat_segments'], ref['rat_mapped_fraction']
    print(f'Lake cortex donors with ≥ {MIN_NUCLEI} nuclei in each segment: {len(lake_donors)} '
          f'({lake_disease.loc[lake_donors].value_counts().to_dict()}); mouse snRNA donors: {sn_sex.value_counts().to_dict()}')

    def reference_from_micro_male(contrast):
        late, early = CONTRASTS[contrast]
        late = 'S3' if late == 'S3c' else late
        parts_early = ['S1', 'S2'] if early == 'early' else [early]
        by_segment = {s: micro_male[[c for c in micro_male.columns if c[0] == s]] for s in ('S1', 'S2', 'S3')}

        def gradient(rng):
            means = {}
            for s, frame in by_segment.items():
                pick = np.arange(frame.shape[1]) if rng is None else rng.integers(0, frame.shape[1], frame.shape[1])
                means[s] = frame.iloc[:, pick].mean(axis=1)
            return means[late] - sum(means[p] for p in parts_early) / len(parts_early), means

        observed, means = gradient(None)
        expressed = pd.concat([means[late], *[means[p] for p in parts_early]], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
        error_variance = float(sum((by_segment[s].var(axis=1, ddof=1) / by_segment[s].shape[1] * (1 if s == late else 1 / len(parts_early) ** 2))
                                   for s in {late, *parts_early})[expressed].mean())
        return {'gradient': observed[expressed], 'draw': lambda rng: gradient(rng)[0][expressed],
                'error_variance': error_variance, 'n_donors': int(min(f.shape[1] for f in by_segment.values()))}

    def reference_from_micro_sex(sex, contrast):
        late, early = CONTRASTS[contrast]
        late = 'S3' if late == 'S3c' else late
        table = micro_sex[sex]
        early_value = table[['S1', 'S2']].mean(axis=1) if early == 'early' else table[early]
        gradient = table[late] - early_value
        expressed = pd.concat([table[late], early_value], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
        return {'gradient': gradient[expressed], 'draw': lambda rng: gradient[expressed], 'error_variance': np.nan, 'n_donors': 0}

    # Section 5 · Part 1 (D2): gradients with sampling noise for every dataset.
    def ours_gradient(contrast, specimens):
        late, early = CONTRASTS[contrast]
        expressed = set(gene_space[count_filter(ours_counts, specimens, [late, early]).to_numpy()])
        return paired_gradient(ours[[(contrast, n) for n in specimens]].T.reset_index(drop=True)), expressed

    def donor_gradient(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
        return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])

    def census_human_gradient():
        donors = census_h_sex.index
        late = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}).T
        early = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}).T
        keep = np.maximum(late.mean(), early.mean()).ge(REFERENCE_LEVEL) & (late - early).notna().all()
        return paired_gradient((late - early).loc[:, keep]), set(keep.index[keep.to_numpy()])

    def segment_replicates_gradient(by_segment, contrast):
        """Unpaired replicate gradient from per-segment replicate frames (rows = replicates)."""
        late, early = CONTRASTS[contrast]
        late = 'S3' if late == 'S3c' else late
        if early == 'early':
            parts = [by_segment['S1'], by_segment['S2']]
            mean = by_segment[late].mean() - (parts[0].mean() + parts[1].mean()) / 2
            noise = by_segment[late].var(ddof=1) / len(by_segment[late]) + sum(p.var(ddof=1) / len(p) for p in parts) / 4
            level = pd.concat([by_segment[late].mean(), (parts[0].mean() + parts[1].mean()) / 2], axis=1).max(axis=1)
        else:
            mean, noise = unpaired_gradient(by_segment[late], by_segment[early])
            level = pd.concat([by_segment[late].mean(), by_segment[early].mean()], axis=1).max(axis=1)
        keep = level.ge(REFERENCE_LEVEL) & mean.notna()
        return (mean, noise), set(keep.index[keep.to_numpy()])

    micro_male_segments = {s: micro_male[[c for c in micro_male.columns if c[0] == s]].T.reset_index(drop=True) for s in ('S1', 'S2', 'S3')}
    print(f'Rat symbols matched to our gene space: {rat_mapped_fraction:.0%}')

    def micro_sex_gradient(sex, contrast):
        ref_ = reference_from_micro_sex(sex, contrast)
        mean = ref_['gradient']
        return (mean, pd.Series(np.nan, index=mean.index)), set(mean.index)

    male_sn, female_sn = sn_sex.index[sn_sex.eq('male')], sn_sex.index[sn_sex.eq('female')]

    def datasets_for(contrast):
        human_contrast = 'S3-early' if contrast == 'S3c-early' else contrast
        out_ = {('human', 'ours'): ours_gradient(contrast, expected['human']),
                ('mouse', 'ours'): ours_gradient(contrast, expected['mouse']),
                ('human', 'Lake healthy'): donor_gradient(lake_counts, lake_totals, healthy_lake, human_contrast),
                ('mouse', 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, human_contrast),
                ('mouse', 'snRNA female'): donor_gradient(sn_counts, sn_totals, female_sn, human_contrast),
                ('mouse', 'micro male GSE150338'): segment_replicates_gradient(micro_male_segments, human_contrast),
                ('mouse', 'micro male GSE212213'): micro_sex_gradient('male', human_contrast),
                ('mouse', 'micro female GSE212213'): micro_sex_gradient('female', human_contrast),
                ('rat', 'micro GSE56743'): segment_replicates_gradient(rat_segments, human_contrast)}
        if contrast != 'S2-S1':
            out_[('human', 'Census human')] = census_human_gradient()
        return out_

    D2_CONTRASTS = ('S2-S1', 'S3-early', 'S3c-early')
    datasets = {contrast: datasets_for(contrast) for contrast in D2_CONTRASTS}
    pair_rows = []
    for contrast, data in datasets.items():
        keys = list(data)
        for i, a in enumerate(keys):
            for b in keys[i + 1:]:
                genes = pd.Index(sorted(data[a][1] & data[b][1]))
                if len(genes) < 200:
                    continue
                species_pair = '–'.join(sorted({a[0], b[0]})) if a[0] != b[0] else f'within {a[0]}'
                result = disattenuated(data[a][0], data[b][0], genes)
                pair_rows.append({'contrast': contrast, 'comparison': species_pair, 'dataset_a': f'{a[0]}: {a[1]}',
                                  'dataset_b': f'{b[0]}: {b[1]}', **result})
    pairs = pd.DataFrame(pair_rows)
    pairs.to_csv(tables / 'pairwise_gradient_correlations.csv', index=False)
    display(pairs[pairs.contrast.ne('S3c-early') | pairs.dataset_a.str.contains('mouse: ours') | pairs.dataset_b.str.contains('mouse: ours')]
            .round(3).sort_values(['contrast', 'comparison', 'r_corrected']))

    index_rows = []
    for contrast, data in datasets.items():
        h, m, l, s = data[('human', 'ours')], data[('mouse', 'ours')], data[('human', 'Lake healthy')], data[('mouse', 'snRNA male')]
        genes = pd.Index(sorted(h[1] & m[1] & l[1] & s[1]))
        index_rows.append({'contrast': contrast, 'version': 'primary: ours human vs ours mouse',
                           **conservation_index(h[0], m[0], l[0], s[0], genes, n_boot=N_BOOT, seed=SEED)})
        index_rows.append({'contrast': contrast, 'version': 'external: Lake vs mouse snRNA male',
                           **conservation_index(l[0], s[0], h[0], m[0], genes, n_boot=N_BOOT, seed=SEED)})
        micro = data[('mouse', 'micro male GSE150338')]
        if ('human', 'Census human') in data:
            c = data[('human', 'Census human')]
            genes_alt = pd.Index(sorted(l[1] & s[1] & c[1] & micro[1]))
            index_rows.append({'contrast': contrast, 'version': 'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)',
                               **conservation_index(l[0], s[0], c[0], micro[0], genes_alt, n_boot=N_BOOT, seed=SEED)})
    conservation = pd.DataFrame(index_rows)
    conservation.to_csv(tables / 'conservation_index.csv', index=False)
    display(conservation.round(3))
    primary_index = conservation[conservation.version.str.startswith('primary') & conservation.contrast.isin(['S2-S1', 'S3-early'])]
    if (primary_index.ci_high < INDEX_STRONG).all():
        D2_WORDING = 'largely different genes (stands)'
    elif (primary_index.ci_low >= INDEX_PARTIAL).all():
        D2_WORDING = 'largely conserved; apparent divergence is a cross-dataset ceiling'
    else:
        D2_WORDING = 'partly conserved'
    rat_reliability = pairs[pairs.dataset_b.eq('rat: micro GSE56743') | pairs.dataset_a.eq('rat: micro GSE56743')].groupby('contrast').apply(
        lambda d: float(np.where(d.dataset_b.eq('rat: micro GSE56743'), d.reliability_b, d.reliability_a).mean()))
    RAT_USABLE = bool((rat_reliability >= RAT_RELIABILITY).all() and rat_mapped_fraction >= RAT_MAPPED)
    print(f'D2 wording: {D2_WORDING}')
    print(f'Rat: reliability {rat_reliability.round(3).to_dict()}, mapped {rat_mapped_fraction:.0%} → '
          + ('usable' if RAT_USABLE else 'excluded as unreliable; withdraw rat-based arguments'))

    # Section 6 · Part 2 (D3): symmetric classification.
    CLASS_CONTRASTS = ('S2-S1', 'S3c-early', 'S3-early')

    def species_level(pb, pb_library, specimens, parts):
        return pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in specimens for p in parts], axis=1).mean(axis=1)

    def reference_table(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
        return moderated_contrast(per_donor, level.mean())

    tables_ours, tables_ref = {}, {}
    for contrast in CLASS_CONTRASTS:
        late, early = CONTRASTS[contrast]
        both = gene_space[(count_filter(ours_counts, expected['mouse'], [late, early])
                           & count_filter(ours_counts, expected['human'], [late, early])).to_numpy()]
        for species in ('mouse', 'human'):
            replicates = ours.loc[both, [(contrast, n) for n in expected[species]]].T.reset_index(drop=True)
            tables_ours[(contrast, species)] = moderated_contrast(replicates, species_level(ours_counts, ours_library, expected[species], [late, early]))
        tables_ref[(contrast, 'mouse male')] = reference_table(sn_counts, sn_totals, male_sn, contrast)
        tables_ref[(contrast, 'mouse female')] = reference_table(sn_counts, sn_totals, female_sn, contrast)
        tables_ref[(contrast, 'human')] = reference_table(lake_counts, lake_totals, healthy_lake, contrast)
    micro_gradient = {(sex, c): (reference_from_micro_male(c) if sex == 'male' else reference_from_micro_sex('female', c))['gradient']
                      for sex in ('male', 'female') for c in CLASS_CONTRASTS}

    def noise_corrected_amplitude(contrast):
        genes = tables_ours[(contrast, 'mouse')].index

        def signal(specimens):
            reps = ours.loc[genes, [(contrast, n) for n in specimens]].to_numpy()
            return reps.mean(axis=1).var(ddof=1) - reps.var(axis=1, ddof=1).mean() / reps.shape[1]
        return float(np.sqrt(max(signal(expected['human']), 0) / signal(expected['mouse'])))

    amplitude = {c: noise_corrected_amplitude(c) for c in CLASS_CONTRASTS}
    print('Noise-corrected human ÷ mouse amplitude:', {k: round(v, 3) for k, v in amplitude.items()})
    SETTINGS = {
        'v1 (notebook 40 primary)': dict(scale=False, flat_external=False, confirm='snRNA male'),
        'v1 + A6 female snRNA': dict(scale=False, flat_external=False, confirm='snRNA female'),
        'v1 + A7 female microdissection': dict(scale=False, flat_external=False, confirm='micro female'),
        'A3b amplitude-matched': dict(scale=True, flat_external=False, confirm='snRNA male'),
        'A8 external flat confirmation': dict(scale=False, flat_external=True, confirm='snRNA male'),
        'symmetric (A3b + A8)': dict(scale=True, flat_external=True, confirm='snRNA male'),
        'symmetric + A6 female snRNA': dict(scale=True, flat_external=True, confirm='snRNA female'),
        'symmetric + A7 female microdissection': dict(scale=True, flat_external=True, confirm='micro female')}

    def mouse_calls_for(contrast, effect, margin, confirm_by):
        calls, _ = zonation_calls(tables_ours[(contrast, 'mouse')], alpha=ALPHA, effect=effect, margin=margin)
        if confirm_by.startswith('snRNA'):
            return confirm(calls, tables_ref[(contrast, 'mouse ' + confirm_by.split()[1])], alpha=CONFIRM_ALPHA)
        ref_ = micro_gradient[(confirm_by.split()[1], contrast)].reindex(calls.index)
        ok = ((calls == 'up') & (ref_ >= EFFECT)) | ((calls == 'down') & (ref_ <= -EFFECT))
        return calls.where(~calls.isin(['up', 'down']) | ok.fillna(False), 'indeterminate')

    def classify(setting):
        spec = SETTINGS[setting]
        frame = {}
        for contrast in CLASS_CONTRASTS:
            scale = amplitude[contrast] if spec['scale'] else 1.
            mouse_calls = mouse_calls_for(contrast, EFFECT, MARGIN, spec['confirm'])
            human_calls, _ = zonation_calls(tables_ours[(contrast, 'human')], alpha=ALPHA, effect=EFFECT * scale, margin=MARGIN * scale)
            human_calls = confirm(human_calls, tables_ref[(contrast, 'human')], alpha=CONFIRM_ALPHA)
            classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
            if spec['flat_external']:
                mouse_ref = tables_ref[(contrast, 'mouse female' if spec['confirm'] == 'snRNA female' else 'mouse male')]
                classes = confirm_flat(classes, mouse_calls, human_calls, mouse_ref, tables_ref[(contrast, 'human')],
                                       alpha=CONFIRM_ALPHA, effect=EFFECT)
            frame[contrast], frame[contrast + ' mouse call'], frame[contrast + ' human call'] = classes, mouse_calls, human_calls
        frame = pd.DataFrame(frame)
        frame['union S2-S1 + S3c-early'] = union_class(frame[['S2-S1', 'S3c-early']])
        return frame

    classified = {setting: classify(setting) for setting in SETTINGS}
    ORDER = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
    count_rows = []
    for setting, frame in classified.items():
        frame_b = frame[frame.index.isin(balanced)]
        for contrast in (*CLASS_CONTRASTS, 'union S2-S1 + S3c-early'):
            counts_ = frame_b[contrast].value_counts().reindex(ORDER).fillna(0).astype(int)
            count_rows.append({'setting': setting, 'contrast': contrast, **counts_.to_dict(),
                               'mouse-only ÷ human-only': counts_['mouse-only'] / max(counts_['human-only'], 1)})
    class_counts = pd.DataFrame(count_rows)
    class_counts.to_csv(tables / 'class_counts_by_setting_and_contrast.csv', index=False)
    display(class_counts.set_index(['setting', 'contrast'])[['conserved', 'reversal', 'mouse-only', 'human-only', 'mouse-only ÷ human-only']].round(2))
    symmetric = classified['symmetric (A3b + A8)']
    symmetric.assign(probe_balanced=symmetric.index.isin(balanced)).to_csv(tables / 'gene_classes_symmetric.csv')
    ratio_sym = class_counts[class_counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')['mouse-only ÷ human-only']
    if ratio_sym['S2-S1'] >= 2 and ratio_sym['S3c-early'] >= 2:
        D3_ASYMMETRY = 'asymmetry supported'
    elif ratio_sym['S2-S1'] >= 2:
        D3_ASYMMETRY = 'asymmetry confined to S2 − S1'
    else:
        D3_ASYMMETRY = 'no consistent asymmetry'
    print('D3 decision:', D3_ASYMMETRY, ratio_sym.round(2).to_dict())
    strong_rows = {}
    for setting in ('v1 (notebook 40 primary)', 'symmetric (A3b + A8)'):
        frame = classified[setting]
        genes_ = frame.index[frame['S2-S1'].eq('mouse-only') & frame.index.isin(balanced)]
        m, h = tables_ours[('S2-S1', 'mouse')]['mean'].loc[genes_], tables_ours[('S2-S1', 'human')]['mean'].loc[genes_]
        strong_rows[setting] = {'S2 − S1 mouse-only genes': len(genes_), '|mouse| ≥ 1.5 and |human| ≤ 0.3': int(((m.abs() >= 1.5) & (h.abs() <= .3)).sum()),
                                '|mouse| ≥ 1 ': int((m.abs() >= 1).sum()), 'median |mouse|': float(m.abs().median())}
    strong_mouse_only = pd.DataFrame(strong_rows).T
    strong_mouse_only.to_csv(tables / 'mouse_only_gradient_distribution.csv')
    display(strong_mouse_only)

    # Exports for later notebooks (74, 77): per-specimen gradients, species means and flags.
    gradient_table = ours.copy()
    gradient_table.columns = [f'{c} | {n}' for c, n in gradient_table.columns]
    for contrast in CLASS_CONTRASTS:
        for species in ('mouse', 'human'):
            moderated = tables_ours[(contrast, species)]
            gradient_table[f'{contrast} | {species} mean (moderated)'] = moderated['mean'].reindex(gradient_table.index)
    gradient_table['probe_balanced'] = gradient_table.index.isin(balanced)
    gradient_table['label_panel_gene'] = gradient_table.index.isin(set(panel_genes))
    gradient_table.rename_axis('gene').to_csv(tables / 'gene_gradients.csv')

    # Section 8 · Figures.
    COLORS = {'within human': HUMAN, 'within mouse': MOUSE, 'human–mouse': '#009E73', 'mouse–rat': '#CC79A7', 'human–rat': '#999999'}
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={'width_ratios': [1.3, 1, 1.2]})
    ax = axes[0]
    plot_pairs = pairs[pairs.contrast.isin(['S2-S1', 'S3-early'])].dropna(subset=['r_corrected'])
    for k, contrast in enumerate(['S2-S1', 'S3-early']):
        block = plot_pairs[plot_pairs.contrast.eq(contrast)]
        for j, comparison in enumerate(['within human', 'within mouse', 'human–mouse', 'mouse–rat']):
            values = block.loc[block.comparison.eq(comparison), 'r_corrected']
            x = k * 5 + j + np.random.default_rng(j).uniform(-.15, .15, len(values))
            ax.scatter(x, values.clip(-.2, 1.2), color=COLORS[comparison], s=22, label=comparison if k == 0 else None)
    ax.set_xticks([1.5, 6.5], ['S2 − S1', 'S3 − early'])
    ax.axhline(0, color=NULL, lw=.6)
    ax.set_ylabel('Noise-corrected correlation of gene gradients')
    ax.legend(frameon=False, fontsize=7)
    ax.set_title('a · Same-species ceilings and cross-species agreement', loc='left')
    ax = axes[1]
    plot_index = conservation[conservation.contrast.isin(['S2-S1', 'S3-early', 'S3c-early'])]
    yy = np.arange(len(plot_index))[::-1]
    ax.errorbar(plot_index['index'], yy, xerr=[plot_index['index'] - plot_index.ci_low, plot_index.ci_high - plot_index['index']], fmt='o', color=INK)
    ax.set_yticks(yy, [f'{c} · {v.split(":")[0]}' for c, v in zip(plot_index.contrast, plot_index.version)], fontsize=7)
    for value in (INDEX_STRONG, INDEX_PARTIAL, 1):
        ax.axvline(value, color=NULL, ls='--' if value < 1 else '-', lw=.8)
    ax.set_xlabel('Conservation index (cross ÷ within-species ceiling)')
    ax.set_title('b · Conservation index', loc='left')
    ax = axes[2]
    show = class_counts[class_counts.setting.isin(['v1 (notebook 40 primary)', 'symmetric (A3b + A8)', 'symmetric + A6 female snRNA',
                                                   'symmetric + A7 female microdissection']) & class_counts.contrast.isin(CLASS_CONTRASTS)]
    labels_ = [f'{s.split(" (")[0]} · {c}' for s, c in zip(show.setting, show.contrast)]
    yy = np.arange(len(show))[::-1]
    ax.barh(yy + .2, show['mouse-only'], height=.38, color=MOUSE, label='mouse-only')
    ax.barh(yy - .2, show['human-only'], height=.38, color=HUMAN, label='human-only')
    ax.set_yticks(yy, labels_, fontsize=6.5)
    ax.set_xlabel('Probe-balanced genes')
    ax.legend(frameon=False, fontsize=7)
    ax.set_title('c · Species-only classes by setting and contrast', loc='left')
    fig.tight_layout()
    save_figure(fig, figures, 'fig_revision_ceiling_and_classes')
    plt.show()
    plt.close(fig)

    decisions = pd.Series({'D2 wording': D2_WORDING, 'rat usable': RAT_USABLE, 'D3 asymmetry': D3_ASYMMETRY})
    decisions.to_csv(tables / 'decisions.csv')
    display(decisions.to_frame('value'))
    manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'labels': I['labels_name'],
                'decisions': decisions.astype(str).to_dict(), 'amplitude': amplitude,
                'conservation_index': conservation.round(4).to_dict(orient='records'),
                'rat_reliability': rat_reliability.to_dict(), 'rat_mapped_fraction': rat_mapped_fraction}
    (tables / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return {k: v for k, v in locals().items() if not k.startswith('__')}

## 5 · Notebook 49: agreement by gradient strength, and the index's sensitivity

Notebook 49's code on part 43's objects.
- **Strength.** It is cross-fitted. Donors split into sha256 folds (Lake healthy 4/3, Census male 6/6), and
  strength is the maximum percentile rank of |fold-A gradient|. Evaluation runs on fold B and our data, then the
  folds swap.
- **Bins.** 0, 50, 80, 90, 95, 99 and 100%. BOTTOM is below 80%; TOP is 95% and above.
- **Settings.** Winsorisation q 0.01, minimum reliability 0.2, and 1,000 gene and module-block bootstraps.
- **Rules.**
  - (i): TOP − BOTTOM ≥ 0.25 with the interval excluding 0.
  - (ii): the BOTTOM ceilings have lower bounds ≥ 0.3 and the index has an upper bound < 0.5.
- **Guard.** Part 43's index must be reproduced by notebook 43's function and by notebook 49's estimator (G6).
- **Secondary arm.** It runs part A with the gene bootstrap only: strength curves for every scheme, the rules, the
  bin classes and direction agreement.

In [ ]:
import pseudospace.conservation_strength as strength_module
from pseudospace.conservation_strength import (STATS, binned_statistics, bin_labels, block_draws, coexpression_modules,
                                               donor_folds, gene_draws, percentile_interval, percentile_rank,
                                               resampled_gradient, strength_bins, strength_score, subset_statistics)
from statsmodels.stats.proportion import proportion_confint

BINS = bin_labels(EDGES)
SUBSETS = {**{label: [i] for i, label in enumerate(BINS)}, 'BOTTOM (<80%)': [0, 1], 'TOP (≥95%)': [4, 5],
           'all genes': list(range(len(BINS)))}
SUBSET_NAMES = list(SUBSETS)
SCHEMES = {'max rank (primary)': 'max', 'mean rank': 'mean', 'human rank only': 'human', 'mouse rank only': 'mouse'}


def part49(I, R, out, *, full=True, tag='49'):
    """Notebook 49 on part 43's objects. ``full=False``: part A with the gene bootstrap only (secondary arm)."""
    tables, figures = folders(out, tag)
    cache_ = out / 'stage_cache' / tag
    code = digest([NOTEBOOK_LOGIC_VERSION, Path(strength_module.__file__), Path(conservation_index.__code__.co_filename)])
    datasets, ours, gene_space, lognorm = R['datasets'], R['ours'], R['gene_space'], R['lognorm']
    lake_counts, lake_totals, sn_counts, sn_totals = R['lake_counts'], R['lake_totals'], R['sn_counts'], R['sn_totals']
    census_h_counts, micro_male = R['census_h_counts'], R['micro_male']
    specimen = R['specimen']
    male_sn = list(R['male_sn'])
    index_genes = {c: pd.Index(sorted(d[('human', 'ours')][1] & d[('mouse', 'ours')][1] & d[('human', 'Lake healthy')][1]
                                      & d[('mouse', 'snRNA male')][1])) for c, d in datasets.items()}
    print({c: len(g) for c, g in index_genes.items()}, 'genes in the index set (expressed in all four datasets)')

    # Guard (G6): notebook 43's index of part 43 is reproduced by notebook 43's function and by notebook 49's estimator.
    published = R['conservation']
    VERSIONS = {'primary: ours human vs ours mouse': (('human', 'ours'), ('mouse', 'ours'), ('human', 'Lake healthy'), ('mouse', 'snRNA male')),
                'external: Lake vs mouse snRNA male': (('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('human', 'ours'), ('mouse', 'ours')),
                'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)':
                    (('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('human', 'Census human'), ('mouse', 'micro male GSE150338'))}
    FIELDS = ('index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human', 'ceiling_mouse', 'genes')
    for row in published.itertuples(index=False):
        data = datasets[row.contrast]
        keys = VERSIONS[row.version]
        genes = (index_genes[row.contrast] if not row.version.startswith('external, alternative')
                 else pd.Index(sorted(set.intersection(*[data[k][1] for k in keys]))))
        result = conservation_index(*[data[k][0] for k in keys], genes, n_boot=N_BOOT, seed=SEED43)
        for field in FIELDS:
            if not np.isclose(result[field], getattr(row, field), rtol=0, atol=1e-9, equal_nan=True):
                raise ValueError(f'Part 43 not reproduced: {row.contrast} · {row.version} · {field}')

    def as_arrays(gradient, genes):
        mean, noise = gradient
        return mean.reindex(genes).to_numpy(float), noise.reindex(genes).to_numpy(float)

    def full_data(contrast, genes):
        d = datasets[contrast]
        return {'human': as_arrays(d[('human', 'ours')][0], genes), 'mouse': as_arrays(d[('mouse', 'ours')][0], genes),
                'human_ref': as_arrays(d[('human', 'Lake healthy')][0], genes), 'mouse_ref': as_arrays(d[('mouse', 'snRNA male')][0], genes)}

    for row in published[published.version.str.startswith(('primary', 'external: '))].itertuples(index=False):
        data = full_data(row.contrast, index_genes[row.contrast])
        stat = 'index' if row.version.startswith('primary') else 'index_external'
        n = len(index_genes[row.contrast])
        point = subset_statistics(data, np.arange(n), q=Q_PUBLISHED, match_reliability=False)
        draws = np.array([subset_statistics(data, idx, q=Q_PUBLISHED, match_reliability=False)[stat] for idx in gene_draws(n, N_BOOT, SEED43)])
        low, high, _ = percentile_interval(draws[:, None])
        for value, target in ((point[stat], row.index), (low[0], row.ci_low), (high[0], row.ci_high),
                              (point['rstar_ceiling_human'], row.ceiling_human), (point['rstar_ceiling_mouse'], row.ceiling_mouse)):
            if not np.isclose(value, target, rtol=0, atol=1e-9, equal_nan=True):
                raise ValueError(f'Notebook 49 estimator does not reproduce part 43: {row.contrast} · {row.version}')
    print(f'Guard G6: part 43 reproduced exactly ({len(published)} rows), also by notebook 49\'s estimator.')

    # Co-expression modules for the module-block bootstrap.
    module_genes = pd.Index(sorted(set().union(*[set(g) for g in index_genes.values()])))
    species_of_structure = np.where(np.isin(specimen, expected['mouse']), 'mouse', 'human')
    if full:
        def compute_modules():
            matrix = lognorm[:, gene_space.get_indexer(module_genes)]
            return {f'k{k}': coexpression_modules(matrix, specimen, species_of_structure, n_components=N_COMPONENTS, n_modules=k, seed=SEED49)
                    for k in (N_MODULES, *N_MODULES_SENSITIVITY)}

        modules = cached_payload('coexpression_modules', compute_modules, root=cache_,
                                 params={'components': N_COMPONENTS, 'k': [N_MODULES, *N_MODULES_SENSITIVITY], 'seed': SEED49},
                                 inputs={'genes': list(module_genes), 'obs': list(R['unit_ids']), 'lognorm': digest(lognorm)}, code=code)
        module_of = {k: pd.Series(v, index=module_genes) for k, v in modules.items()}
        module_sizes = pd.DataFrame({k: v.value_counts().describe()[['count', 'min', '50%', 'max']] for k, v in module_of.items()}).T
        display(module_sizes.rename(columns={'count': 'modules', '50%': 'median genes'}))

    # Part A: agreement by gradient strength.
    lake_fold = pd.Series(donor_folds(healthy_lake), index=healthy_lake)
    sn_fold = pd.Series(donor_folds(male_sn), index=male_sn)
    print('Lake folds:', lake_fold.value_counts().sort_index().to_dict(), '· snRNA male folds:', sn_fold.value_counts().sort_index().to_dict())

    def reference_contrast(contrast):
        return 'S3-early' if contrast == 'S3c-early' else contrast

    def reference_per_donor(contrast, genes):
        rc = reference_contrast(contrast)
        lake_pd = donor_contrasts(lake_counts, lake_totals, healthy_lake, rc)[0][genes]
        sn_pd = donor_contrasts(sn_counts, sn_totals, male_sn, rc)[0][genes]
        return lake_pd, sn_pd

    def fold_gradient(per_donor, folds, fold):
        mean, noise = paired_gradient(per_donor.loc[folds.index[folds.eq(fold)]])
        return mean.to_numpy(float), noise.to_numpy(float)

    def crossfit_directions(contrast, how):
        genes = index_genes[contrast]
        lake_pd, sn_pd = reference_per_donor(contrast, genes)
        ours_data = full_data(contrast, genes)
        out_ = []
        for f in (0, 1):
            rank_h = percentile_rank(fold_gradient(lake_pd, lake_fold, f)[0])
            rank_m = percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0])
            score = {'human': rank_h, 'mouse': rank_m}.get(how)
            score = strength_score([rank_h, rank_m], how) if score is None else score
            data = {'human': ours_data['human'], 'mouse': ours_data['mouse'],
                    'human_ref': fold_gradient(lake_pd, lake_fold, 1 - f), 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)}
            out_.append((data, strength_bins(score, EDGES)))
        return out_

    def naive_directions(contrast):
        data = full_data(contrast, index_genes[contrast])
        return [(data, strength_bins(percentile_rank(data['mouse'][0]), EDGES))]

    def run_bootstrap(directions, draws, q):
        return np.stack([binned_statistics(directions, idx, SUBSETS, q=q, min_reliability=MIN_RELIABILITY) for idx in draws])

    def strength_stage(contrast):
        genes = index_genes[contrast]
        n = len(genes)
        out_ = {}
        schemes = {**{name: crossfit_directions(contrast, how) for name, how in SCHEMES.items()},
                   'same-data |our mouse| (referee)': naive_directions(contrast)}
        for name, directions in schemes.items():
            for q in ((Q_PUBLISHED, 0.) if name == 'max rank (primary)' else (Q_PUBLISHED,)):
                out_[f'{name}|q{q}|point'] = binned_statistics(directions, np.arange(n), SUBSETS, q=q, min_reliability=MIN_RELIABILITY)
                out_[f'{name}|q{q}|gene'] = run_bootstrap(directions, gene_draws(n, N_BOOT, SEED49), q)
            if name == 'max rank (primary)':
                if full:
                    blocks = module_of[f'k{N_MODULES}'].reindex(genes).to_numpy()
                    out_[f'{name}|q{Q_PUBLISHED}|module'] = run_bootstrap(directions, block_draws(blocks, N_BOOT, SEED49), Q_PUBLISHED)
                for f in (0, 1):
                    out_[f'{name}|q{Q_PUBLISHED}|direction{f}'] = binned_statistics([directions[f]], np.arange(n), SUBSETS, q=Q_PUBLISHED,
                                                                                   min_reliability=MIN_RELIABILITY)
        return out_

    strength_results = {}
    for contrast in index_genes:
        strength_results[contrast] = cached_payload(
            f'strength_{contrast}', lambda c=contrast: strength_stage(c), root=cache_,
            params={'edges': EDGES, 'subsets': SUBSETS, 'schemes': SCHEMES, 'q': [Q_PUBLISHED, 0.], 'n_boot': N_BOOT, 'seed': SEED49,
                    'min_reliability': MIN_RELIABILITY, 'k': N_MODULES, 'full': full},
            inputs={'genes': list(index_genes[contrast]), 'ours': digest(ours), 'lake': digest(lake_counts),
                    'sn': digest(sn_counts.sort_index(axis=1)), 'folds': [lake_fold.to_dict(), sn_fold.to_dict()],
                    'modules': digest(module_of[f'k{N_MODULES}']) if full else None}, code=code)
    BOOTS = ('gene', 'module') if full else ('gene',)

    def summarise(stage, key_prefix, boot):
        point = stage[f'{key_prefix}|point']
        low, high, missing = percentile_interval(stage[f'{key_prefix}|{boot}'], max_missing=MAX_MISSING)
        rows = []
        for i, subset in enumerate(SUBSET_NAMES):
            for j, stat in enumerate(STATS):
                rows.append({'subset': subset, 'statistic': stat, 'value': point[i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                             'missing_draws': missing[i, j]})
        return pd.DataFrame(rows)

    curve_rows = []
    for contrast, stage in strength_results.items():
        for key in sorted({k.rsplit('|', 1)[0] for k in stage if k.endswith('|point')}):
            scheme, q = key.rsplit('|', 1)
            for boot in BOOTS:
                if f'{key}|{boot}' in stage:
                    curve_rows.append(summarise(stage, key, boot).assign(contrast=contrast, scheme=scheme, q=float(q[1:]), bootstrap=boot))
    curves = pd.concat(curve_rows, ignore_index=True)[['contrast', 'scheme', 'q', 'bootstrap', 'subset', 'statistic', 'value', 'ci_low', 'ci_high', 'missing_draws']]
    curves.to_csv(tables / 'strength_curves.csv', index=False)
    direction_rows = []
    for contrast, stage in strength_results.items():
        for f in (0, 1):
            frame = pd.DataFrame(stage[f'max rank (primary)|q{Q_PUBLISHED}|direction{f}'], index=SUBSET_NAMES, columns=STATS)
            direction_rows.append(frame.assign(contrast=contrast, direction=f'strength fold {f} → evaluation fold {1 - f}'))
    pd.concat(direction_rows).rename_axis('subset').reset_index().to_csv(tables / 'strength_curves_by_direction.csv', index=False)

    def wide(contrast, scheme='max rank (primary)', q=Q_PUBLISHED, boot='gene', stats=('genes', 'rstar_cross', 'rstar_ceiling_human',
                                                                                       'rstar_ceiling_mouse', 'index', 'index_external')):
        block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(q) & curves.bootstrap.eq(boot)
                       & curves.statistic.isin(stats)]
        table = block.assign(text=[f'{v:.2f} ({lo:.2f}, {hi:.2f})' if s != 'genes' else f'{v:.0f}'
                                   for v, lo, hi, s in zip(block.value, block.ci_low, block.ci_high, block.statistic)])
        return table.pivot(index='subset', columns='statistic', values='text').reindex(SUBSET_NAMES)[list(stats)]

    for contrast in index_genes:
        print(f'\n{contrast} · primary scheme (cross-fitted max rank), q = 0.01, gene bootstrap')
        display(wide(contrast))

    # Decision rules (i) and (ii).
    def group_draws(stage, key, boot, subset, stat):
        return stage[f'{key}|{boot}'][:, SUBSET_NAMES.index(subset), STATS.index(stat)]

    def group_point(stage, key, subset, stat):
        return stage[f'{key}|point'][SUBSET_NAMES.index(subset), STATS.index(stat)]

    PRIMARY_KEY = f'max rank (primary)|q{Q_PUBLISHED}'
    decision_rows = []
    for contrast, stage in strength_results.items():
        bottom_estimable = np.isfinite(group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', 'index'))
        stat = 'index' if bottom_estimable else 'rstar_cross'
        for boot in BOOTS:
            diff = group_draws(stage, PRIMARY_KEY, boot, 'TOP (≥95%)', stat) - group_draws(stage, PRIMARY_KEY, boot, 'BOTTOM (<80%)', stat)
            d_low, d_high, d_missing = percentile_interval(diff[:, None], max_missing=MAX_MISSING)
            bounds = {}
            for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse', 'index', 'rstar_cross'):
                lo, hi, _ = percentile_interval(group_draws(stage, PRIMARY_KEY, boot, 'BOTTOM (<80%)', s)[:, None], max_missing=MAX_MISSING)
                bounds[s] = (group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', s), lo[0], hi[0])
            point_diff = group_point(stage, PRIMARY_KEY, 'TOP (≥95%)', stat) - group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', stat)
            concentration = bool(point_diff >= CONCENTRATION_MIN and d_low[0] > 0)
            ceilings_low = [bounds[s][1] for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse')]
            ceilings_high = [bounds[s][2] for s in ('rstar_ceiling_human', 'rstar_ceiling_mouse')]
            reproducible_not_shared = bool(all(np.isfinite(ceilings_low)) and min(ceilings_low) >= CEILING_MIN
                                           and np.isfinite(bounds['index'][2]) and bounds['index'][2] < INDEX_STRONG)
            not_reliably_zonated = bool(any(not np.isfinite(v) for v in ceilings_high) or min(ceilings_high) < CEILING_MIN)
            decision_rows.append({'contrast': contrast, 'bootstrap': boot, 'statistic compared': stat,
                                  'TOP': group_point(stage, PRIMARY_KEY, 'TOP (≥95%)', stat),
                                  'BOTTOM': group_point(stage, PRIMARY_KEY, 'BOTTOM (<80%)', stat),
                                  'TOP − BOTTOM': point_diff, 'difference low': d_low[0], 'difference high': d_high[0],
                                  '(i) concentration': concentration,
                                  'BOTTOM human ceiling': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['rstar_ceiling_human']),
                                  'BOTTOM mouse ceiling': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['rstar_ceiling_mouse']),
                                  'BOTTOM index': '{:.2f} ({:.2f}, {:.2f})'.format(*bounds['index']),
                                  '(ii) reproducible but not shared': reproducible_not_shared,
                                  'BOTTOM not reliably zonated': not_reliably_zonated})
    strength_decisions = pd.DataFrame(decision_rows)
    strength_decisions.to_csv(tables / 'strength_decision_rules.csv', index=False)
    display(strength_decisions)
    primary_contrasts = strength_decisions[strength_decisions.contrast.isin(['S2-S1', 'S3-early'])]
    RULE_I = bool(primary_contrasts['(i) concentration'].all())
    RULE_II = bool(primary_contrasts['(ii) reproducible but not shared'].all())
    NOT_ZONATED = bool(primary_contrasts['BOTTOM not reliably zonated'].any())
    if RULE_I and RULE_II:
        TITLE_VERDICT = 'supported: shared zonation concentrates in the strongest genes; reproducible weaker gradients differ'
    elif RULE_I and NOT_ZONATED:
        TITLE_VERDICT = 'reword: agreement is confined to genes that are reliably zonated, and few genes are'
    elif RULE_I:
        TITLE_VERDICT = 'partly: agreement concentrates in the strongest genes; weak-gene ceilings are too uncertain to call species difference'
    else:
        TITLE_VERDICT = 'not supported: no strength dependence; drop the strongest-genes framing'
    print('Rule (i):', RULE_I, '· rule (ii):', RULE_II, '· BOTTOM not reliably zonated:', NOT_ZONATED)
    print('Title verdict:', TITLE_VERDICT)

    def bin_class(stage, key, boot, subset):
        lo_h, hi_h, _ = percentile_interval(group_draws(stage, key, boot, subset, 'rstar_ceiling_human')[:, None], max_missing=MAX_MISSING)
        lo_m, hi_m, _ = percentile_interval(group_draws(stage, key, boot, subset, 'rstar_ceiling_mouse')[:, None], max_missing=MAX_MISSING)
        lo_i, hi_i, _ = percentile_interval(group_draws(stage, key, boot, subset, 'index')[:, None], max_missing=MAX_MISSING)
        if not (np.isfinite(hi_h[0]) and np.isfinite(hi_m[0])) or min(hi_h[0], hi_m[0]) < CEILING_MIN:
            return 'not reliably zonated'
        if min(lo_h[0], lo_m[0]) >= CEILING_MIN and np.isfinite(hi_i[0]) and hi_i[0] < INDEX_STRONG:
            return 'reproducible, not shared'
        if np.isfinite(lo_i[0]) and lo_i[0] >= INDEX_STRONG:
            return 'shared'
        return 'uncertain'

    bin_rows = []
    for contrast, stage in strength_results.items():
        for subset in BINS:
            bin_rows.append({'contrast': contrast, 'bin': subset, 'genes': int(group_point(stage, PRIMARY_KEY, subset, 'genes')),
                             'class (gene bootstrap)': bin_class(stage, PRIMARY_KEY, 'gene', subset),
                             'class (module bootstrap)': bin_class(stage, PRIMARY_KEY, 'module', subset) if full else np.nan})
    bin_classes = pd.DataFrame(bin_rows)
    bin_classes.to_csv(tables / 'strength_bin_classes.csv', index=False)
    display(bin_classes.pivot(index='bin', columns='contrast', values='class (gene bootstrap)').reindex(BINS))
    gene_counts = bin_classes.pivot_table(index='class (gene bootstrap)', columns='contrast', values='genes', aggfunc='sum', fill_value=0)
    display(gene_counts)

    scheme_rows = []
    for contrast in index_genes:
        for scheme in (*SCHEMES, 'same-data |our mouse| (referee)'):
            block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(Q_PUBLISHED) & curves.bootstrap.eq('gene')]
            for subset in ('BOTTOM (<80%)', 'TOP (≥95%)', BINS[-1]):
                values = block[block.subset.eq(subset)].set_index('statistic')
                scheme_rows.append({'contrast': contrast, 'scheme': scheme, 'subset': subset,
                                    'raw r': values.loc['r_cross', 'value'], 'r*': values.loc['rstar_cross', 'value'],
                                    'human ceiling': values.loc['rstar_ceiling_human', 'value'],
                                    'mouse ceiling': values.loc['rstar_ceiling_mouse', 'value'],
                                    'index': values.loc['index', 'value'],
                                    'index CI': f"({values.loc['index', 'ci_low']:.2f}, {values.loc['index', 'ci_high']:.2f})",
                                    'sign agreement': values.loc['sign_agreement', 'value']})
    scheme_table = pd.DataFrame(scheme_rows)
    scheme_table.to_csv(tables / 'strength_schemes.csv', index=False)
    display(scheme_table.round(3))

    # Direction agreement among genes zonated in both species (part 43's symmetric classes).
    classes43 = R['symmetric'].assign(probe_balanced=R['symmetric'].index.isin(balanced))
    agreement_rows = []
    for contrast in index_genes:
        genes = index_genes[contrast]
        data = full_data(contrast, genes)
        full_bins = pd.Series(strength_bins(strength_score([percentile_rank(data['human_ref'][0]), percentile_rank(data['mouse_ref'][0])]), EDGES),
                              index=genes)
        klass = classes43.loc[classes43.probe_balanced.astype(bool), contrast].dropna()
        both = klass[klass.isin(['conserved', 'reversal'])]
        missing = int((~both.index.isin(genes)).sum())
        both = both[both.index.isin(genes)]
        sign = pd.Series(np.sign(data['human'][0]) == np.sign(data['mouse'][0]), index=genes)
        for b, label in enumerate([*BINS, 'all genes']):
            members = genes if label == 'all genes' else genes[full_bins.eq(b).to_numpy()]
            zb = both[both.index.isin(members)]
            n_both, n_conserved = len(zb), int(zb.eq('conserved').sum())
            low, high = proportion_confint(n_conserved, n_both, method='wilson') if n_both else (np.nan, np.nan)
            agreement_rows.append({'contrast': contrast, 'bin': label, 'genes': len(members), 'zonated in both': n_both,
                                   'conserved': n_conserved, 'share conserved': n_conserved / n_both if n_both else np.nan,
                                   'share low': low, 'share high': high, 'raw sign agreement (all genes)': float(sign.loc[members].mean()),
                                   'zonated-in-both genes outside the index set': missing})
    direction_agreement = pd.DataFrame(agreement_rows)
    direction_agreement.to_csv(tables / 'direction_agreement_by_strength.csv', index=False)
    display(direction_agreement.round(3))

    WORDING_STANDS, primary_b, index_sensitivity, posthoc, human_ceiling = np.nan, None, None, None, None
    if full:
        # External pair binned on our strength.
        def external_stage():
            out_ = {}
            for contrast in index_genes:
                d = datasets[contrast]
                keys = [('human', 'ours'), ('mouse', 'ours'), ('human', 'Lake healthy'), ('mouse', 'snRNA male'), ('mouse', 'micro male GSE150338')]
                keys += [('human', 'Census human')] if contrast != 'S2-S1' else []
                genes = pd.Index(sorted(set.intersection(*[d[k][1] for k in keys])))
                human_ref = d[('human', 'Census human')][0] if contrast != 'S2-S1' else d[('human', 'Lake healthy')][0]
                data = {'human': as_arrays(d[('human', 'Lake healthy')][0], genes), 'mouse': as_arrays(d[('mouse', 'snRNA male')][0], genes),
                        'human_ref': as_arrays(human_ref, genes), 'mouse_ref': as_arrays(d[('mouse', 'micro male GSE150338')][0], genes)}
                score = strength_score([percentile_rank(as_arrays(d[('human', 'ours')][0], genes)[0]),
                                        percentile_rank(as_arrays(d[('mouse', 'ours')][0], genes)[0])])
                directions = [(data, strength_bins(score, EDGES))]
                out_[f'{contrast}|point'] = binned_statistics(directions, np.arange(len(genes)), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
                out_[f'{contrast}|gene'] = run_bootstrap(directions, gene_draws(len(genes), N_BOOT, SEED49), Q_PUBLISHED)
            return out_

        external_results = cached_payload('external_pair_by_our_strength', external_stage, root=cache_,
                                          params={'edges': EDGES, 'subsets': SUBSETS, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED49,
                                                  'min_reliability': MIN_RELIABILITY},
                                          inputs={'ours': digest(ours), 'lake': digest(lake_counts), 'sn': digest(sn_counts.sort_index(axis=1)),
                                                  'micro': digest(micro_male), 'census_human': digest(census_h_counts.sort_index(axis=1))}, code=code)
        external_rows = []
        for contrast in index_genes:
            frame = summarise(external_results, contrast, 'gene').assign(contrast=contrast)
            if contrast == 'S2-S1':      # no independent human reference: the human ceiling and index are undefined
                undefined = frame.statistic.isin(['r_ceiling_human', 'rstar_ceiling_human', 'index', 'index_external', 'r_external',
                                                  'rstar_external', 'reliability_human_ref'])
                frame.loc[undefined, ['value', 'ci_low', 'ci_high']] = np.nan
            external_rows.append(frame)
        external_curves = pd.concat(external_rows, ignore_index=True)
        external_curves['statistic'] = external_curves.statistic.replace({
            'rstar_cross': 'rstar Lake vs snRNA male', 'r_cross': 'r Lake vs snRNA male', 'rstar_ceiling_human': 'rstar Lake vs Census human',
            'rstar_ceiling_mouse': 'rstar snRNA male vs microdissection', 'index': 'index (external pair, independent ceilings)'})
        external_curves.to_csv(tables / 'external_pair_by_our_strength.csv', index=False)

        # Mouse-strength bins with the full Lake reference (notebook 49's post hoc diagnostic; pre-specified here).
        def mouse_strength_full_lake(contrast):
            genes = index_genes[contrast]
            _, sn_pd = reference_per_donor(contrast, genes)
            ours_data = full_data(contrast, genes)
            return [({**ours_data, 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)},
                     strength_bins(percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0]), EDGES)) for f in (0, 1)]

        def posthoc_stage():
            out_ = {}
            for contrast in index_genes:
                directions = mouse_strength_full_lake(contrast)
                n = len(index_genes[contrast])
                out_[f'{contrast}|point'] = binned_statistics(directions, np.arange(n), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
                out_[f'{contrast}|gene'] = run_bootstrap(directions, gene_draws(n, N_BOOT, SEED49), Q_PUBLISHED)
            return out_

        posthoc = cached_payload('posthoc_mouse_strength_full_lake', posthoc_stage, root=cache_,
                                 params={'edges': EDGES, 'subsets': SUBSETS, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED49,
                                         'min_reliability': MIN_RELIABILITY},
                                 inputs={'ours': digest(ours), 'lake': digest(lake_counts), 'sn': digest(sn_counts.sort_index(axis=1)),
                                         'folds': sn_fold.to_dict()}, code=code)
        posthoc_table = pd.concat([summarise(posthoc, contrast, 'gene').assign(contrast=contrast) for contrast in index_genes], ignore_index=True)
        posthoc_table.to_csv(tables / 'posthoc_mouse_strength_full_lake.csv', index=False)

        # Part B: sensitivity of the conservation index.
        SETTINGS_B = [(q, match) for q in Q_GRID for match in (False, True)]
        B_STATS = ('index', 'index_external', 'rstar_cross', 'rstar_external', 'rstar_ceiling_human', 'rstar_ceiling_mouse')

        def evaluate_settings(data, idx):
            out_ = np.empty((len(SETTINGS_B), len(B_STATS)))
            for i, (q, match) in enumerate(SETTINGS_B):
                result = subset_statistics(data, idx, q=q, match_reliability=match)
                out_[i] = [result[s] for s in B_STATS]
            return out_

        def sensitivity_stage(contrast):
            genes = index_genes[contrast]
            n = len(genes)
            data = full_data(contrast, genes)
            lake_pd, sn_pd = reference_per_donor(contrast, genes)
            out_ = {'point': evaluate_settings(data, np.arange(n)),
                    'gene': np.stack([evaluate_settings(data, idx) for idx in gene_draws(n, N_BOOT, SEED43)])}
            for k in (N_MODULES, *N_MODULES_SENSITIVITY):
                blocks = module_of[f'k{k}'].reindex(genes).to_numpy()
                out_[f'module k={k}'] = np.stack([evaluate_settings(data, idx) for idx in block_draws(blocks, N_BOOT, SEED49)])
            for name in ('donor', 'donor + gene'):
                rng = np.random.default_rng(SEED49)
                draws = []
                for _ in range(N_BOOT):
                    resampled = {**data, 'human_ref': resampled_gradient(lake_pd.to_numpy(), rng), 'mouse_ref': resampled_gradient(sn_pd.to_numpy(), rng)}
                    idx = rng.integers(0, n, n) if name == 'donor + gene' else np.arange(n)
                    draws.append(evaluate_settings(resampled, idx))
                out_[name] = np.stack(draws)
            loo = []
            for donor in healthy_lake:
                mean, noise = paired_gradient(lake_pd.drop(index=donor))
                loo.append(evaluate_settings({**data, 'human_ref': (mean.to_numpy(float), noise.to_numpy(float))}, np.arange(n)))
            out_['leave one Lake donor out'] = np.stack(loo)
            return out_

        sensitivity = {contrast: cached_payload(f'sensitivity_{contrast}', lambda c=contrast: sensitivity_stage(c), root=cache_,
                                                params={'settings': SETTINGS_B, 'stats': B_STATS, 'n_boot': N_BOOT, 'seeds': [SEED43, SEED49],
                                                        'k': [N_MODULES, *N_MODULES_SENSITIVITY]},
                                                inputs={'genes': list(index_genes[contrast]), 'ours': digest(ours), 'lake': digest(lake_counts),
                                                        'sn': digest(sn_counts.sort_index(axis=1)), 'modules': digest(modules)}, code=code)
                       for contrast in index_genes}
        BOOTSTRAPS = ['gene', f'module k={N_MODULES}', *[f'module k={k}' for k in N_MODULES_SENSITIVITY], 'donor', 'donor + gene']
        sens_rows = []
        for contrast, stage in sensitivity.items():
            for boot in BOOTSTRAPS:
                low, high, missing = percentile_interval(stage[boot], max_missing=MAX_MISSING)
                for i, (q, match) in enumerate(SETTINGS_B):
                    for version, stat in (('primary (ours)', 'index'), ('external (Lake vs snRNA male)', 'index_external')):
                        j = B_STATS.index(stat)
                        sens_rows.append({'contrast': contrast, 'version': version, 'winsor_q': q,
                                          'reliability': 'matched (winsorised)' if match else 'notebook 43 (unwinsorised)', 'bootstrap': boot,
                                          'index': stage['point'][i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                                          'upper < 0.5': bool(high[i, j] < INDEX_STRONG),
                                          'ceiling_human': stage['point'][i, B_STATS.index('rstar_ceiling_human')],
                                          'ceiling_mouse': stage['point'][i, B_STATS.index('rstar_ceiling_mouse')]})
        index_sensitivity = pd.DataFrame(sens_rows)
        index_sensitivity.to_csv(tables / 'index_sensitivity.csv', index=False)
        check = index_sensitivity[index_sensitivity.winsor_q.eq(Q_PUBLISHED) & index_sensitivity.reliability.str.startswith('notebook')
                                  & index_sensitivity.bootstrap.eq('gene')]
        for row in check.itertuples(index=False):
            target = published[published.contrast.eq(row.contrast) & published.version.str.startswith(
                'primary' if row.version.startswith('primary') else 'external: ')].iloc[0]
            if not all(np.isclose(a, b, rtol=0, atol=1e-9, equal_nan=True) for a, b in ((row.index, target['index']), (row.ci_low, target.ci_low),
                                                                                         (row.ci_high, target.ci_high))):
                raise ValueError(f'Part B does not reproduce part 43: {row.contrast} {row.version}')
        primary_b = index_sensitivity[index_sensitivity.version.str.startswith('primary') & index_sensitivity.contrast.isin(['S2-S1', 'S3-early'])]
        WORDING_STANDS = bool(primary_b['upper < 0.5'].all())
        print(f'"Largely different genes" (every primary S2 − S1 / S3 − early upper bound < 0.5): {WORDING_STANDS}')
        published_setting = SETTINGS_B.index((Q_PUBLISHED, False))
        ceiling_rows = []
        for contrast, stage in sensitivity.items():
            j_ceiling, j_index = B_STATS.index('rstar_ceiling_human'), B_STATS.index('index')
            loo = stage['leave one Lake donor out'][:, published_setting, :]
            d_low, d_high, _ = percentile_interval(stage['donor'][:, published_setting, j_ceiling][:, None])
            point = stage['point'][published_setting]
            _, i_high, _ = percentile_interval(stage['gene'][:, published_setting, j_index][:, None])
            ceiling_rows.append({'contrast': contrast, 'human ceiling': point[j_ceiling],
                                 'leave-one-donor-out min': loo[:, j_ceiling].min(), 'leave-one-donor-out max': loo[:, j_ceiling].max(),
                                 'donor bootstrap low': d_low[0], 'donor bootstrap high': d_high[0],
                                 'index, leave-one-donor-out range': f'{loo[:, j_index].min():.3f}–{loo[:, j_index].max():.3f}',
                                 'index upper bound': i_high[0],
                                 'human ceiling at which the upper bound reaches 0.5': point[j_ceiling] * (i_high[0] / INDEX_STRONG) ** 2,
                                 'human datasets giving a ceiling': 1 if contrast == 'S2-S1' else 2})
        human_ceiling = pd.DataFrame(ceiling_rows)
        human_ceiling.to_csv(tables / 'human_ceiling_dependence.csv', index=False)
        display(human_ceiling.round(3))

    # Figure: correlations, index and direction agreement by strength bin.
    CROSS, MUTED, ACCENT = '#009E73', '#9A9A9A', '#AA4499'
    TITLES = {'S2-S1': 'S2 − S1', 'S3-early': 'S3 − early', 'S3c-early': 'S3c − early (region-matched)'}
    X = np.arange(len(BINS))

    def curve(ax, contrast, stat, color, label, *, scheme='max rank (primary)', dashed=False, offset=0., band=True):
        block = curves[curves.contrast.eq(contrast) & curves.scheme.eq(scheme) & curves.q.eq(Q_PUBLISHED)
                       & curves.bootstrap.eq('gene') & curves.statistic.eq(stat)].set_index('subset').reindex(BINS)
        if band:
            ax.fill_between(X + offset, block.ci_low.astype(float), block.ci_high.astype(float), color=color, alpha=.14, lw=0)
        ax.plot(X + offset, block.value.astype(float), color=color, lw=2, ls='--' if dashed else '-', marker='o', ms=4, label=label)
        return block.value.astype(float).to_numpy()

    fig, axes = plt.subplots(3, 3, figsize=(13, 10.5), sharex=True, gridspec_kw={'height_ratios': [1.15, 1, .8]})
    for col, contrast in enumerate(TITLES):
        ax = axes[0, col]
        ceiling_h = curve(ax, contrast, 'rstar_ceiling_human', HUMAN, 'within human: ours vs Lake')
        curve(ax, contrast, 'rstar_ceiling_mouse', MOUSE, 'within mouse: ours vs snRNA')
        curve(ax, contrast, 'rstar_cross', CROSS, 'human vs mouse: ours')
        curve(ax, contrast, 'rstar_external', CROSS, 'human vs mouse: Lake vs snRNA', dashed=True, band=False)
        for i in np.flatnonzero(np.isnan(ceiling_h)):
            ax.text(X[i], -.27, 'human\nceiling n.e.', color=HUMAN, ha='center', va='bottom', fontsize=6.5)
        ax.axhline(0, color=MUTED, lw=.6)
        ax.set_ylim(-.3, 1.25)
        ax.set_title(f'{"abc"[col]} · {TITLES[contrast]}', loc='left')
        ax = axes[1, col]
        index_values = curve(ax, contrast, 'index', INK, 'conservation index: ours')
        curve(ax, contrast, 'index_external', ACCENT, 'conservation index: external atlases', dashed=True, band=False)
        for i in np.flatnonzero(np.isnan(index_values)):
            ax.text(X[i], .02, 'n.e.', color=MUTED, ha='center', va='bottom', fontsize=7)
        ax.axhline(INDEX_STRONG, color=MUTED, lw=.8, ls=':')
        ax.axhline(0, color=MUTED, lw=.6)
        ax.set_ylim(-.25, 1.05)
        ax.set_title(f'{"def"[col]} · conservation index', loc='left')
        ax = axes[2, col]
        block = direction_agreement[direction_agreement.contrast.eq(contrast)].set_index('bin').reindex(BINS)
        block.loc[block['zonated in both'] < 10, ['share conserved', 'share low', 'share high']] = np.nan
        ax.errorbar(X, block['share conserved'], yerr=[block['share conserved'] - block['share low'], block['share high'] - block['share conserved']],
                    fmt='o-', color=INK, lw=1.6, ms=4, capsize=0, label='zonated in both: same direction')
        ax.plot(X, block['raw sign agreement (all genes)'], color=MUTED, ls='--', marker='o', ms=3, lw=1.4, label='all genes: same sign')
        ax.axhline(.5, color=MUTED, lw=.6, ls=':')
        ax.set_ylim(.25, 1.02)
        ax.set_title(f'{"ghi"[col]} · direction agreement', loc='left')
        ax.set_xticks(X, BINS, rotation=30)
        ax.set_xlabel('Gradient-strength bin (percentile, cross-fitted)')
    axes[0, 0].set_ylabel('Noise-corrected correlation r*')
    axes[1, 0].set_ylabel('Index = r*(h, m) ÷ √(ceiling_h · ceiling_m)')
    axes[2, 0].set_ylabel('Share in the same direction')
    axes[0, 0].legend(frameon=False, fontsize=7, loc='lower right')
    fig.tight_layout()
    save_figure(fig, figures, 'fig_conservation_by_strength')
    plt.show()
    plt.close(fig)

    headline = {}
    for contrast in index_genes:
        block = curves[curves.contrast.eq(contrast) & curves.scheme.eq('max rank (primary)') & curves.q.eq(Q_PUBLISHED) & curves.bootstrap.eq('gene')]
        for subset in ('BOTTOM (<80%)', 'TOP (≥95%)', BINS[-1], 'all genes'):
            for stat in ('rstar_cross', 'rstar_ceiling_human', 'rstar_ceiling_mouse', 'index', 'index_external'):
                v = block[block.subset.eq(subset) & block.statistic.eq(stat)].iloc[0]
                headline[f'{contrast} · {subset} · {stat}'] = f'{v.value:.3f} ({v.ci_low:.3f}, {v.ci_high:.3f})'
    decisions = pd.Series({'part 43 reproduced': True, 'rule (i) concentration (S2−S1 and S3−early, ' + ' and '.join(BOOTS) + ')': RULE_I,
                           'rule (ii) BOTTOM reproducible but not shared': RULE_II,
                           'BOTTOM not reliably zonated (any primary contrast)': NOT_ZONATED, 'title verdict': TITLE_VERDICT,
                           '"largely different genes" stands under every sensitivity': WORDING_STANDS,
                           'max upper bound, primary index (S2−S1, S3−early)': float(primary_b.ci_high.max()) if full else np.nan,
                           'genes in "reproducible, not shared" bins': gene_counts.loc['reproducible, not shared'].to_dict()
                           if 'reproducible, not shared' in gene_counts.index else {},
                           'genes in "not reliably zonated" bins': gene_counts.loc['not reliably zonated'].to_dict()
                           if 'not reliably zonated' in gene_counts.index else {}})
    decisions.to_csv(tables / 'decisions.csv')
    display(decisions.to_frame('value'))
    (tables / 'run_manifest.json').write_text(json.dumps({'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'full': full,
                                                           'lake_folds': lake_fold.to_dict(), 'snRNA_male_folds': sn_fold.to_dict(),
                                                           'decisions': decisions.astype(str).to_dict(), 'headline': headline},
                                                          indent=2, default=str))
    return {k: v for k, v in locals().items() if not k.startswith('__')}

## 6 · Notebook 46: does the claim survive other labels? (cross-fitted)

- **Labels.** They are transferred by notebook 42's centred score, using one sha256 gene half's reference-zonated
  genes, and evaluated only on the other half; then the halves swap.
  - (i): human Lake, mouse package labels.
  - (ii): human Lake, mouse microdissection.
  - (iii): package labels on the halves, which isolates the effect of halving genes.
- **Claim stands.** The primary index is ≤ 0.5 with an upper bound < 0.6, for S2 − S1 and S3 − early, in every
  setting and half.
- **Unskewed.** The mouse-only ÷ human-only ratio is < 2 in S2 − S1 and S3c − early.
- **Guard (G6).** The package labels on all genes reproduce part 43.

In [ ]:
from pseudospace.coordinate_evaluation import sha256_folds


def part46(I, R, out):
    """Notebook 46 on part 43's objects; returns its tables and the evaluation function for part 47."""
    tables, _ = folders(out, '46')
    label_dir = tables / 'label_sets'
    label_dir.mkdir(exist_ok=True)
    gene_space, counts, library, specimen, depth = R['gene_space'], R['counts'], R['library'], R['specimen'], R['depth']
    species = I['obs'].species.astype(str).to_numpy()
    REVIEWED = R['segment']
    lake_counts, lake_totals, sn_counts, sn_totals = R['lake_counts'], R['lake_totals'], R['sn_counts'], R['sn_totals']
    male_sn, female_sn = R['male_sn'], R['female_sn']
    HALF = pd.Series(sha256_folds(gene_space, 2), index=gene_space)
    print(f'{len(gene_space):,} genes; halves {HALF.value_counts().to_dict()}')

    def donor_gradient(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
        return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])

    def reference_table(count_frame, totals, donors, contrast):
        per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
        per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
        return moderated_contrast(per_donor, level.mean())

    REF_GRADIENT = {(c, 'Lake'): donor_gradient(lake_counts, lake_totals, healthy_lake, 'S3-early' if c == 'S3c-early' else c)
                    for c in CONTRASTS}
    REF_GRADIENT.update({(c, 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, 'S3-early' if c == 'S3c-early' else c)
                         for c in CONTRASTS})
    REF_TABLE = {(c, k): reference_table(*args, c) for c in CONTRASTS for k, args in
                 (('mouse male', (sn_counts, sn_totals, male_sn)), ('mouse female', (sn_counts, sn_totals, female_sn)),
                  ('human', (lake_counts, lake_totals, healthy_lake)))}

    def evaluate(segment, eval_genes, tag):
        """Primary conservation index and symmetric class counts (notebook 43) for given labels and genes."""
        eval_genes = set(eval_genes)
        cortical_s3 = np.zeros(len(specimen), bool)
        for name in expected['mouse']:
            limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
            cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
        region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
        names_, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), region.astype(str)))
        frame = pd.DataFrame(sums, index=names_, columns=gene_space)
        pb, pb_library = {}, {}
        for name in names:
            for label, members in PARTS.items():
                rows = [f'{name}|{s}' for s in members if f'{name}|{s}' in frame.index]
                pb[(name, label)] = frame.loc[rows].sum()
                pb_library[(name, label)] = float(library[(specimen == name) & np.isin(region, members)].sum())
        pb = pd.DataFrame(pb)
        ours = pd.DataFrame({(contrast, name): log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
                             for contrast, (late, early) in CONTRASTS.items() for name in names}, index=gene_space)

        def count_filter(specimens, members):
            return pb[[(n, p) for n in specimens for p in members]].min(axis=1).ge(MIN_COUNTS)

        index_rows, count_rows, amplitude = [], [], {}
        for contrast, (late, early) in CONTRASTS.items():
            sets = {}
            for sp in expected:
                expressed = set(gene_space[count_filter(expected[sp], [late, early]).to_numpy()])
                sets[sp] = (paired_gradient(ours[[(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)), expressed)
            lake_g, snm_g = REF_GRADIENT[(contrast, 'Lake')], REF_GRADIENT[(contrast, 'snRNA male')]
            genes = pd.Index(sorted(sets['human'][1] & sets['mouse'][1] & lake_g[1] & snm_g[1] & eval_genes))
            index_rows.append({'labels': tag, 'contrast': contrast,
                               **conservation_index(sets['human'][0], sets['mouse'][0], lake_g[0], snm_g[0], genes, n_boot=N_BOOT, seed=SEED)})
            both = gene_space[(count_filter(expected['mouse'], [late, early]) & count_filter(expected['human'], [late, early])).to_numpy()]
            tabs = {}
            for sp in expected:
                replicates = ours.loc[both, [(contrast, n) for n in expected[sp]]].T.reset_index(drop=True)
                level = pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in expected[sp] for p in (late, early)], axis=1).mean(axis=1)
                tabs[sp] = moderated_contrast(replicates, level)
            amp_genes = [g for g in tabs['mouse'].index if g in eval_genes]
            reps = {sp: ours.loc[amp_genes, [(contrast, n) for n in expected[sp]]].to_numpy() for sp in expected}
            signal = {sp: r.mean(axis=1).var(ddof=1) - r.var(axis=1, ddof=1).mean() / r.shape[1] for sp, r in reps.items()}
            amplitude[contrast] = float(np.sqrt(max(signal['human'], 0) / signal['mouse']))
            mouse_calls, _ = zonation_calls(tabs['mouse'], alpha=ALPHA, effect=EFFECT, margin=MARGIN)
            mouse_calls = confirm(mouse_calls, REF_TABLE[(contrast, 'mouse male')], alpha=CONFIRM_ALPHA)
            human_calls, _ = zonation_calls(tabs['human'], alpha=ALPHA, effect=EFFECT * amplitude[contrast], margin=MARGIN * amplitude[contrast])
            human_calls = confirm(human_calls, REF_TABLE[(contrast, 'human')], alpha=CONFIRM_ALPHA)
            classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
            classes = confirm_flat(classes, mouse_calls, human_calls, REF_TABLE[(contrast, 'mouse male')], REF_TABLE[(contrast, 'human')],
                                   alpha=CONFIRM_ALPHA, effect=EFFECT)
            counted = classes[classes.index.isin(balanced) & classes.index.isin(eval_genes)].value_counts()
            count_rows.append({'labels': tag, 'contrast': contrast, 'amplitude human/mouse': amplitude[contrast],
                               **{k: int(counted.get(k, 0)) for k in ('conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither')}})
            count_rows[-1]['mouse-only ÷ human-only'] = count_rows[-1]['mouse-only'] / max(count_rows[-1]['human-only'], 1)
        return pd.DataFrame(index_rows), pd.DataFrame(count_rows), amplitude

    # Guard (G6): the package labels on all genes reproduce part 43 exactly.
    check_index, check_counts, check_amplitude = evaluate(REVIEWED, gene_space, f'{I["labels_name"]} labels, all genes')
    saved_index = R['conservation'][R['conservation'].version.str.startswith('primary')].set_index('contrast')
    saved_counts = R['class_counts'][R['class_counts'].setting.eq('symmetric (A3b + A8)')].set_index('contrast')
    for row in check_index.itertuples():
        for field in ('index', 'ci_low', 'ci_high', 'genes'):
            if not np.isclose(getattr(row, field), saved_index.loc[row.contrast, field], rtol=0, atol=1e-9, equal_nan=True):
                raise ValueError(f'Index does not reproduce part 43: {row.contrast} {field}')
    for _, row in check_counts.iterrows():
        for field in ('conserved', 'reversal', 'mouse-only', 'human-only'):
            if row[field] != saved_counts.loc[row['contrast'], field]:
                raise ValueError(f'Class counts do not reproduce part 43: {row["contrast"]} {field}')
    print('Guard G6: the labels on all genes reproduce part 43 (index and symmetric class counts).')

    # Label transfer on one gene half (notebook 42's centred score).
    X_all = sparse.csr_matrix(R['lognorm'], dtype=float)
    genes_all = pd.Index(gene_space)
    measured_all = np.ones(len(genes_all), bool)
    detected = {sp: np.asarray((X_all[species == sp] > 0).mean(axis=0)).ravel() for sp in expected}
    lake_means = lake_segment_means.reindex(genes_all)[['S1', 'S2', 'S3']]
    micro_means = micro_segment_means.reindex(genes_all)[['S1', 'S2', 'S3']]
    half_all = pd.Series(sha256_folds(genes_all, 2), index=genes_all)

    def transfer_centred(sp, reference, half):
        diffs = pd.concat([(reference[a] - reference[b]).abs() for a, b in (('S2', 'S1'), ('S3', 'S2'), ('S3', 'S1'))], axis=1).max(axis=1)
        keep = (measured_all & (detected[sp] >= MIN_DETECTION) & reference.notna().all(axis=1).to_numpy()
                & (reference.max(axis=1) >= MIN_REFERENCE_LEVEL).to_numpy() & (diffs >= MIN_REFERENCE_DIFFERENCE).to_numpy()
                & (half_all.to_numpy() == half))
        cols = np.flatnonzero(keep)
        rows = np.flatnonzero(species == sp)
        dense = X_all[rows][:, cols].toarray()
        for name in np.unique(specimen[rows]):
            m = specimen[rows] == name
            dense[m] -= dense[m].mean(axis=0)
        dense /= np.linalg.norm(dense, axis=1, keepdims=True).clip(1e-12)
        ref = reference.iloc[cols].to_numpy(float)
        ref = ref - ref.mean(axis=1, keepdims=True)
        ref /= np.linalg.norm(ref, axis=0, keepdims=True).clip(1e-12)
        return rows, np.array(['S1', 'S2', 'S3'])[(dense @ ref).argmax(axis=1)], len(cols)

    label_sets, transfer_rows = {}, []
    for half in (0, 1):
        h_rows, h_labels, h_genes = transfer_centred('human', lake_means, half)
        m_rows, m_labels, m_genes = transfer_centred('mouse', micro_means, half)
        for setting in ('(i) human Lake, mouse reviewed', '(ii) human Lake, mouse microdissection', '(iii) reviewed'):
            seg = REVIEWED.copy()
            if not setting.startswith('(iii)'):
                seg[h_rows] = h_labels
            if setting.startswith('(ii)'):
                seg[m_rows] = m_labels
            label_sets[(setting, half)] = seg
            stem = {'(i)': 'crossfit_i', '(ii)': 'crossfit_ii', '(iii)': 'crossfit_iii'}[setting.split(' ')[0]] + f'_labels_from_half{half}'
            pd.DataFrame({'structure': R['unit_ids'], 'segment': seg}).to_csv(label_dir / f'{stem}.csv', index=False)
            for sp in expected:
                m = species == sp
                transfer_rows.append({'setting': setting, 'labels from half': half, 'species': sp,
                                      'transfer genes': h_genes if sp == 'human' else m_genes,
                                      'agreement with reviewed': float((seg[m] == REVIEWED[m]).mean()),
                                      **{f'{s} fraction': float((seg[m] == s).mean()) for s in ('S1', 'S2', 'S3')}})
    TRANSFERS = pd.DataFrame(transfer_rows)
    TRANSFERS.to_csv(tables / 'crossfit_label_sets.csv', index=False)
    display(TRANSFERS.round(3))

    index_parts, count_parts = [], []
    for (setting, half), seg in label_sets.items():
        eval_half = 1 - half
        eval_genes = gene_space[HALF.to_numpy() == eval_half]
        idx, cnt, _ = evaluate(seg, eval_genes, setting)
        index_parts.append(idx.assign(**{'labels from half': half, 'evaluated on half': eval_half}))
        count_parts.append(cnt.assign(**{'labels from half': half, 'evaluated on half': eval_half}))
    INDEX = pd.concat(index_parts, ignore_index=True)
    COUNTS = pd.concat(count_parts, ignore_index=True)
    INDEX.to_csv(tables / 'crossfit_conservation_index.csv', index=False)
    COUNTS.to_csv(tables / 'crossfit_class_counts.csv', index=False)
    check_index.to_csv(tables / 'reproduction_reviewed_all_genes_index.csv', index=False)
    check_counts.to_csv(tables / 'reproduction_reviewed_all_genes_counts.csv', index=False)
    display(INDEX[['labels', 'labels from half', 'contrast', 'index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human',
                   'ceiling_mouse', 'genes']].round(3))
    display(COUNTS[['labels', 'labels from half', 'contrast', 'amplitude human/mouse', 'conserved', 'reversal', 'mouse-only',
                    'human-only', 'mouse-only ÷ human-only']].round(2))
    primary = INDEX[INDEX.contrast.isin(['S2-S1', 'S3-early'])]
    passes = (primary['index'] <= INDEX_MAX) & (primary.ci_high < INDEX_UPPER_MAX)
    CLAIM_STANDS = bool(passes.all())
    failing = primary.loc[~passes, ['labels', 'labels from half', 'contrast', 'index', 'ci_high']]
    skew = COUNTS[COUNTS.contrast.isin(['S2-S1', 'S3c-early'])]
    UNSKEWED = bool((skew['mouse-only ÷ human-only'] < SKEW_RATIO).all())
    decision = pd.Series({'claim "largely species-specific" stands': CLAIM_STANDS,
                          'failing settings': '; '.join(f'{r["labels"]}, labels from half {r["labels from half"]}, {r["contrast"]} '
                                                        f'(index {r["index"]:.2f}, upper {r["ci_high"]:.2f})'
                                                        for _, r in failing.iterrows()) or 'none',
                          'species-only counts unskewed (ratio < 2, S2−S1 and S3c−early)': UNSKEWED,
                          'max index (S2−S1, S3−early)': float(primary['index'].max()),
                          'max upper bound (S2−S1, S3−early)': float(primary.ci_high.max())})
    decision.to_csv(tables / 'decisions.csv')
    display(decision.to_frame('value'))
    return {k: v for k, v in locals().items() if not k.startswith('__')}

## 7 · Notebook 47: integration genes, transporters and reversals

- **(a) Without integration genes.** The evaluation genes drop the genes whose embedding was clustered into the
  labels. Notebook 46's evaluation and decision rule apply.
  - Primary gene set: the package's pass-1 Harmony features.
  - Secondary gene set: those plus the 9 PT-S1/S2/S3 panel genes, which named the labels.
- **(b, b′) Transporters.** Genes matching `^(Slc|Abc|Aqp)`, probe-balanced, on the union class of S2 − S1 and
  S3c − early.
- **(d) Integration genes by symmetric class.**
- **(f) Reversals under every label setting.**

Parts (c) and (e) read the retired notebook 37 runs and are not carried over.

In [ ]:
def part47(I, R, R46, R42, out):
    """Notebook 47 parts (a), (b), (b′), (d) and (f)."""
    tables, _ = folders(out, '47')
    evaluate, gene_space, HALF = R46['evaluate'], R['gene_space'], R46['HALF']
    REVIEWED, label_sets = R['segment'], R46['label_sets']
    labels_name = I['labels_name']
    evaluations = {(labels_name, 'all genes'): (REVIEWED, gene_space)}
    for setting, stem in (('(i) human Lake, mouse reviewed', 'i'), ('(ii) human Lake, mouse microdissection', 'ii')):
        for half in (0, 1):
            evaluations[(setting, f'labels from half {half}, evaluated on half {1 - half}')] = (label_sets[(setting, half)],
                                                                                              gene_space[HALF.to_numpy() == 1 - half])
    index_parts, count_parts, CLAIM_STANDS = [], [], {}
    for arm, set_name in I['integration_arms'].items():
        INTEGRATION = I['integration'][set_name]
        print(f'{arm}: {set_name}: {len(INTEGRATION & set(gene_space))} of {len(INTEGRATION)} genes are in the gene space.')
        for (setting, genes_label), (seg, genes) in evaluations.items():
            kept = pd.Index([g for g in genes if g not in INTEGRATION])
            idx, cnt, _ = evaluate(seg, kept, setting)
            index_parts.append(idx.assign(evaluation=genes_label, **{'removed set': set_name, 'arm': arm,
                                                                     'integration genes removed': len(genes) - len(kept)}))
            count_parts.append(cnt.assign(evaluation=genes_label, **{'removed set': set_name, 'arm': arm}))
    INDEX = pd.concat(index_parts, ignore_index=True)
    COUNTS = pd.concat(count_parts, ignore_index=True)
    INDEX.to_csv(tables / 'no_integration_genes_conservation_index.csv', index=False)
    COUNTS.to_csv(tables / 'no_integration_genes_class_counts.csv', index=False)
    display(INDEX[['arm', 'labels', 'evaluation', 'contrast', 'index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human',
                   'ceiling_mouse', 'genes', 'integration genes removed']].round(3))
    display(COUNTS[['arm', 'labels', 'evaluation', 'contrast', 'amplitude human/mouse', 'conserved', 'reversal', 'mouse-only',
                    'human-only', 'mouse-only ÷ human-only']].round(2))
    for arm in I['integration_arms']:
        primary = INDEX[INDEX.contrast.isin(['S2-S1', 'S3-early']) & INDEX.arm.eq(arm)]
        passes = (primary['index'] <= INDEX_MAX) & (primary.ci_high < INDEX_UPPER_MAX)
        CLAIM_STANDS[arm] = bool(passes.all())
        print(f'(a) {arm}: claim "largely species-specific" without integration genes: {"stands" if CLAIM_STANDS[arm] else "weakened"} '
              f'(max index {primary["index"].max():.3f}, max upper bound {primary.ci_high.max():.3f})')

    # (b) Transporter shares among zonated genes.
    symmetric = R['symmetric'].assign(probe_balanced=R['symmetric'].index.isin(balanced))
    union_column = UNION
    balanced_classes = symmetric[symmetric.probe_balanced.astype(bool)]
    family = balanced_classes.index.str.match(TRANSPORTER_REGEX)
    rows = []
    for label, frame in (('Slc/Abc/Aqp', balanced_classes[family]), ('all genes', balanced_classes)):
        counts_ = frame[union_column].value_counts()
        zonated = int(sum(counts_.get(k, 0) for k in ZONATED_CLASSES))
        rows.append({'genes': label, 'probe-balanced genes': len(frame), **{k: int(counts_.get(k, 0)) for k in ZONATED_CLASSES},
                     'zonated': zonated, 'conserved share of zonated': counts_.get('conserved', 0) / zonated if zonated else np.nan})
    TRANSPORTERS = pd.DataFrame(rows)
    TRANSPORTERS.to_csv(tables / 'transporter_class_shares.csv', index=False)
    balanced_classes[family & balanced_classes[union_column].isin(ZONATED_CLASSES)][[union_column]].to_csv(tables / 'transporter_zonated_genes.csv')
    display(TRANSPORTERS.round(3))

    # (b′) Direction agreement among genes zonated in both species.
    def agreement(frame, label):
        c = frame[union_column].value_counts()
        zonated = int(sum(c.get(k, 0) for k in ZONATED_CLASSES))
        both = int(c.get('conserved', 0) + c.get('reversal', 0))
        return {'genes': label, 'zonated': zonated, 'zonated in both species': both,
                'zonated-in-both rate': both / zonated if zonated else np.nan,
                'conserved': int(c.get('conserved', 0)), 'reversal': int(c.get('reversal', 0)),
                'direction agreement among zonated in both': c.get('conserved', 0) / both if both else np.nan}

    AGREEMENT = pd.DataFrame([agreement(balanced_classes[family], 'Slc/Abc/Aqp'), agreement(balanced_classes, 'all genes')])
    conserved_all = balanced_classes[union_column].eq('conserved')
    AGREEMENT['transporter share of the conserved class'] = [(family & conserved_all).sum() / max(conserved_all.sum(), 1), np.nan]
    AGREEMENT.to_csv(tables / 'transporter_direction_agreement.csv', index=False)
    display(AGREEMENT.round(3))

    # (d) Integration genes by symmetric class.
    enrichment_rows = []
    for scope, frame in (('all classified genes', symmetric), ('probe-balanced', balanced_classes)):
        for set_label, genes_ in I['integration'].items():
            member = frame.index.isin(sorted(genes_))
            for klass, size in frame[union_column].value_counts().items():
                in_class = frame[union_column].eq(klass)
                enrichment_rows.append({'scope': scope, 'gene set': set_label, 'class': klass, 'class size': int(size),
                                        'integration genes': int((in_class & member).sum()),
                                        'share of class': float((in_class & member).sum() / size)})
    ENRICHMENT = pd.DataFrame(enrichment_rows)
    ENRICHMENT.to_csv(tables / 'integration_genes_by_symmetric_class.csv', index=False)
    display(ENRICHMENT.pivot_table(index='class', columns=['scope', 'gene set'], values='share of class').round(3))

    # (f) Reversal genes under each label setting (symmetric setting, probe-balanced genes).
    reversal_rows = []
    frame = R['class_counts'][R['class_counts'].setting.eq('symmetric (A3b + A8)')].set_index('contrast')
    reversal_rows.append({'labels': f'{labels_name} labels, all genes', 'evaluation genes': 'all',
                          **{f'{c} reversal': int(frame.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                          **{f'{c} conserved': int(frame.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
    alternative = R42['labels_table'].set_index('structure_id').reindex(R['unit_ids'])
    human = alternative.species.eq('human').to_numpy()
    transferred = {'transferred (full-gene, circular): human Lake, mouse reviewed':
                       np.where(human, alternative.transfer_lake_centred, REVIEWED),
                   'transferred (full-gene, circular): human Lake, mouse microdissection':
                       np.where(human, alternative.transfer_lake_centred, alternative.transfer_microdissection_centred)}
    for description, seg in transferred.items():
        _, block, _ = evaluate(np.asarray(seg).astype(str), gene_space, description)
        block = block.set_index('contrast')
        reversal_rows.append({'labels': description, 'evaluation genes': 'all',
                              **{f'{c} reversal': int(block.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                              **{f'{c} conserved': int(block.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
    for (setting, half), block in R46['COUNTS'].groupby(['labels', 'labels from half'], sort=False):
        block = block.set_index('contrast')
        reversal_rows.append({'labels': f'cross-fit {setting}', 'evaluation genes': f'half {1 - half} (labels from half {half})',
                              **{f'{c} reversal': int(block.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                              **{f'{c} conserved': int(block.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
    for (arm, setting, evaluation), block in COUNTS.groupby(['arm', 'labels', 'evaluation'], sort=False):
        block = block.set_index('contrast')
        reversal_rows.append({'labels': f'{setting}, without {I["integration_arms"][arm]}', 'evaluation genes': evaluation,
                              **{f'{c} reversal': int(block.loc[c, 'reversal']) for c in ('S2-S1', 'S3c-early')},
                              **{f'{c} conserved': int(block.loc[c, 'conserved']) for c in ('S2-S1', 'S3c-early')}})
    REVERSALS = pd.DataFrame(reversal_rows)
    REVERSALS.to_csv(tables / 'reversal_label_sensitivity.csv', index=False)
    display(REVERSALS)
    return {k: v for k, v in locals().items() if not k.startswith('__')}

## 8 · Notebook 48 part B: pathway enrichment of the symmetric classes

Notebook 40's class-enrichment pipeline, unchanged, on part 43's symmetric union classes.
- **Universe and sets.** The classified genes; pathways with 10–300 classified members.
- **Strata.** 3 bins (5 as sensitivity) of mean expression, detection and minimum species detection over all PT
  units.
- **CAMERA inflation.** From residual correlations after removing the specimen × segment-region means.
- **Tests.** 9,999 matched draws (seed 40), then the joint test.
- **Calls.** Enriched means q ≤ 0.10 with a positive effect. Not expression-driven means also q ≤ 0.10 in the
  5-bin and upper-expression runs.
- **Naming rule** (probe-balanced run): q ≤ 0.10, not expression-driven, and ≥ 3 class genes.

Notebook 48's guard against notebook 40's saved table, and its claim table, rest on notebook 40's classes and are
not carried over.

In [ ]:
from pseudospace.pathway_decoys import joint_matched_test, overlap_programs
from pseudospace.pathway_remodeling import (correlation_adjusted_rank_tests, matched_pathway_tests, matching_strata,
                                            residual_pathway_correlations)
from pseudospace.pathways import build_pathway_membership

CLASS_LABELS = ('conserved', 'reversal', 'mouse-only', 'human-only')
PATHWAY_LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')


def part48B(I, R, out):
    """Notebook 48 part B (notebook 40's enrichment on the symmetric classes), all classified and probe-balanced genes."""
    tables, _ = folders(out, '48B')
    cache_ = out / 'stage_cache' / '48B'
    code = digest([NOTEBOOK_LOGIC_VERSION, Path(matched_pathway_tests.__code__.co_filename),
                   Path(joint_matched_test.__code__.co_filename)])
    gene_space = R['gene_space']
    lognorm40 = sparse.csr_matrix(R['lognorm'], dtype=float)
    specimen40, segment_region = R['specimen'], R['segment_region']
    classes43 = R['symmetric'][UNION]
    universe_genes = classes43.index[classes43.notna()]
    membership40 = pd.concat([build_pathway_membership(
        json.loads((GENE_SET_DIR / f'{library}.json').read_text()),
        gene_space, ortholog_map=orthologs, library_name=library, tested=list(universe_genes), min_genes=1)
        for library in PATHWAY_LIBRARIES], ignore_index=True)
    membership40['pathway_id'] = membership40.library + '::' + membership40.pathway
    membership40 = membership40[membership40.n_tested.between(10, 300)]
    class_sets = {row.pathway_id: [g for g in row.genes_present if g in set(universe_genes)] for row in membership40.itertuples()}
    class_sets = {p: v for p, v in class_sets.items() if 10 <= len(v) <= 300}
    class_names = pd.Series({p: p.split('::', 1)[1] for p in class_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)
    structure_values = lognorm40[:, gene_space.get_indexer(universe_genes)]
    mean_expression = pd.Series(np.asarray(structure_values.mean(axis=0)).ravel(), index=universe_genes)
    detection = pd.Series(np.asarray((structure_values > 0).mean(axis=0)).ravel(), index=universe_genes)
    min_detection = pd.Series(np.minimum(*[np.asarray((structure_values[np.isin(specimen40, expected[s])] > 0).mean(axis=0)).ravel()
                                            for s in ('mouse', 'human')]), index=universe_genes)
    covariates40 = pd.DataFrame({'mean_expression': mean_expression, 'detection': detection, 'coverage': min_detection})
    strata3, strata5 = matching_strata(covariates40, bins=3), matching_strata(covariates40, bins=5)

    def class_residual_correlations():
        dense = structure_values.toarray()
        groups = pd.Series(pd.Series(specimen40).astype(str) + '|' + pd.Series(segment_region).astype(str))
        for _, idx in groups.groupby(groups).groups.items():
            rows_ = np.asarray(idx)
            dense[rows_] -= dense[rows_].mean(axis=0)
        return residual_pathway_correlations(dense, specimen40, list(universe_genes), class_sets)

    class_correlations = cached_frame('class_residual_correlations', class_residual_correlations, root=cache_,
                                      inputs={'genes': list(universe_genes), 'sets': digest(sorted(class_sets.items())),
                                              'structures': len(specimen40), 'lognorm': digest(lognorm40)}, code=code)
    upper = mean_expression.index[mean_expression.ge(mean_expression.quantile(1 / 3))]
    upper_sets = {p: [g for g in v if g in set(upper)] for p, v in class_sets.items()}
    upper_sets = {p: v for p, v in upper_sets.items() if len(v) >= 10}
    ALL_INPUTS = {'universe': universe_genes, 'sets': class_sets, 'strata3': strata3, 'strata5': strata5, 'upper': upper,
                  'upper_sets': upper_sets, 'correlations': class_correlations}

    def class_enrichment(classes, tag, inputs=ALL_INPUTS):
        """Notebook 40, section 7, for one class assignment and one classified-gene universe."""
        universe_, sets_all = inputs['universe'], inputs['sets']
        indicator = pd.DataFrame({k: classes.loc[universe_].eq(k).astype(float) for k in CLASS_LABELS})
        vif = correlation_adjusted_rank_tests(indicator['conserved'], sets_all, inputs['correlations']).set_index('pathway_id').variance_inflation
        enrich = {}
        for label, frame_, sets_, strata_ in (('primary', indicator, sets_all, inputs['strata3']),
                                              ('5-bin', indicator, sets_all, inputs['strata5']),
                                              ('upper expression', indicator.loc[inputs['upper']], inputs['upper_sets'],
                                               pd.Series(inputs['strata3'], index=universe_).loc[inputs['upper']].to_numpy())):
            result = cached_frame(f'class_enrichment_{tag}_{label.replace(" ", "_")}',
                                  lambda frame_=frame_, sets_=sets_, strata_=strata_:
                                  matched_pathway_tests(frame_, sets_, strata_, n_null=N_NULL40, seed=SEED40), root=cache_,
                                  inputs={'indicator': frame_, 'sets': digest(sorted(sets_.items())), 'strata': strata_}, code=code)
            enrich[label] = joint_matched_test(result, vif, keys=('statistic',))
        table = enrich['primary'].copy()
        for label, column in (('5-bin', 'q_5bin'), ('upper expression', 'q_upper_expression')):
            table = table.merge(enrich[label][['pathway_id', 'statistic', 'q_joint']].rename(columns={'q_joint': column}),
                                on=['pathway_id', 'statistic'], how='left')
        table['name'] = table.pathway_id.map(class_names)
        table['class_members'] = [int(indicator.loc[sets_all[p], c].sum()) for p, c in zip(table.pathway_id, table.statistic)]
        table['class_genes'] = [', '.join(g for g in sets_all[p] if indicator.loc[g, c] == 1) for p, c in zip(table.pathway_id, table.statistic)]
        table['enriched'] = table.effect.gt(0) & table.q_joint.le(ENRICH_Q)
        table['not_expression_driven'] = table.enriched & table.q_5bin.le(ENRICH_Q) & table.q_upper_expression.le(ENRICH_Q)
        groups_ = []
        for klass in CLASS_LABELS:
            ids = table.loc[table.statistic.eq(klass) & table.enriched, 'pathway_id'].tolist()
            if ids:
                members = set(indicator.index[indicator[klass].eq(1)])
                groups_.append(overlap_programs(sets_all, ids, contributing=members, cut=.5).assign(statistic=klass))
        groups_ = pd.concat(groups_, ignore_index=True) if groups_ else pd.DataFrame(columns=['pathway_id', 'program', 'statistic'])
        return table.merge(groups_, on=['pathway_id', 'statistic'], how='left'), indicator

    enrichment43, indicator43 = class_enrichment(classes43, 'symmetric')
    enrichment43.sort_values(['statistic', 'q_joint']).to_csv(tables / 'class_pathway_enrichment_symmetric.csv', index=False)

    # Probe-balanced symmetric classes, with the declared naming rule (notebook 48, round 2b).
    balanced_genes = universe_genes[universe_genes.isin(balanced)]
    sets_b = {p: [g for g in v if g in set(balanced_genes)] for p, v in class_sets.items()}
    sets_b = {p: v for p, v in sets_b.items() if 10 <= len(v) <= 300}
    expression_b = mean_expression.loc[balanced_genes]
    upper_b = expression_b.index[expression_b.ge(expression_b.quantile(1 / 3))]
    upper_sets_b = {p: [g for g in v if g in set(upper_b)] for p, v in sets_b.items()}
    upper_sets_b = {p: v for p, v in upper_sets_b.items() if len(v) >= 10}

    def balanced_residual_correlations():
        dense = structure_values[:, universe_genes.get_indexer(balanced_genes)].toarray()
        groups = pd.Series(pd.Series(specimen40).astype(str) + '|' + pd.Series(segment_region).astype(str))
        for _, idx in groups.groupby(groups).groups.items():
            rows_ = np.asarray(idx)
            dense[rows_] -= dense[rows_].mean(axis=0)
        return residual_pathway_correlations(dense, specimen40, list(balanced_genes), sets_b)

    inputs_b = {'universe': balanced_genes, 'sets': sets_b, 'strata3': matching_strata(covariates40.loc[balanced_genes], bins=3),
                'strata5': matching_strata(covariates40.loc[balanced_genes], bins=5), 'upper': upper_b, 'upper_sets': upper_sets_b,
                'correlations': cached_frame('class_residual_correlations_balanced', balanced_residual_correlations, root=cache_,
                                             inputs={'genes': list(balanced_genes), 'sets': digest(sorted(sets_b.items())),
                                                     'structures': len(specimen40), 'lognorm': digest(lognorm40)}, code=code)}
    enrichment_b, _ = class_enrichment(classes43, 'symmetric_balanced', inputs_b)
    enrichment_b.sort_values(['statistic', 'q_joint']).to_csv(tables / 'class_pathway_enrichment_symmetric_probe_balanced.csv', index=False)
    summary_classes = pd.DataFrame({
        'genes, symmetric': classes43.loc[universe_genes].value_counts().reindex(CLASS_LABELS),
        'genes, symmetric, probe-balanced': classes43.loc[balanced_genes].value_counts().reindex(CLASS_LABELS),
        'enriched pathways, symmetric': enrichment43.groupby('statistic').enriched.sum().reindex(CLASS_LABELS),
        'not expression-driven, symmetric': enrichment43.groupby('statistic').not_expression_driven.sum().reindex(CLASS_LABELS),
        'enriched pathways, probe-balanced': enrichment_b.groupby('statistic').enriched.sum().reindex(CLASS_LABELS)})
    summary_classes.to_csv(tables / 'class_enrichment_counts.csv')
    display(summary_classes)
    naming_rows = []
    for run, table_ in (('all classified genes', enrichment43), ('probe-balanced', enrichment_b)):
        for klass in ('mouse-only', 'human-only', 'conserved', 'reversal'):
            block_ = table_[table_.statistic.eq(klass) & table_.enriched]
            counting = block_[block_.not_expression_driven & block_.class_members.ge(3)]
            naming_rows.append({'run': run, 'class': klass, 'enriched (q ≤ 0.10)': len(block_),
                                'not expression-driven': int(block_.not_expression_driven.sum()),
                                'counting (also ≥ 3 class genes)': len(counting),
                                'counting pathways': '; '.join(f"{r['name']} ({r['class_genes']})" for _, r in counting.iterrows())})
    naming = pd.DataFrame(naming_rows)
    naming.to_csv(tables / 'declared_naming_rule.csv', index=False)
    display(naming)
    species_only_b = naming[naming.run.eq('probe-balanced') & naming['class'].isin(['mouse-only', 'human-only'])]
    NAMING_DECISION = ('no pathway named for species-only zonation'
                       if species_only_b['counting (also ≥ 3 class genes)'].sum() == 0 else
                       'name only: ' + ' | '.join(f"{r['class']}: {r['counting pathways']}" for _, r in species_only_b.iterrows()
                                                  if r['counting (also ≥ 3 class genes)'] > 0))
    print('Declared rule (probe-balanced):', NAMING_DECISION)
    return {k: v for k, v in locals().items() if not k.startswith('__')}

## 9 · Notebook 64: does the pairing inside paralog families change the species results?

Notebook 43's primary analysis is a function of the ortholog pair table. Counts come from the native stage-01
matrices (every gene, native symbols) through `rebuild_pt_expression`; with the paper map this reproduces the
export (guard G5) and part 43 (guard G6). Notebook 49's top strength bins are added.
- **Families.** Notebook 62's orthogroups: HCOP support ≥ 3, classes 1:many, many:1 and many:many.
- **Main PT paralog.** The highest pooled CP10k among PT-expressed, probe-clean members.
  - PT-expressed means detected in ≥ 2% of units in both specimens of the species.
  - The expression is recomputed on the package PT units, against each unit's native panel total without the
    mt-encoded genes.
- **(a) Swap.** The main PT paralog replaces the least-expressed kept member.
- **(b) Drop.** Every pair in such a family is removed.
- **Unaffected.** Every index changes by < 0.03, and each zonated class count by < 10% (≤ 2 genes when the count
  is < 20).

In [ ]:
from pseudospace.pathway_inputs import rebuild_pt_expression


def pt_expression_inventory(pt_obs, sources, exclude_patterns=(r'^mt-', r'^MT-')):
    """Notebook 62's PT-expression columns on the given units: per-specimen detection and pseudobulk CP10k against
    each unit's native panel total (minus genes matching ``exclude_patterns``); pt_expressed, log2_cp10k."""
    tables = {}
    for species, specimens in expected.items():
        parts = {}
        for name in specimens:
            ids = pt_obs.index[pt_obs['sample'].astype(str).eq(name)].str.slice(len(name) + 1)
            native = ad.read_h5ad(sources[name])
            native = native[ids.to_numpy()].copy()
            X = sparse.csr_matrix(native.layers.get('counts', native.X), dtype=float)
            panel = pd.Index(native.var_names.astype(str))
            kept = ~panel.str.contains('|'.join(exclude_patterns)) if exclude_patterns else np.ones(len(panel), bool)
            L = np.asarray(X[:, kept].sum(axis=1)).ravel()
            parts[name] = pd.DataFrame({'detection': np.asarray((X > 0).mean(axis=0)).ravel(),
                                        'pseudobulk': np.asarray(X.sum(axis=0)).ravel() / L.sum() * 1e4}, index=panel)
        a, b = (parts[n] for n in specimens)
        if not a.index.equals(b.index):
            raise ValueError(f'{species}: the two specimens have different panels.')
        table = pd.DataFrame({'detection_1': a.detection, 'cp10k_1': a.pseudobulk, 'detection_2': b.detection, 'cp10k_2': b.pseudobulk})
        table['pt_expressed'] = table[['detection_1', 'detection_2']].min(axis=1).ge(PT_DETECTION)
        table['log2_cp10k'] = np.log2((table.cp10k_1 + table.cp10k_2) / 2 + PSEUDOCOUNT62)
        tables[species] = table
    return tables


def part64(I, R, R49, out):
    """Notebook 64: notebook 43's primary analysis and notebook 49's top bins under the swap and drop pairings."""
    tables, _ = folders(out, '64')
    cache_ = out / 'stage_cache' / '64'
    pt_obs, sources = I['native']['pt_obs'], I['native']['sources']
    depth_all = pd.Series(np.asarray(I['depth'], float), index=I['obs'].index)
    paper_map = orthologs

    def probe_table_for(pairs):
        return pairing.probe_balance(pairs, mouse_probe_count, human_probe_count, I['mouse_gene_ids'], I['human_gene_ids'])

    D2_CONTRASTS = ('S2-S1', 'S3-early', 'S3c-early')
    CLASS_CONTRASTS = ('S2-S1', 'S3c-early', 'S3-early')

    def primary_analysis(pairs, label):
        """Notebook 43's conservation index and symmetric classes for one pair table (notebook 64)."""
        adata = rebuild_pt_expression(pt_obs, sources, pairs, target_sum=1e4)
        measured = adata.var.measured_in_both_inputs.to_numpy() & ~adata.var_names.isin(I['exclude_genes'])
        gene_space = adata.var_names[measured]
        counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
        specimen = adata.obs['sample'].astype(str).to_numpy()
        segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
        library = np.asarray(counts.sum(axis=1)).ravel()
        depth = depth_all.reindex(adata.obs_names).to_numpy(float)
        segment_region, _ = segment_regions(specimen, segment, depth)
        names_, sums = group_sums(counts, np.char.add(np.char.add(specimen.astype(str), '|'), segment_region.astype(str)))
        frame = pd.DataFrame(sums, index=names_, columns=gene_space)
        ours_counts = {}
        for name in np.unique(specimen):
            rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
            ours_counts[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
            ours_counts[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
            ours_counts[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
            ours_counts[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
            ours_counts[(name, 'early')] = ours_counts[(name, 'S1')] + ours_counts[(name, 'S2')]
        ours_counts = pd.DataFrame(ours_counts)
        ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(segment_region, PARTS[key[1]])].sum())
                                  for key in ours_counts.columns})
        ours = pd.DataFrame({(contrast, name): log_ratio(ours_counts[(name, late)], ours_library[(name, late)],
                                                         ours_counts[(name, early)], ours_library[(name, early)])
                             for contrast, (late, early) in CONTRASTS.items() for name in names}, index=ours_counts.index)

        def count_filter(specimens, parts):
            return ours_counts[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)

        ref = reference_frames(gene_space, pairs, I['human_gene_ids'])
        micro_male, sn_counts, sn_totals, sn_sex = ref['micro_male'], ref['sn_counts'], ref['sn_totals'], ref['sn_sex']
        census_h_counts, census_h_totals, census_h_sex = ref['census_h_counts'], ref['census_h_totals'], ref['census_h_sex']
        lake_counts, lake_totals = ref['lake_counts'], ref['lake_totals']

        def ours_gradient(contrast, specimens):
            late, early = CONTRASTS[contrast]
            expressed_ = set(gene_space[count_filter(specimens, [late, early]).to_numpy()])
            return paired_gradient(ours[[(contrast, n) for n in specimens]].T.reset_index(drop=True)), expressed_

        def donor_gradient(count_frame, totals, donors, contrast):
            per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
            keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
            return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])

        def census_human_gradient():
            donors = census_h_sex.index
            late = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}).T
            early = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}).T
            keep = np.maximum(late.mean(), early.mean()).ge(REFERENCE_LEVEL) & (late - early).notna().all()
            return paired_gradient((late - early).loc[:, keep]), set(keep.index[keep.to_numpy()])

        micro_male_segments = {s: micro_male[[c for c in micro_male.columns if c[0] == s]].T.reset_index(drop=True) for s in ('S1', 'S2', 'S3')}

        def segment_replicates_gradient(by_segment, contrast):
            late, early = CONTRASTS[contrast]
            late = 'S3' if late == 'S3c' else late
            if early == 'early':
                parts = [by_segment['S1'], by_segment['S2']]
                mean = by_segment[late].mean() - (parts[0].mean() + parts[1].mean()) / 2
                noise = by_segment[late].var(ddof=1) / len(by_segment[late]) + sum(p.var(ddof=1) / len(p) for p in parts) / 4
                level = pd.concat([by_segment[late].mean(), (parts[0].mean() + parts[1].mean()) / 2], axis=1).max(axis=1)
            else:
                mean, noise = unpaired_gradient(by_segment[late], by_segment[early])
                level = pd.concat([by_segment[late].mean(), by_segment[early].mean()], axis=1).max(axis=1)
            keep = level.ge(REFERENCE_LEVEL) & mean.notna()
            return (mean, noise), set(keep.index[keep.to_numpy()])

        male_sn = sn_sex.index[sn_sex.eq('male')]

        def datasets_for(contrast):
            human_contrast = 'S3-early' if contrast == 'S3c-early' else contrast
            out_ = {('human', 'ours'): ours_gradient(contrast, expected['human']),
                    ('mouse', 'ours'): ours_gradient(contrast, expected['mouse']),
                    ('human', 'Lake healthy'): donor_gradient(lake_counts, lake_totals, healthy_lake, human_contrast),
                    ('mouse', 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, human_contrast),
                    ('mouse', 'micro male GSE150338'): segment_replicates_gradient(micro_male_segments, human_contrast)}
            if contrast != 'S2-S1':
                out_[('human', 'Census human')] = census_human_gradient()
            return out_

        datasets = {contrast: datasets_for(contrast) for contrast in D2_CONTRASTS}
        index_rows = []
        for contrast, data in datasets.items():
            h, m, l, s = data[('human', 'ours')], data[('mouse', 'ours')], data[('human', 'Lake healthy')], data[('mouse', 'snRNA male')]
            genes = pd.Index(sorted(h[1] & m[1] & l[1] & s[1]))
            index_rows.append({'contrast': contrast, 'version': 'primary: ours human vs ours mouse',
                               **conservation_index(h[0], m[0], l[0], s[0], genes, n_boot=N_BOOT, seed=SEED)})
            index_rows.append({'contrast': contrast, 'version': 'external: Lake vs mouse snRNA male',
                               **conservation_index(l[0], s[0], h[0], m[0], genes, n_boot=N_BOOT, seed=SEED)})
            micro = data[('mouse', 'micro male GSE150338')]
            if ('human', 'Census human') in data:
                c = data[('human', 'Census human')]
                genes_alt = pd.Index(sorted(l[1] & s[1] & c[1] & micro[1]))
                index_rows.append({'contrast': contrast, 'version': 'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)',
                                   **conservation_index(l[0], s[0], c[0], micro[0], genes_alt, n_boot=N_BOOT, seed=SEED)})
        conservation = pd.DataFrame(index_rows)

        def species_level(specimens, parts):
            return pd.concat([log2_cpm(ours_counts[(n, p)], ours_library[(n, p)]) for n in specimens for p in parts], axis=1).mean(axis=1)

        def reference_table(count_frame, totals, donors, contrast):
            per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
            per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
            return moderated_contrast(per_donor, level.mean())

        tables_ours, tables_ref = {}, {}
        for contrast in CLASS_CONTRASTS:
            late, early = CONTRASTS[contrast]
            both = gene_space[(count_filter(expected['mouse'], [late, early]) & count_filter(expected['human'], [late, early])).to_numpy()]
            for species in ('mouse', 'human'):
                replicates = ours.loc[both, [(contrast, n) for n in expected[species]]].T.reset_index(drop=True)
                tables_ours[(contrast, species)] = moderated_contrast(replicates, species_level(expected[species], [late, early]))
            tables_ref[(contrast, 'mouse male')] = reference_table(sn_counts, sn_totals, male_sn, contrast)
            tables_ref[(contrast, 'human')] = reference_table(lake_counts, lake_totals, healthy_lake, contrast)

        def noise_corrected_amplitude(contrast):
            genes_ = tables_ours[(contrast, 'mouse')].index

            def signal(specimens):
                reps = ours.loc[genes_, [(contrast, n) for n in specimens]].to_numpy()
                return reps.mean(axis=1).var(ddof=1) - reps.var(axis=1, ddof=1).mean() / reps.shape[1]
            return float(np.sqrt(max(signal(expected['human']), 0) / signal(expected['mouse'])))

        amplitude = {c: noise_corrected_amplitude(c) for c in CLASS_CONTRASTS}
        frame_ = {}
        for contrast in CLASS_CONTRASTS:
            scale = amplitude[contrast]
            mouse_calls, _ = zonation_calls(tables_ours[(contrast, 'mouse')], alpha=ALPHA, effect=EFFECT, margin=MARGIN)
            mouse_calls = confirm(mouse_calls, tables_ref[(contrast, 'mouse male')], alpha=CONFIRM_ALPHA)
            human_calls, _ = zonation_calls(tables_ours[(contrast, 'human')], alpha=ALPHA, effect=EFFECT * scale, margin=MARGIN * scale)
            human_calls = confirm(human_calls, tables_ref[(contrast, 'human')], alpha=CONFIRM_ALPHA)
            classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
            classes = confirm_flat(classes, mouse_calls, human_calls, tables_ref[(contrast, 'mouse male')], tables_ref[(contrast, 'human')],
                                   alpha=CONFIRM_ALPHA, effect=EFFECT)
            frame_[contrast], frame_[contrast + ' mouse call'], frame_[contrast + ' human call'] = classes, mouse_calls, human_calls
        symmetric = pd.DataFrame(frame_)
        symmetric['union S2-S1 + S3c-early'] = union_class(symmetric[['S2-S1', 'S3c-early']])
        probe_ = probe_table_for(pairs).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
        balanced_ = set(probe_.index[probe_.probe_balance.eq('balanced')])
        symmetric = symmetric.assign(probe_balanced=symmetric.index.isin(balanced_))
        ORDER = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
        rows = []
        for contrast in (*CLASS_CONTRASTS, 'union S2-S1 + S3c-early'):
            counts_ = symmetric.loc[symmetric.probe_balanced, contrast].value_counts().reindex(ORDER).fillna(0).astype(int)
            rows.append({'setting': label, 'contrast': contrast, **counts_.to_dict(),
                         'mouse-only ÷ human-only': counts_['mouse-only'] / max(counts_['human-only'], 1)})
        index_genes = {c: pd.Index(sorted(d[('human', 'ours')][1] & d[('mouse', 'ours')][1] & d[('human', 'Lake healthy')][1]
                                          & d[('mouse', 'snRNA male')][1])) for c, d in datasets.items()}
        return {'conservation': conservation.assign(setting=label), 'symmetric': symmetric, 'class_counts': pd.DataFrame(rows),
                'datasets': datasets, 'index_genes': index_genes, 'gene_space': gene_space, 'amplitude': amplitude,
                'lake_counts': lake_counts, 'lake_totals': lake_totals, 'sn_counts': sn_counts, 'sn_totals': sn_totals,
                'male_sn': male_sn, 'counts': counts}

    # Guard (G6): the baseline reproduces part 43 exactly.
    baseline = primary_analysis(paper_map, 'baseline (paper map)')
    if not (pd.Index(baseline['gene_space']).equals(pd.Index(R['gene_space']))
            and (sparse.csr_matrix(baseline['counts']) != sparse.csr_matrix(R['counts'])).nnz == 0):
        raise ValueError('The native route does not reproduce the bundle counts (guard G5).')
    published_index = R['conservation']
    FIELDS = ('index', 'ci_low', 'ci_high', 'cross_corrected', 'ceiling_human', 'ceiling_mouse', 'genes')
    merged = published_index.merge(baseline['conservation'], on=['contrast', 'version'], suffixes=('_43', ''))
    if len(merged) != len(published_index):
        raise ValueError('Conservation-index rows do not match part 43.')
    for field in FIELDS:
        if not np.allclose(merged[field].astype(float), merged[field + '_43'].astype(float), rtol=0, atol=1e-9, equal_nan=True):
            raise ValueError(f'Part 43 not reproduced: {field}')
    published_classes = R['symmetric'].assign(probe_balanced=R['symmetric'].index.isin(balanced))
    mine = baseline['symmetric'].reindex(columns=published_classes.columns)
    pd.testing.assert_frame_equal(mine.sort_index().astype(str), published_classes.sort_index().astype(str), check_names=False)
    published_counts = R['class_counts'][R['class_counts'].setting.eq('symmetric (A3b + A8)')].set_index('contrast')
    assert (baseline['class_counts'].set_index('contrast')[ZONATED_CLASSES] == published_counts[ZONATED_CLASSES]).all().all()
    print(f'Guard G6: part 43 reproduced exactly: {len(merged)} index rows, {len(published_classes):,} class rows, class counts.')

    # Notebook 49's strength curve, top bins (primary scheme, gene bootstrap).
    TOP_SUBSETS = (BINS[-1], 'TOP (≥95%)')

    def strength_top(result):
        lake_fold = pd.Series(donor_folds(healthy_lake), index=healthy_lake)
        sn_fold = pd.Series(donor_folds(list(result['male_sn'])), index=result['male_sn'])
        rows = []
        for contrast, genes in result['index_genes'].items():
            rc = 'S3-early' if contrast == 'S3c-early' else contrast
            d = result['datasets'][contrast]
            lake_pd = donor_contrasts(result['lake_counts'], result['lake_totals'], healthy_lake, rc)[0][genes]
            sn_pd = donor_contrasts(result['sn_counts'], result['sn_totals'], result['male_sn'], rc)[0][genes]

            def arrays(gradient):
                mean, noise = gradient
                return mean.reindex(genes).to_numpy(float), noise.reindex(genes).to_numpy(float)

            def fold_gradient(per_donor, folds, fold):
                mean, noise = paired_gradient(per_donor.loc[folds.index[folds.eq(fold)]])
                return mean.to_numpy(float), noise.to_numpy(float)

            directions = []
            for f in (0, 1):
                score = strength_score([percentile_rank(fold_gradient(lake_pd, lake_fold, f)[0]),
                                        percentile_rank(fold_gradient(sn_pd, sn_fold, f)[0])], 'max')
                data = {'human': arrays(d[('human', 'ours')][0]), 'mouse': arrays(d[('mouse', 'ours')][0]),
                        'human_ref': fold_gradient(lake_pd, lake_fold, 1 - f), 'mouse_ref': fold_gradient(sn_pd, sn_fold, 1 - f)}
                directions.append((data, strength_bins(score, EDGES)))
            n = len(genes)
            point = binned_statistics(directions, np.arange(n), SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
            draws = np.stack([binned_statistics(directions, idx, SUBSETS, q=Q_PUBLISHED, min_reliability=MIN_RELIABILITY)
                              for idx in gene_draws(n, N_BOOT, SEED49)])
            low, high, _ = percentile_interval(draws)
            for subset in TOP_SUBSETS:
                i, j = SUBSET_NAMES.index(subset), STATS.index('index')
                rows.append({'contrast': contrast, 'subset': subset, 'index': point[i, j], 'ci_low': low[i, j], 'ci_high': high[i, j],
                             'genes': point[i, STATS.index('genes')]})
        return pd.DataFrame(rows)

    def cached_strength(result, label, stage):
        payload = cached_payload(f'strength_{stage}', lambda: {'table': strength_top(result).to_numpy(dtype=object).astype(str)},
                                 root=cache_, params={'edges': EDGES, 'q': Q_PUBLISHED, 'n_boot': N_BOOT, 'seed': SEED49},
                                 inputs={'genes': {c: list(g) for c, g in result['index_genes'].items()},
                                         'sn': digest(result['sn_counts'].sort_index(axis=1)), 'lake': digest(result['lake_counts']),
                                         'ours': digest({c: [result['datasets'][c][k][0][0] for k in (('human', 'ours'), ('mouse', 'ours'))]
                                                         for c in result['datasets']})},
                                 code=[strength_top, binned_statistics, NOTEBOOK_LOGIC_VERSION])
        table = pd.DataFrame(payload['table'], columns=['contrast', 'subset', 'index', 'ci_low', 'ci_high', 'genes'])
        return table.astype({'index': float, 'ci_low': float, 'ci_high': float, 'genes': float}).assign(setting=label)

    strength = {'baseline (paper map)': cached_strength(baseline, 'baseline (paper map)', 'baseline')}
    published49 = R49['curves']
    published49 = published49[published49.scheme.eq('max rank (primary)') & published49.q.eq(Q_PUBLISHED)
                              & published49.bootstrap.eq('gene') & published49.statistic.eq('index') & published49.subset.isin(TOP_SUBSETS)]
    check49 = strength['baseline (paper map)'].merge(published49, on=['contrast', 'subset'])
    if len(check49) != 6 or not (np.allclose(check49['index'], check49['value'], atol=1e-9, equal_nan=True)
                                 and np.allclose(check49['ci_low_x'], check49['ci_low_y'], atol=1e-9, equal_nan=True)
                                 and np.allclose(check49['ci_high_x'], check49['ci_high_y'], atol=1e-9, equal_nan=True)):
        raise ValueError("Part 49's top-bin indices are not reproduced.")
    print("Guard G6: part 49's top-bin and TOP indices reproduced exactly.")

    # The two alternative pairings.
    mouse62, human62 = I['inventory']['mouse'], I['inventory']['human']
    swap_map, swap_log = pairing.swap_pairs(paper_map, mouse62, human62)
    drop_map, dropped_pairs = pairing.drop_pairs(paper_map, mouse62, human62)
    print(f'Swap: {len(swap_log)} pairs changed in {swap_log.component.nunique()} families '
          f'({int(swap_log["mouse changed"].sum())} mouse members, {int(swap_log["human changed"].sum())} human members); '
          f'{len(swap_map):,} pairs.')
    print(f'Drop: {len(dropped_pairs):,} pairs removed; {len(drop_map):,} pairs remain.')
    swap_log.to_csv(tables / 'swap_log.csv', index=False)
    dropped_pairs.to_csv(tables / 'dropped_pairs.csv', index=False)
    swapped_in = set(swap_log['new pair'].str.split('–').str[1]) - set(paper_map.mouse_symbol)
    missing_census = sorted(swapped_in - set(census_tables['mouse'].columns))
    print('Swapped-in mouse genes absent from Census (excluded from the Census reference, counted):', missing_census)
    results = {'baseline (paper map)': baseline,
               '(a) swap to main PT paralog': primary_analysis(swap_map, '(a) swap to main PT paralog'),
               '(b) drop family pairs': primary_analysis(drop_map, '(b) drop family pairs')}
    strength['(a) swap to main PT paralog'] = cached_strength(results['(a) swap to main PT paralog'], '(a) swap to main PT paralog', 'swap')
    strength['(b) drop family pairs'] = cached_strength(results['(b) drop family pairs'], '(b) drop family pairs', 'drop')

    index_table = pd.concat([r['conservation'] for r in results.values()], ignore_index=True)
    index_wide = index_table.pivot_table(index=['contrast', 'version'], columns='setting', values='index', sort=False)
    for setting in list(results)[1:]:
        index_wide[f'Δ {setting}'] = index_wide[setting] - index_wide['baseline (paper map)']
    display(index_wide.round(4))
    index_table.to_csv(tables / 'conservation_index_by_setting.csv', index=False)
    strength_table = pd.concat(strength.values(), ignore_index=True)
    strength_wide = strength_table.pivot_table(index=['contrast', 'subset'], columns='setting', values='index', sort=False)
    for setting in list(results)[1:]:
        strength_wide[f'Δ {setting}'] = strength_wide[setting] - strength_wide['baseline (paper map)']
    strength_table.to_csv(tables / 'strength_top_bins_by_setting.csv', index=False)
    counts_table = pd.concat([r['class_counts'] for r in results.values()], ignore_index=True)
    display(counts_table.set_index(['setting', 'contrast'])[ZONATED_CLASSES + ['indeterminate', 'neither', 'mouse-only ÷ human-only']].round(2))
    counts_table.to_csv(tables / 'class_counts_by_setting.csv', index=False)
    base_counts = counts_table[counts_table.setting.eq('baseline (paper map)')].set_index('contrast')[ZONATED_CLASSES]
    decision_rows = []
    for setting in list(results)[1:]:
        other = counts_table[counts_table.setting.eq(setting)].set_index('contrast')[ZONATED_CLASSES]
        diff = (other - base_counts)
        relative = diff / base_counts.replace(0, np.nan)
        count_ok = np.where(base_counts >= SMALL_COUNT, relative.abs() < COUNT_TOLERANCE, diff.abs() <= SMALL_TOLERANCE)
        index_delta = index_wide[f'Δ {setting}'][index_wide.index.get_level_values('version').str.startswith(('primary', 'external: '))]
        strength_delta = strength_wide[f'Δ {setting}']
        failures = ([f'{c} · {v}: Δ {d:+.3f}' for (c, v), d in index_delta.items() if abs(d) >= INDEX_TOLERANCE]
                    + [f'strength {c} · {s}: Δ {d:+.3f}' for (c, s), d in strength_delta.items() if abs(d) >= INDEX_TOLERANCE]
                    + [f'{contrast} · {klass}: {int(base_counts.loc[contrast, klass])} → {int(other.loc[contrast, klass])}'
                       for i, contrast in enumerate(base_counts.index) for j, klass in enumerate(ZONATED_CLASSES) if not count_ok[i, j]])
        decision_rows.append({'setting': setting, 'max |Δ index| (primary, external)': float(index_delta.abs().max()),
                              'max |Δ index| (strength top bins)': float(strength_delta.abs().max()),
                              'max relative count change': float(relative.abs().max().max()),
                              'unaffected': not failures, 'failures': '; '.join(failures)})
    decision = pd.DataFrame(decision_rows)
    decision.to_csv(tables / 'decision.csv', index=False)
    display(decision)

    base_sym = results['baseline (paper map)']['symmetric']
    COLUMNS = ['S2-S1', 'S3c-early', 'S3-early', 'union S2-S1 + S3c-early']
    swap_sym = results['(a) swap to main PT paralog']['symmetric']
    pair_rows = []
    for r in swap_log.itertuples():
        old_h, old_m = r._2.split('–', 1)
        new_h, new_m = r._3.split('–', 1)
        row = {'component': r.component, 'kept pair': r._2, 'new pair': r._3}
        for c in COLUMNS:
            row[f'{c} before'] = base_sym[c].get(old_m, 'not classified')
            row[f'{c} after'] = swap_sym[c].get(new_m, 'not classified')
        row['probe-balanced before / after'] = f"{base_sym.probe_balanced.get(old_m, False)} / {swap_sym.probe_balanced.get(new_m, False)}"
        pair_rows.append(row)
    swap_classes = pd.DataFrame(pair_rows)
    swap_classes.to_csv(tables / 'swap_pair_classes.csv', index=False)
    retained_rows = []
    for setting in list(results)[1:]:
        other = results[setting]['symmetric']
        common = base_sym.index.intersection(other.index)
        for c in COLUMNS:
            moved = common[(base_sym.loc[common, c].astype(str) != other.loc[common, c].astype(str)).to_numpy()]
            for g in moved:
                retained_rows.append({'setting': setting, 'contrast': c, 'gene': g, 'before': base_sym.loc[g, c], 'after': other.loc[g, c],
                                      'probe-balanced': bool(base_sym.loc[g, 'probe_balanced'])})
    retained = pd.DataFrame(retained_rows, columns=['setting', 'contrast', 'gene', 'before', 'after', 'probe-balanced'])
    retained.to_csv(tables / 'class_changes_of_unchanged_pairs.csv', index=False)
    QUOTED_GENES = {'mouse-only examples (R3)': ['Fah', 'Comt', 'Maoa', 'Igfbp4', 'Ebp', 'Hsd17b4'],
                    'human-only examples (R3)': ['Sel1l3', 'Ace2', 'Ahcyl1', 'Hipk2', 'Wwtr1', 'Prkaa2', 'Taf4'],
                    'conserved transporter examples (R3)': ['Slc5a2', 'Slc5a1', 'Slc5a8', 'Slc7a13', 'Slc6a18', 'Slc22a7', 'Slc22a12',
                                                            'Slc34a3', 'Agt', 'Ace', 'Enpep']}
    statement_rows = []
    for setting, result in results.items():
        sym = result['symmetric']
        balanced_sym = sym[sym.probe_balanced]
        union = balanced_sym[UNION].value_counts()
        zonated_both = int(union.get('conserved', 0) + union.get('reversal', 0))
        transporter = balanced_sym.index.str.match(TRANSPORTER_REGEX)
        s21 = balanced_sym['S2-S1'].value_counts()
        statement_rows.append({
            'setting': setting,
            'union conserved / reversal / mouse-only / human-only': '/'.join(str(int(union.get(c, 0))) for c in ZONATED_CLASSES),
            'S2−S1 tested; conserved / reversal / mouse-only / human-only': f"{int(balanced_sym['S2-S1'].notna().sum())}; "
                + '/'.join(str(int(s21.get(c, 0))) for c in ZONATED_CLASSES),
            'tested in S2−S1 or S3c−early': int(balanced_sym[['S2-S1', 'S3c-early']].notna().any(axis=1).sum()),
            'reversal share of genes zonated in both': f"{int(union.get('reversal', 0))}/{zonated_both} = "
                                                      f"{union.get('reversal', 0) / max(zonated_both, 1):.0%}",
            'transporters among conserved': f"{int((balanced_sym[UNION].eq('conserved') & transporter).sum())}/{int(union.get('conserved', 0))}",
            'mouse-only : human-only': f"{int(union.get('mouse-only', 0))} : {int(union.get('human-only', 0))}"})
    statements = pd.DataFrame(statement_rows).set_index('setting').T
    display(statements)
    gene_rows = []
    for group, genes_ in QUOTED_GENES.items():
        for g in genes_:
            gene_rows.append({'group': group, 'gene': g, **{setting: (r['symmetric'][UNION].get(g, 'not classified')) for setting, r in results.items()}})
    gene_rows.append({'group': 'human-only example, S2 − S1 (R3)', 'gene': 'Slc9a3',
                      **{setting: r['symmetric']['S2-S1'].get('Slc9a3', 'not classified') for setting, r in results.items()}})
    quoted = pd.DataFrame(gene_rows)
    quoted['changes'] = quoted[list(results)].nunique(axis=1).gt(1)
    display(quoted)
    statements.to_csv(tables / 'draft_statements_by_setting.csv')
    quoted.to_csv(tables / 'draft_example_genes_by_setting.csv', index=False)
    return {k: v for k, v in locals().items() if not k.startswith('__') and k != 'results'} | {
        'decision': decision, 'index_table': index_table, 'counts_table': counts_table, 'strength_table': strength_table}

## 10 · Notebook 44 part B: our donor on the Lake injury axis

- **Signature.** Genes up or down in Lake aPT nuclei against healthy PT nuclei of the same donor: the top 30 up
  and top 30 down. Donors need ≥ 30 aPT and ≥ 100 healthy nuclei, and genes a mean CPM ≥ 20.
- **Score.** The mean within-sample percentile of the up genes minus that of the down genes.
- **Platform offset.** Estimated in mouse, from our control mice against the healthy Census mice.
- **Inputs.** Whole-PT pseudobulks per specimen from both package exports; the AKI export gives the AKI − control
  score. The universe drops the mt-encoded genes in the primary arm.

In [ ]:
from scipy import stats


def part44B(ours, mouse_only, out, *, exclude_genes=()):
    """Notebook 44 part B. ``ours`` and ``mouse_only`` are (counts, genes, specimen) of the PT units (human vs mouse
    and AKI datasets)."""
    tables, _ = folders(out, '44B')
    cache_ = out / 'stage_cache' / '44B'
    human_ids = ours[3]

    def obs_column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        return item[:].astype(str)

    def lake_states():
        with h5py.File(LAKE_FILE, 'r') as handle:
            obs = pd.DataFrame({c: obs_column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'donor_id']})
            keep = (obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
                    & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']))
            label = (obs.donor_id + '|' + obs.disease + '|' + obs['subclass.l2'])[keep]
            groups, codes = np.unique(label.to_numpy(), return_inverse=True)
            raw = handle['raw/X']
            var_ids = handle['raw/var/_index'][:].astype(str)
            var_names = np.asarray(obs_column(handle['raw/var'], 'feature_name'))
            sums, totals, counts = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                                  codes, len(groups), len(var_ids), block_rows=5000)
        by_id = dict(zip(orthologs.human_symbol.map(human_ids), orthologs.mouse_symbol))
        by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
        target = np.array([by_id.get(i, by_symbol.get(n, '')) for i, n in zip(var_ids, var_names)])
        mapped = target != ''
        frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
        return {'groups': groups.astype(str), 'counts': counts, 'totals': totals, 'sums': frame.to_numpy(),
                'genes': np.asarray(frame.columns, dtype=str)}

    lake = cached_payload('lake_cortex_pt_states', lake_states, root=cache_,
                          params={'labels': 'PT-S1,PT-S2,PT-S3,aPT,dPT,cycPT', 'region': 'C cortex'},
                          inputs={'lake': [LAKE_FILE.name, LAKE_FILE.stat().st_size], 'orthologs': digest(orthologs),
                                  'human_ids': digest(human_ids)},
                          code=digest(['44.revision_state.1', NOTEBOOK_LOGIC_VERSION]))
    lake_meta_ = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'state'])
    lake_meta_['nuclei'], lake_meta_['total'] = lake['counts'], lake['totals']
    lake_sums = pd.DataFrame(lake['sums'], columns=lake['genes'])
    ours_counts, ours_genes, ours_specimen = ours[:3]
    ours_pb = pd.DataFrame({n: np.asarray(ours_counts[ours_specimen == n].sum(axis=0)).ravel() for n in names}, index=ours_genes)
    mo_counts, mo_genes, mo_sample = mouse_only
    mo_pb = pd.DataFrame({n: np.asarray(mo_counts[mo_sample == n].sum(axis=0)).ravel() for n in ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4']},
                         index=mo_genes)
    census_counts = pd.read_csv(CENSUS_DIR / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = census_meta['mouse'].copy()
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)]
    meta['segment'] = meta.cell_type.map({'epithelial cell of proximal tubule segment 1': 'S1',
                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                          'epithelial cell of proximal tubule segment 3': 'S3'})
    meta = meta[meta.segment.notna() & meta.n_cells.ge(50)]
    census_pb = pd.DataFrame({donor: census_counts.loc[block.index].sum() for donor, block in meta.groupby('donor_id')})
    universe = (pd.Index(ours_pb.index).intersection(mo_pb.index).intersection(lake_sums.columns).intersection(census_pb.index))
    universe = universe[~universe.isin(set(exclude_genes))]
    print(f'Gene universe U: {len(universe):,} genes')

    def lake_state_pb(donor, states):
        rows = lake_meta_.index[lake_meta_.donor.eq(donor) & lake_meta_.state.isin(states)]
        return lake_sums.loc[rows].sum(), lake_meta_.loc[rows, 'total'].sum(), lake_meta_.loc[rows, 'nuclei'].sum()

    healthy_states = ['PT-S1', 'PT-S2', 'PT-S3']
    signature_rows = {}
    for donor in lake_meta_.donor.unique():
        a_sum, a_tot, a_n = lake_state_pb(donor, ['aPT'])
        h_sum, h_tot, h_n = lake_state_pb(donor, healthy_states)
        if a_n >= 30 and h_n >= 100:
            signature_rows[donor] = {'a_cpm': a_sum[universe] / a_tot * 1e6, 'h_cpm': h_sum[universe] / h_tot * 1e6}
    a_cpm = pd.DataFrame({d: v['a_cpm'] for d, v in signature_rows.items()})
    h_cpm = pd.DataFrame({d: v['h_cpm'] for d, v in signature_rows.items()})
    lfc = (np.log2(a_cpm + 1) - np.log2(h_cpm + 1)).median(axis=1)
    UP = lfc[a_cpm.mean(axis=1).ge(20)].nlargest(30).index.tolist()
    DOWN = lfc[h_cpm.mean(axis=1).ge(20)].nsmallest(30).index.tolist()
    print(f'Signature from {len(signature_rows)} Lake donors. Up: {UP}')
    print(f'Down: {DOWN}')

    def injury_score(pseudobulk):
        ranks = pseudobulk.reindex(universe).fillna(0).rank(pct=True)
        return float(ranks[UP].mean() - ranks[DOWN].mean())

    lake_rows = []
    for donor in lake_meta_.donor.unique():
        total_sum, total, n = lake_state_pb(donor, healthy_states + ['aPT', 'dPT', 'cycPT'])
        altered = lake_meta_.loc[lake_meta_.donor.eq(donor) & lake_meta_.state.isin(['aPT', 'dPT', 'cycPT']), 'nuclei'].sum()
        if n >= 100:
            lake_rows.append({'donor': donor, 'disease': lake_meta_.loc[lake_meta_.donor.eq(donor), 'disease'].iloc[0],
                              'pt_nuclei': int(n), 'altered_fraction': altered / n, 'injury_score': injury_score(total_sum)})
    lake_scores = pd.DataFrame(lake_rows)
    rho = stats.spearmanr(lake_scores.injury_score, lake_scores.altered_fraction).statistic
    census_scores = pd.Series({d: injury_score(census_pb[d]) for d in census_pb.columns})
    ours_scores = pd.Series({n: injury_score(ours_pb[n]) for n in ours_pb.columns})
    mo_scores = pd.Series({n: injury_score(mo_pb[n]) for n in mo_pb.columns})
    offset = float(ours_scores[expected['mouse']].mean() - census_scores.median())
    normal = lake_scores[lake_scores.disease.eq('normal')]
    fit = np.polyfit(lake_scores.injury_score, lake_scores.altered_fraction, 1)
    placement = pd.DataFrame({'raw score': ours_scores[expected['human']], 'calibrated score': ours_scores[expected['human']] - offset})
    placement['percentile among Lake normal donors'] = [float((normal.injury_score < s).mean()) for s in placement['calibrated score']]
    placement['predicted altered fraction (Lake fit)'] = np.polyval(fit, placement['calibrated score'])
    placement['within Lake normal range'] = placement['calibrated score'].between(normal.injury_score.min(), normal.injury_score.max())
    placement['above Lake AKI median'] = placement['calibrated score'] > lake_scores.loc[lake_scores.disease.eq('acute kidney injury'), 'injury_score'].median()
    b_summary = pd.Series({
        'Lake donors scored': len(lake_scores), 'Spearman score vs altered fraction (Lake)': rho,
        'Lake normal median': normal.injury_score.median(), 'Lake normal range': f'{normal.injury_score.min():.3f} to {normal.injury_score.max():.3f}',
        'Lake CKD median': lake_scores.loc[lake_scores.disease.eq('chronic kidney disease'), 'injury_score'].median(),
        'Lake AKI median': lake_scores.loc[lake_scores.disease.eq('acute kidney injury'), 'injury_score'].median(),
        'Census healthy mouse median': census_scores.median(), 'our control mice (mean)': ours_scores[expected['mouse']].mean(),
        'platform offset (mouse)': offset, 'mouse-only: AKI − control': mo_scores[['IR2A2', 'IR2A4']].mean() - mo_scores[['Ctrl1A2', 'Ctrl1A4']].mean()})
    lake_scores.to_csv(tables / 'B_lake_donor_injury_scores.csv', index=False)
    placement.to_csv(tables / 'B_our_donor_placement.csv')
    pd.DataFrame({'gene': UP + DOWN, 'direction': ['up in aPT'] * len(UP) + ['down in aPT'] * len(DOWN),
                  'median_log2fc_aPT_vs_healthy': lfc[UP + DOWN].to_numpy()}).to_csv(tables / 'B_injury_signature.csv', index=False)
    pd.concat({'census mouse (healthy)': census_scores, 'mouse-only Visium HD': mo_scores, 'ours (rebuild)': ours_scores}).to_csv(
        tables / 'B_calibration_scores.csv')
    b_summary.to_csv(tables / 'B_summary.csv')
    display(b_summary.to_frame('value'))
    display(placement.round(3))
    return {'placement': placement, 'summary': b_summary, 'lake_scores': lake_scores}

## 11 · Guards and the two input bundles (package export)

**Guards (not cached).**
- G1: the export contract (`read_pt_units`).
- G2: the confirmed pass-1 label fingerprint (n_cells, n_clusters, membership sha1 and the label map).
- G3: PT-S1, PT-S2 and PT-S3 in every specimen; at least 3 Glomerulus units per mouse specimen.
- G4: the 2 px polygon join, checked in part 42.
- G5: the native stage-01 route reproduces the export counts.

The gene order is the ortholog map's, as `rebuild_pt_expression` gave it to notebooks 42–64. Seeded matched draws
(notebook 48 B) depend on it.
- G8: the primary library is the full library minus exactly the mt genes' counts.

**Bundles.** Both hold every PT unit, with no coordinate trim.
- **Primary:** the mt-encoded genes leave the gene space and every library.
- **Secondary:** 43's rule, with mt reads in the library and the gene space.

The expression glomeruli and the cortical-depth anchors are the QC units of family Glomerulus (label Podocyte).

In [ ]:
from pseudospace.package_labels import (all_units, check_fingerprint, cortical_depth, harmony_features, label_panel_genes,
                                        label_units, native_panel, native_sources)

RUN = results_root / 'package_pseudospace' / 'runs' / 'hvm'
GENE_ORDER = orthologs.mouse_symbol.astype(str).tolist()   # rebuild_pt_expression's order, as in notebooks 42-64
units = {'primary': label_units(results_root, 'hvm', exclude_genes=MT_GENES, gene_order=GENE_ORDER),
         'secondary': label_units(results_root, 'hvm', gene_order=GENE_ORDER)}
provenance = units['primary']['provenance']
CONFIRMED = check_fingerprint(results_root, 'hvm', provenance, CONFIRMED_PASS1_FINGERPRINT)
print('G1, G2: export contract and confirmed labels:', {k: v for k, v in CONFIRMED.items() if k != 'labels_pass1'})
pt_obs_ = units['primary']['obs']
segment_table = pd.crosstab(pt_obs_['sample'], pt_obs_.segment)
display(segment_table)
if (segment_table.reindex(index=names, columns=['S1', 'S2', 'S3']).fillna(0) == 0).any().any():
    raise ValueError('G3: a PT segment is missing in a specimen.')
print('mt genes leaving the gene space and the libraries:', units['primary']['excluded'])
mt_counts = np.asarray(units['secondary']['counts'][:, units['secondary']['gene_space'].isin(units['primary']['excluded'])].sum(axis=1)).ravel()
if not np.allclose(units['secondary']['library'] - mt_counts, units['primary']['library']):
    raise ValueError('G8: the primary library is not the full library minus the mt counts.')
print(f'G8: the mt genes carry {np.median(mt_counts / units["secondary"]["library"]):.2%} of the median unit library.')

cohort_obs, (marker_counts, marker_genes), cohort_library_full = all_units(
    results_root, 'hvm', genes=[g for genes in MARKERS.values() for g in genes])
cohort_library_mtfree = all_units(results_root, 'hvm', exclude_genes=MT_GENES)[2]
cohort_species = cohort_obs['sample'].map({n: sp for sp, v in expected.items() for n in v})
cohort = pd.DataFrame({'sample': cohort_obs['sample'], 'feature_index': cohort_obs.feature_index.astype(int),
                       'x_centroid': cohort_obs.x_centroid.astype(float), 'y_centroid': cohort_obs.y_centroid.astype(float),
                       'label': np.where(cohort_obs.family.eq('Glomerulus'), 'Glomerulus', cohort_obs.label),
                       'comparison_species': cohort_species}, index=cohort_obs.index)
if not (cohort.loc[pt_obs_.index, 'label'].to_numpy() == ('PT-' + pt_obs_.segment).to_numpy()).all():
    raise ValueError('The PT labels of all_units_positions.csv and pt_units.h5ad disagree.')
anchors = cohort[cohort.label.eq('Glomerulus')]
if (anchors[anchors['sample'].isin(expected['mouse'])].groupby('sample').size().reindex(expected['mouse']).fillna(0) < 3).any():
    raise ValueError('G3: fewer than 3 Glomerulus units in a mouse specimen.')
depth = cortical_depth(pt_obs_, anchors, expected['mouse'], k=3)
print('Glomerulus-family units per specimen:', anchors.groupby('sample').size().to_dict())

segmentation = {s: provenance['segmentations'][s]['file'] for s in names}
registration = {}
for name in names:
    stage01 = ad.read_h5ad(RUN / '01_units' / f'{name}_units.h5ad', backed='r')
    registration[name] = stage01.obs[['feature_index', 'n_spots']].copy()
    stage01.file.close()
human_gene_ids = native_panel(RUN / '01_units' / 'HUK1_COR1_units.h5ad')
mouse_gene_ids = native_panel(RUN / '01_units' / 'Ctrl1A2_units.h5ad')
PANEL_GENES = label_panel_genes(provenance)
INTEGRATION = {'pass-1 Harmony features': harmony_features(results_root, 'hvm', 'pass1'),
               'pass-2 Harmony features': harmony_features(results_root, 'hvm', 'pass2'), 'PT panel genes': PANEL_GENES}
INTEGRATION['pass-1 Harmony features + PT panel genes'] = INTEGRATION['pass-1 Harmony features'] | PANEL_GENES
INTEGRATION_ARMS = {'primary': 'pass-1 Harmony features', 'secondary': 'pass-1 Harmony features + PT panel genes'}
print({k: len(v) for k, v in INTEGRATION.items()}, '· PT panel genes:', sorted(PANEL_GENES))

# Notebook 64's native route (G5) and notebook 62's PT-expression columns recomputed on the package units.
native_obs, native_paths = native_sources(results_root, 'hvm', pt_obs_)
rebuilt = rebuild_pt_expression(native_obs, native_paths, orthologs, target_sum=1e4)
rebuilt_measured = rebuilt.var.measured_in_both_inputs.to_numpy()
if not (pd.Index(rebuilt.var_names[rebuilt_measured]).equals(units['secondary']['gene_space'])
        and (sparse.csr_matrix(rebuilt.layers['counts'])[:, rebuilt_measured] != units['secondary']['counts']).nnz == 0):
    raise ValueError('G5: the native stage-01 route does not reproduce the export.')
print('G5: the native stage-01 route reproduces the export counts exactly.')
del rebuilt
expression62 = pt_expression_inventory(native_obs, native_paths)
inventory, inventory_changes = {}, []
for species in ('mouse', 'human'):
    old = pd.read_csv(UPSTREAM62 / f'gene_inventory_{species}.csv', index_col=0)
    new = old.copy()
    fresh = expression62[species].reindex(old.index)
    if fresh.pt_expressed.isna().any():
        raise ValueError(f'{species}: notebook 62 genes missing from the native panel.')
    new['pt_expressed'] = fresh.pt_expressed.astype(bool).to_numpy()
    new['log2_cp10k'] = fresh.log2_cp10k.to_numpy()
    inventory[species] = new
    inventory_changes.append({'species': species, 'genes': len(old),
                              'PT-expressed (notebook 62)': int(old.pt_expressed.astype(bool).sum()),
                              'PT-expressed (package units)': int(new.pt_expressed.sum()),
                              'Spearman log2 CP10k': float(old.log2_cp10k.corr(new.log2_cp10k, method='spearman'))})
display(pd.DataFrame(inventory_changes))

BUNDLES = {}
for arm, data in units.items():
    exclude = MT_GENES if arm == 'primary' else []
    BUNDLES[arm] = {
        'obs': data['obs'], 'gene_space': data['gene_space'], 'counts': data['counts'], 'lognorm': data['lognorm'],
        'library': data['library'], 'depth': depth, 'human_gene_ids': human_gene_ids, 'mouse_gene_ids': mouse_gene_ids,
        'cohort': cohort, 'cohort_counts': marker_counts, 'cohort_genes': marker_genes,
        'cohort_measured': np.ones(len(marker_genes), bool),
        'cohort_library': cohort_library_mtfree if arm == 'primary' else cohort_library_full,
        'segmentation': segmentation, 'glomerulus_geometry': segmentation, 'registration': registration,
        'integration': INTEGRATION, 'integration_arms': INTEGRATION_ARMS,
        'native': {'pt_obs': native_obs, 'sources': native_paths}, 'inventory': inventory,
        'exclude_genes': exclude, 'labels_name': 'package'}

## 12 · Cohort, bins and UMI per unit, and the size of human S2

Notebook 70's stage counts are read, not recomputed. The PT units per specimen and segment are shown beside the
old notebook-13 structures. Bins and UMI are given per unit, by specimen and species: PT units and every QC unit.
The protocol's inputs give human PT-S2 only 373 + 259 units; no rule changes, and any statistic that becomes
undefined is reported where it occurs.

In [ ]:
COMPARE = output / 'tables' / 'compare'
COMPARE.mkdir(parents=True, exist_ok=True)
stage_counts = pd.read_csv(results_root / 'package_pseudospace' / 'hvm' / 'tables' / 'stage_counts.csv', index_col=0)
old_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs[['sample', 'segment_class']].copy()
old_units = old_pt.groupby(['sample', 'segment_class'], observed=True).size().unstack()
new_units = pt_obs_.assign(segment_class='PT-' + pt_obs_.segment).groupby(['sample', 'segment_class']).size().unstack()
unit_counts = pd.concat({'old (notebook 13)': old_units, 'package': new_units}, axis=1).reindex(names)
unit_counts.loc['total'] = unit_counts.sum()
display(stage_counts)
display(unit_counts)
pt_depth = pd.DataFrame({'sample': pt_obs_['sample'], 'species': pt_obs_.species, 'bins': pt_obs_.n_spots.astype(float),
                         'UMI': pt_obs_.total_counts.astype(float)})
all_depth = pd.DataFrame({'sample': cohort_obs['sample'], 'species': cohort_species, 'bins': cohort_obs.n_spots.astype(float),
                          'UMI': cohort_obs.total_counts.astype(float)})
quantile_rows = []
for scope, frame in (('PT units', pt_depth), ('all QC units', all_depth)):
    for key, column in (('specimen', 'sample'), ('species', 'species')):
        for group, block in frame.groupby(column):
            for measure in ('bins', 'UMI'):
                q = block[measure].quantile([.1, .25, .5, .75, .9])
                quantile_rows.append({'scope': scope, 'by': key, 'group': group, 'measure': measure, 'units': len(block),
                                      **{f'q{int(p * 100)}': v for p, v in q.items()}})
cohort_table = pd.DataFrame(quantile_rows)
stage_counts.to_csv(COMPARE / 'cohort_stage_counts.csv')
unit_counts.to_csv(COMPARE / 'cohort_pt_units.csv')
cohort_table.to_csv(COMPARE / 'cohort_bins_umi.csv', index=False)
display(cohort_table[cohort_table.by.eq('species')].round(1))

## 13 · Primary arm: notebooks 42, 43, 49, 46, 47, 48 B, 64 and 44 B (mt-free library)

In [ ]:
R42 = part42(BUNDLES['primary'], output)

In [ ]:
R43 = part43(BUNDLES['primary'], output, panel_genes=PANEL_GENES)

In [ ]:
R49 = part49(BUNDLES['primary'], R43, output, full=True)

In [ ]:
R46 = part46(BUNDLES['primary'], R43, output)

In [ ]:
R47 = part47(BUNDLES['primary'], R43, R46, R42, output)

In [ ]:
R48 = part48B(BUNDLES['primary'], R43, output)

In [ ]:
R64 = part64(BUNDLES['primary'], R43, R49, output)

In [ ]:
aki_units = label_units(results_root, 'aki', exclude_genes=MT_GENES, gene_order=GENE_ORDER)
check_fingerprint(results_root, 'aki', aki_units['provenance'], CONFIRMED_AKI_PASS1)
R44 = part44B((units['primary']['counts'], units['primary']['gene_space'], pt_obs_['sample'].astype(str).to_numpy(), human_gene_ids),
              (aki_units['counts'], aki_units['gene_space'], aki_units['obs']['sample'].astype(str).to_numpy()),
              output, exclude_genes=MT_GENES)

## 14 · Secondary arm: mt reads in the library (the pure input swap)

This arm runs part 42's transfer and spillover, part 43 in full, and part 49's part A with the gene bootstrap. Its
tables are under `tables/mt_in_library/`. Old → secondary isolates the change of units and labels; secondary →
primary isolates the change of library.

In [ ]:
S42 = part42(BUNDLES['secondary'], output, geometry=False, tag='mt_in_library/42')
S43 = part43(BUNDLES['secondary'], output, panel_genes=PANEL_GENES, tag='mt_in_library/43')
S49 = part49(BUNDLES['secondary'], S43, output, full=False, tag='mt_in_library/49')

## 15 · Comparison with the old results, and the material-change rules M1–M9

The old values are read from the published tables of notebooks 42, 43, 46, 47, 48 B, 49, 64 and 44 B.
- **Columns.** Each headline value is given old → secondary (mt in the library) → primary (mt-free).
- **Which arm the rules use.** The rules M1–M9 apply to the primary arm. The secondary arm says whether a change
  comes from the units and labels or from the library.
- **Descriptive comparisons.**
  - Gene gradients: old against new, per species and contrast, with the old gradients rebuilt from the old inputs
    by notebook 43's pseudobulk rule.
  - The old symmetric class against the new one.
  - On the units matched to an old structure: the reference transfer's agreement with the old reviewed labels and
    with the package labels.

In [ ]:
OLD = {'42': results_root / 'pt_revision_labels', '43': results_root / 'pt_revision_classes' / 'reviewed' / 'tables',
       '46': results_root / 'pt_revision_classes' / 'crossfit' / 'tables', '47': results_root / 'pt_revision_addenda' / 'tables',
       '48B': results_root / 'pt_revision_addenda_pathways' / 'tables', '49': results_root / 'pt_conservation_strength' / 'tables',
       '64': results_root / 'pt_paralog_pairing_check' / 'tables', '44B': results_root / 'pt_revision_state' / 'tables'}
old = {'transfer': pd.read_csv(OLD['42'] / 'transfer_agreement.csv'), 'distance': pd.read_csv(OLD['42'] / 'glomerulus_distance_by_label.csv'),
       'attached': pd.read_csv(OLD['42'] / 'glomerulus_attached_pt.csv'), 'spill': pd.read_csv(OLD['42'] / 'spillover_index.csv'),
       'fractions': pd.read_csv(OLD['42'] / 'segment_fractions.csv'),
       'conservation': pd.read_csv(OLD['43'] / 'conservation_index.csv'), 'decisions43': pd.read_csv(OLD['43'] / 'decisions.csv', index_col=0).iloc[:, 0],
       'class_counts': pd.read_csv(OLD['43'] / 'class_counts_by_setting_and_contrast.csv'),
       'symmetric': pd.read_csv(OLD['43'] / 'gene_classes_symmetric.csv', index_col=0),
       'amplitude': json.loads((results_root / 'pt_revision_classes' / 'reviewed' / 'run_manifest.json').read_text())['amplitude'],
       'curves': pd.read_csv(OLD['49'] / 'strength_curves.csv'), 'rules49': pd.read_csv(OLD['49'] / 'strength_decision_rules.csv'),
       'decisions49': pd.read_csv(OLD['49'] / 'decisions.csv', index_col=0).iloc[:, 0],
       'direction': pd.read_csv(OLD['49'] / 'direction_agreement_by_strength.csv'),
       'crossfit': pd.read_csv(OLD['46'] / 'crossfit_conservation_index.csv'), 'decisions46': pd.read_csv(OLD['46'] / 'decisions.csv', index_col=0).iloc[:, 0],
       'nointegration': pd.read_csv(OLD['47'] / 'no_integration_genes_conservation_index.csv'),
       'transporters': pd.read_csv(OLD['47'] / 'transporter_direction_agreement.csv'),
       'transporter_shares': pd.read_csv(OLD['47'] / 'transporter_class_shares.csv'),
       'integration_classes': pd.read_csv(OLD['47'] / 'integration_genes_by_symmetric_class.csv'),
       'naming': pd.read_csv(OLD['48B'] / 'declared_naming_rule.csv'), 'decision64': pd.read_csv(OLD['64'] / 'decision.csv'),
       'placement': pd.read_csv(OLD['44B'] / 'B_our_donor_placement.csv', index_col=0)}
PRIMARY_VERSION = 'primary: ours human vs ours mouse'
D2C = ['S2-S1', 'S3-early', 'S3c-early']


def transfer_value(table, species, reference, field):
    block = table[table.species.eq(species) & table.reference.str.startswith(reference) & table.score.str.startswith('primary')]
    return float(block[field].iloc[0])


def support(table):
    h, m = transfer_value(table, 'human', 'Lake', 'agreement with current'), transfer_value(table, 'mouse', 'micro', 'agreement with current')
    return bool(h >= .70 and abs(h - m) <= .10)


def spearman_glom(distance, species):
    block = distance[distance.glomeruli.eq('morphological') & distance.species.eq(species)
                     & distance.segment.str.startswith('Spearman')]
    return float(block['median um'].iloc[0])


def odds(attached, species, threshold):
    block = attached[attached.glomeruli.eq('morphological') & attached['threshold um'].eq(threshold) & attached.group.eq(species)]
    return float(block['odds ratio S1'].iloc[0]), int(block['attached PT'].iloc[0])


def index_value(conservation, contrast, version=PRIMARY_VERSION, field='index'):
    return float(conservation[conservation.contrast.eq(contrast) & conservation.version.eq(version)][field].iloc[0])


def union_counts(class_counts):
    row = class_counts[class_counts.setting.eq('symmetric (A3b + A8)') & class_counts.contrast.eq(UNION)].iloc[0]
    return {k: int(row[k]) for k in ZONATED_CLASSES}


def curve_value(curves, contrast, subset, stat='index', boot='gene'):
    block = curves[curves.contrast.eq(contrast) & curves.scheme.eq('max rank (primary)') & curves.q.eq(Q_PUBLISHED)
                   & curves.bootstrap.eq(boot) & curves.subset.eq(subset) & curves.statistic.eq(stat)]
    return float(block.value.iloc[0])


def direction_share(direction, contrast, bin_label):
    return float(direction[direction.contrast.eq(contrast) & direction.bin.eq(bin_label)]['share conserved'].iloc[0])


def naming_set(naming, klass):
    value = naming[naming.run.eq('probe-balanced') & naming['class'].eq(klass)]['counting pathways'].fillna('').iloc[0]
    return sorted({p.split(' (')[0].strip() for p in str(value).split(';') if p.strip()})


def zonated_both_agreement(class_counts):
    c = union_counts(class_counts)
    return c['conserved'] / max(c['conserved'] + c['reversal'], 1)


rows = []


def add(part, metric, old_value, secondary, primary):
    rows.append({'part': part, 'metric': metric, 'old': old_value, 'secondary (mt in library)': secondary, 'primary (mt-free)': primary})


NA = np.nan
for sp, ref in (('human', 'Lake'), ('mouse', 'micro'), ('mouse', 'male')):
    for field in ('agreement with current', 'Cohen kappa'):
        add('42', f'transfer {sp} vs {ref}: {field}', transfer_value(old['transfer'], sp, ref, field),
            transfer_value(S42['TRANSFER'], sp, ref, field), transfer_value(R42['TRANSFER'], sp, ref, field))
add('42', 'pre-specified reading: human labels supported', support(old['transfer']), support(S42['TRANSFER']), support(R42['TRANSFER']))
for sp in ('mouse', 'human'):
    add('42', f'{sp}: Spearman(label order, distance to glomerulus)', spearman_glom(old['distance'], sp), NA, spearman_glom(R42['DISTANCE'], sp))
    for threshold in (ATTACH_UM, ATTACH_UM_SENSITIVITY):
        o, n = odds(old['attached'], sp, threshold), odds(R42['ATTACHED'], sp, threshold)
        add('42', f'{sp}: S1 odds ratio among PT within {threshold:g} µm (attached PT)', f'{o[0]:.2f} ({o[1]})', NA, f'{n[0]:.2f} ({n[1]})')
    for s in ('S1', 'S2', 'S3'):
        add('42', f'{sp}: median µm to glomerulus, {s}',
            float(old['distance'][old['distance'].glomeruli.eq('morphological') & old['distance'].species.eq(sp) & old['distance'].segment.eq(s)]['median um'].iloc[0]),
            NA, float(R42['DISTANCE'][R42['DISTANCE'].glomeruli.eq('morphological') & R42['DISTANCE'].species.eq(sp)
                                      & R42['DISTANCE'].segment.eq(s)]['median um'].iloc[0]))
for sp in ('human', 'mouse'):
    o = old['fractions'][old['fractions'].species.eq(sp) & old['fractions'].score.str.startswith('primary')]
    n = R42['FRACTIONS'][R42['FRACTIONS'].species.eq(sp) & R42['FRACTIONS'].score.str.startswith('primary')]
    add('42', f'{sp}: S1 share of PT (labels), per specimen', ', '.join(f'{v:.2f}' for v in o.drop_duplicates('specimen')['S1 fraction (current)']),
        NA, ', '.join(f'{v:.2f}' for v in n.drop_duplicates('specimen')['S1 fraction (current)']))
    o_pt, s_pt, n_pt = (t[t.comparison_species.eq(sp) & t.label.str.startswith('PT-')]['median'] for t in (old['spill'], S42['SPILL'], R42['SPILL']))
    add('42', f'{sp}: spillover median share in PT, range over segments', f'{o_pt.min():.4f}–{o_pt.max():.4f}',
        f'{s_pt.min():.4f}–{s_pt.max():.4f}', f'{n_pt.min():.4f}–{n_pt.max():.4f}')
for contrast in D2C:
    for version in (PRIMARY_VERSION, 'external: Lake vs mouse snRNA male'):
        tag = 'primary index' if version == PRIMARY_VERSION else 'external index'
        values = [index_value(t, contrast, version) for t in (old['conservation'], S43['conservation'], R43['conservation'])]
        cis = [f"{index_value(t, contrast, version, 'ci_low'):.2f}–{index_value(t, contrast, version, 'ci_high'):.2f}"
               for t in (old['conservation'], S43['conservation'], R43['conservation'])]
        add('43', f'{contrast}: {tag}', *[f'{v:.3f} ({c})' for v, c in zip(values, cis)])
    add('43', f'{contrast}: genes in the index', *[int(index_value(t, contrast, field='genes'))
                                                   for t in (old['conservation'], S43['conservation'], R43['conservation'])])
add('43', 'D2 wording', old['decisions43']['D2 wording'], S43['D2_WORDING'], R43['D2_WORDING'])
add('43', 'D3 asymmetry', old['decisions43']['D3 asymmetry'], S43['D3_ASYMMETRY'], R43['D3_ASYMMETRY'])
add('43', 'rat usable', old['decisions43']['rat usable'], S43['RAT_USABLE'], R43['RAT_USABLE'])
for contrast in ('S2-S1', 'S3c-early', 'S3-early'):
    add('43', f'{contrast}: noise-corrected human ÷ mouse amplitude', old['amplitude'][contrast], S43['amplitude'][contrast], R43['amplitude'][contrast])
for contrast in ('S2-S1', 'S3c-early', 'S3-early', UNION):
    o, s, n = (t[t.setting.eq('symmetric (A3b + A8)') & t.contrast.eq(contrast)].iloc[0] for t in (old['class_counts'], S43['class_counts'], R43['class_counts']))
    add('43', f'{contrast}: conserved / reversal / mouse-only / human-only (probe-balanced)',
        *['/'.join(str(int(r[k])) for k in ZONATED_CLASSES) for r in (o, s, n)])
for contrast in ('S2-S1', 'S3c-early', 'S3-early'):
    add('43', f'{contrast}: genes classified (testable in both species)',
        int(old['symmetric'][contrast].notna().sum()), int(S43['symmetric'][contrast].notna().sum()), int(R43['symmetric'][contrast].notna().sum()))
add('43', 'direction agreement among genes zonated in both (union, probe-balanced)', zonated_both_agreement(old['class_counts']),
    zonated_both_agreement(S43['class_counts']), zonated_both_agreement(R43['class_counts']))
for contrast in D2C:
    for subset in (BINS[-1], 'TOP (≥95%)', 'BOTTOM (<80%)', 'all genes'):
        add('49', f'{contrast}: index, {subset}', curve_value(old['curves'], contrast, subset), curve_value(S49['curves'], contrast, subset),
            curve_value(R49['curves'], contrast, subset))
    add('49', f'{contrast}: bins with an undefined human ceiling (of 6)',
        *[int(np.isnan([curve_value(c, contrast, b, 'rstar_ceiling_human') for b in BINS]).sum()) for c in (old['curves'], S49['curves'], R49['curves'])])
    for bin_label in ('50–80%', BINS[-1]):
        if bin_label in BINS:
            add('49', f'{contrast}: direction agreement, bin {bin_label}', direction_share(old['direction'], contrast, bin_label),
                direction_share(S49['direction_agreement'], contrast, bin_label), direction_share(R49['direction_agreement'], contrast, bin_label))
rule_i_old = bool(old['rules49'][old['rules49'].contrast.isin(['S2-S1', 'S3-early'])]['(i) concentration'].all())
add('49', 'rule (i) concentration, S2−S1 and S3−early (primary: gene and module; secondary: gene)', rule_i_old, S49['RULE_I'], R49['RULE_I'])
add('49', 'rule (ii) BOTTOM reproducible but not shared', old['decisions49']['rule (ii) BOTTOM reproducible but not shared'], S49['RULE_II'], R49['RULE_II'])
add('49', 'title verdict', old['decisions49']['title verdict'], S49['TITLE_VERDICT'], R49['TITLE_VERDICT'])
add('49', '"largely different genes" under every sensitivity (part B)', old['decisions49']['"largely different genes" stands under every sensitivity'],
    NA, R49['WORDING_STANDS'])
add('46', 'claim "largely species-specific" stands', old['decisions46']['claim "largely species-specific" stands'], NA, R46['CLAIM_STANDS'])
add('46', 'max index / max upper bound (S2−S1, S3−early)',
    f"{float(old['decisions46']['max index (S2−S1, S3−early)']):.3f} / {float(old['decisions46']['max upper bound (S2−S1, S3−early)']):.3f}", NA,
    f"{R46['primary']['index'].max():.3f} / {R46['primary'].ci_high.max():.3f}")
add('46', 'species-only counts unskewed', old['decisions46']['species-only counts unskewed (ratio < 2, S2−S1 and S3c−early)'], NA, R46['UNSKEWED'])
old_noint = old['nointegration'][old['nointegration'].contrast.isin(['S2-S1', 'S3-early'])]
for arm, set_name in INTEGRATION_ARMS.items():
    block = R47['INDEX'][R47['INDEX'].arm.eq(arm) & R47['INDEX'].contrast.isin(['S2-S1', 'S3-early'])]
    add('47', f'(a) without {set_name}: claim stands; max index / upper', f"{bool(((old_noint['index'] <= .5) & (old_noint.ci_high < .6)).all())}; "
        f"{old_noint['index'].max():.3f} / {old_noint.ci_high.max():.3f} (611 PCA genes)", NA,
        f"{R47['CLAIM_STANDS'][arm]}; {block['index'].max():.3f} / {block.ci_high.max():.3f}")
for label in ('Slc/Abc/Aqp', 'all genes'):
    o, n = (t[t.genes.eq(label)].iloc[0] for t in (old['transporters'], R47['AGREEMENT']))
    add('47', f'(b′) {label}: direction agreement among zonated in both', f"{int(o.conserved)}/{int(o['zonated in both species'])} = "
        f"{o['direction agreement among zonated in both']:.3f}", NA,
        f"{int(n.conserved)}/{int(n['zonated in both species'])} = {n['direction agreement among zonated in both']:.3f}")
o, n = old['transporter_shares'], R47['TRANSPORTERS']
add('47', '(b) transporters among conserved genes', f"{int(o.conserved.iloc[0])}/{int(o.conserved.iloc[1])}", NA,
    f"{int(n.conserved.iloc[0])}/{int(n.conserved.iloc[1])}")
for klass in ('mouse-only', 'human-only'):
    add('48B', f'probe-balanced naming rule, {klass}', '; '.join(naming_set(old['naming'], klass)) or 'none', NA,
        '; '.join(naming_set(R48['naming'], klass)) or 'none')
for setting in ('(a) swap to main PT paralog', '(b) drop family pairs'):
    o, n = (t[t.setting.eq(setting)].iloc[0] for t in (old['decision64'], R64['decision']))
    add('64', f'{setting}: unaffected (max |Δ index|)', f"{o.unaffected} ({o['max |Δ index| (primary, external)']:.3f})", NA,
        f"{n.unaffected} ({n['max |Δ index| (primary, external)']:.3f})")
for name in expected['human']:
    o, n = old['placement'].loc[name], R44['placement'].loc[name]
    add('44B', f'{name}: percentile among Lake normal donors (within normal range)',
        f"{o['percentile among Lake normal donors']:.2f} ({o['within Lake normal range']})", NA,
        f"{n['percentile among Lake normal donors']:.2f} ({n['within Lake normal range']})")
HEADLINE = pd.DataFrame(rows)
HEADLINE.to_csv(COMPARE / 'compare_old.csv', index=False)
display(HEADLINE)

### The material-change rules (primary arm)

In [ ]:
m_rows = []
h_old, m_old = transfer_value(old['transfer'], 'human', 'Lake', 'agreement with current'), transfer_value(old['transfer'], 'mouse', 'micro', 'agreement with current')
h_new, m_new = transfer_value(R42['TRANSFER'], 'human', 'Lake', 'agreement with current'), transfer_value(R42['TRANSFER'], 'mouse', 'micro', 'agreement with current')
h_sec, m_sec = transfer_value(S42['TRANSFER'], 'human', 'Lake', 'agreement with current'), transfer_value(S42['TRANSFER'], 'mouse', 'micro', 'agreement with current')
m_rows.append({'rule': 'M1', 'what': "42's reading of the human labels changes, or agreement falls by ≥ 0.10 in either species",
               'old': f'support {support(old["transfer"])}; human {h_old:.3f}, mouse {m_old:.3f}',
               'new (primary)': f'support {support(R42["TRANSFER"])}; human {h_new:.3f}, mouse {m_new:.3f}',
               'secondary': f'support {support(S42["TRANSFER"])}; human {h_sec:.3f}, mouse {m_sec:.3f}',
               'material': bool(support(R42['TRANSFER']) != support(old['transfer']) or h_old - h_new >= .10 or m_old - m_new >= .10)})
m2 = {sp: (spearman_glom(R42['DISTANCE'], sp), odds(R42['ATTACHED'], sp, ATTACH_UM_SENSITIVITY)[0]) for sp in ('mouse', 'human')}
m_rows.append({'rule': 'M2', 'what': 'label-order Spearman with glomerulus distance ≤ 0, or S1 odds ratio at 10 µm ≤ 1, in either species',
               'old': '; '.join(f'{sp} ρ {spearman_glom(old["distance"], sp):.2f}, OR {odds(old["attached"], sp, ATTACH_UM_SENSITIVITY)[0]:.2f}'
                                for sp in ('mouse', 'human')),
               'new (primary)': '; '.join(f'{sp} ρ {v[0]:.2f}, OR {v[1]:.2f}' for sp, v in m2.items()), 'secondary': 'not run (library-free)',
               'material': bool(any(v[0] <= 0 or v[1] <= 1 for v in m2.values()))})
deltas = {c: index_value(R43['conservation'], c) - index_value(old['conservation'], c) for c in D2C}
deltas_sec = {c: index_value(S43['conservation'], c) - index_value(old['conservation'], c) for c in D2C}
m_rows.append({'rule': 'M3', 'what': "43's D2 wording changes, or any primary index moves by ≥ 0.10",
               'old': f"{old['decisions43']['D2 wording']}; " + ', '.join(f'{index_value(old["conservation"], c):.3f}' for c in D2C),
               'new (primary)': f"{R43['D2_WORDING']}; " + ', '.join(f'{index_value(R43["conservation"], c):.3f} (Δ {deltas[c]:+.3f})' for c in D2C),
               'secondary': f"{S43['D2_WORDING']}; " + ', '.join(f'Δ {deltas_sec[c]:+.3f}' for c in D2C),
               'material': bool(R43['D2_WORDING'] != old['decisions43']['D2 wording'] or any(abs(d) >= .10 for d in deltas.values()))})
m_rows.append({'rule': 'M4', 'what': "43's D3 decision changes", 'old': old['decisions43']['D3 asymmetry'], 'new (primary)': R43['D3_ASYMMETRY'],
               'secondary': S43['D3_ASYMMETRY'], 'material': bool(R43['D3_ASYMMETRY'] != old['decisions43']['D3 asymmetry'])})
m_rows.append({'rule': 'M5', 'what': "49's rule (i) fails in S2−S1 or S3−early under either bootstrap, or the verdict changes",
               'old': f"rule (i) {rule_i_old}; {old['decisions49']['title verdict']}", 'new (primary)': f"rule (i) {R49['RULE_I']}; {R49['TITLE_VERDICT']}",
               'secondary': f"rule (i, gene) {S49['RULE_I']}; {S49['TITLE_VERDICT']}",
               'material': bool((not R49['RULE_I']) or R49['TITLE_VERDICT'] != old['decisions49']['title verdict'])})
m_rows.append({'rule': 'M6', 'what': 'the claim "largely species-specific" fails in any 46 or 47 (a) evaluation',
               'old': 'stands (46 and 47 a)', 'new (primary)': f"46 {R46['CLAIM_STANDS']}; 47 a {R47['CLAIM_STANDS']}", 'secondary': 'not run',
               'material': bool(not R46['CLAIM_STANDS'] or not all(R47['CLAIM_STANDS'].values()))})
agree = {k: zonated_both_agreement(t) for k, t in (('old', old['class_counts']), ('new', R43['class_counts']), ('sec', S43['class_counts']))}
tr_old = old['transporters'].set_index('genes')['direction agreement among zonated in both']
tr_new = R47['AGREEMENT'].set_index('genes')['direction agreement among zonated in both']
names_old = {k: naming_set(old['naming'], k) for k in ('mouse-only', 'human-only')}
names_new = {k: naming_set(R48['naming'], k) for k in ('mouse-only', 'human-only')}
m_rows.append({'rule': 'M7', 'what': 'direction agreement moves by ≥ 0.10; transporters no longer agree more than all genes; '
                                     'the probe-balanced species-only naming changes',
               'old': f"{agree['old']:.3f}; transporters {tr_old['Slc/Abc/Aqp']:.3f} vs {tr_old['all genes']:.3f}; {names_old}",
               'new (primary)': f"{agree['new']:.3f}; transporters {tr_new['Slc/Abc/Aqp']:.3f} vs {tr_new['all genes']:.3f}; {names_new}",
               'secondary': f"{agree['sec']:.3f}",
               'material': bool(abs(agree['new'] - agree['old']) >= .10 or not tr_new['Slc/Abc/Aqp'] > tr_new['all genes'] or names_new != names_old)})
swap_new = R64['decision'].set_index('setting').loc['(a) swap to main PT paralog']
m_rows.append({'rule': 'M8', 'what': "64's swap verdict becomes affected", 'old': 'unaffected', 'new (primary)': f"unaffected {swap_new.unaffected}: {swap_new.failures}",
               'secondary': 'not run', 'material': bool(not swap_new.unaffected)})
m_rows.append({'rule': 'M9', 'what': 'in 44 B, either human section leaves the Lake normal range',
               'old': ', '.join(f"{n} {old['placement'].loc[n, 'within Lake normal range']}" for n in expected['human']),
               'new (primary)': ', '.join(f"{n} {R44['placement'].loc[n, 'within Lake normal range']}" for n in expected['human']),
               'secondary': 'not run', 'material': bool(not R44['placement']['within Lake normal range'].all())})
MATERIAL = pd.DataFrame(m_rows)
MATERIAL.to_csv(COMPARE / 'material_changes.csv', index=False)
display(MATERIAL)

### Genes, classes and matched units (descriptive)

In [ ]:
old_pt_obs = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
old_pt_obs = old_pt_obs[old_pt_obs.broad_tubule_marker_call.astype(str).eq('PT')]
old_expr = rebuild_pt_expression(old_pt_obs, {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad' for n in names},
                                 orthologs, target_sum=1e4)
old_measured = old_expr.var.measured_in_both_inputs.to_numpy()
old_counts = sparse.csr_matrix(old_expr.layers['counts'], dtype=float)[:, old_measured]
old_genes = old_expr.var_names[old_measured]
old_specimen = old_expr.obs['sample'].astype(str).to_numpy()
old_segment = old_expr.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
old_library = np.asarray(old_counts.sum(axis=1)).ravel()


def pseudobulk_gradients(counts_, genes_, specimen_, segment_, library_):
    """Notebook 43's S2 − S1 and S3 − early gradients (all S3, so no depth is needed), with the ≥ 20-count filter."""
    groups = np.char.add(np.char.add(specimen_.astype(str), '|'), segment_.astype(str))
    labels_, sums = group_sums(counts_, groups)
    frame = pd.DataFrame(sums, index=labels_, columns=genes_)
    lib = pd.Series(library_).groupby(groups).sum()
    out_ = {}
    for contrast, (late, early) in (('S2-S1', (['S2'], ['S1'])), ('S3-early', (['S3'], ['S1', 'S2']))):
        for n in names:
            c_late, c_early = frame.loc[[f'{n}|{s}' for s in late]].sum(), frame.loc[[f'{n}|{s}' for s in early]].sum()
            l_late, l_early = lib[[f'{n}|{s}' for s in late]].sum(), lib[[f'{n}|{s}' for s in early]].sum()
            gradient = pd.Series(log_ratio(c_late, l_late, c_early, l_early), index=genes_)
            out_[(contrast, n)] = gradient.where(pd.concat([c_late, c_early], axis=1).min(axis=1).ge(MIN_COUNTS))
    return pd.DataFrame(out_)


gradients_old = pseudobulk_gradients(old_counts, old_genes, old_specimen, old_segment, old_library)
gene_rows = []
for arm, data in units.items():
    new_gradients = pseudobulk_gradients(data['counts'], data['gene_space'], data['obs']['sample'].astype(str).to_numpy(),
                                         data['obs'].segment.astype(str).to_numpy(), data['library'])
    for contrast in ('S2-S1', 'S3-early'):
        for sp in expected:
            a = gradients_old[[(contrast, n) for n in expected[sp]]].mean(axis=1)
            b = new_gradients[[(contrast, n) for n in expected[sp]]].mean(axis=1)
            common = a.dropna().index.intersection(b.dropna().index)
            gene_rows.append({'arm': arm, 'contrast': contrast, 'species': sp, 'genes in both': len(common),
                              'Spearman old vs new': float(a[common].corr(b[common], method='spearman')),
                              'median gradient, old': float(a[common].median()), 'median gradient, new': float(b[common].median())})
GENES = pd.DataFrame(gene_rows)
GENES.to_csv(COMPARE / 'gene_gradients_old_vs_new.csv', index=False)
display(GENES.round(3))

balanced_both = old['symmetric'].index[old['symmetric'].probe_balanced.astype(bool)].intersection(R43['symmetric'].index[R43['symmetric'].index.isin(balanced)])
CLASS_CROSSTAB = pd.crosstab(old['symmetric'].loc[balanced_both, UNION].fillna('not classified').rename('old'),
                             R43['symmetric'].loc[balanced_both, UNION].fillna('not classified').rename('package (primary)'))
CLASS_CROSSTAB.to_csv(COMPARE / 'union_class_old_vs_new.csv')
display(CLASS_CROSSTAB)

old_label = old_pt_obs.segment_class.astype(str).str.replace('PT-', '', regex=False)
export_obs = ad.read_h5ad(results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad', backed='r').obs[['old_structure_id']].copy()
labels_table = R42['labels_table'].set_index('structure_id')
labels_table['old_label'] = export_obs.old_structure_id.reindex(labels_table.index).map(old_label)
matched = labels_table[labels_table.old_label.notna()]
matched_rows = []
for sp, column, ref in (('human', 'transfer_lake', 'Lake/KPMP'), ('mouse', 'transfer_microdissection', 'microdissection'),
                        ('mouse', 'transfer_male', 'Census male snRNA')):
    block = matched[matched.species.eq(sp)]
    matched_rows.append({'species': sp, 'reference': ref, 'matched units': len(block),
                         'transfer agrees with old reviewed labels': float((block[column] == block.old_label).mean()),
                         'transfer agrees with package labels': float((block[column] == block.current_label).mean()),
                         'old and package labels identical': float((block.old_label == block.current_label).mean())})
MATCHED = pd.DataFrame(matched_rows)
MATCHED.to_csv(COMPARE / 'matched_units_transfer.csv', index=False)
display(MATCHED.round(3))

## 16 · Run record

In [ ]:
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-package-label-comparison.md',
    'package_provenance': {k: provenance.get(k) for k in ('package', 'notebook', 'notebook_logic_version', 'run_folder', 'config_sha256')},
    'confirmed_labels': {k: v for k, v in CONFIRMED.items()},
    'units': {'pt_units': int(len(pt_obs_)), 'per specimen and segment': segment_table.to_dict()},
    'mt_genes_excluded': units['primary']['excluded'],
    'integration_gene_sets': {k: len(v) for k, v in INTEGRATION.items()},
    'notebook62_expression_recomputed': inventory_changes,
    'material_changes': MATERIAL[['rule', 'material']].set_index('rule').material.to_dict(),
    'decisions': {'42 support': R42['SUPPORT'], '43 D2': R43['D2_WORDING'], '43 D3': R43['D3_ASYMMETRY'],
                  '49 rule (i)': R49['RULE_I'], '49 verdict': R49['TITLE_VERDICT'], '46 claim': R46['CLAIM_STANDS'],
                  '47 claim': R47['CLAIM_STANDS'], '48B naming': R48['NAMING_DECISION'],
                  '64 swap unaffected': bool(swap_new.unaffected), '44B within normal range': bool(R44['placement']['within Lake normal range'].all())},
    'elapsed_minutes': round((time.perf_counter() - NOTEBOOK_START) / 60, 1),
}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(json.dumps(record['material_changes'], indent=1))
print(f"Done in {record['elapsed_minutes']} min.")

## 17 · Reading the results (written after the run; numbers from the tables above)

**Code equivalence.** Fed the old inputs, these functions reproduced 47 published tables of notebooks 42, 43, 46,
47, 48 B, 49, 64 and 44 B. Every number matched; three tables of notebook 47 differ only in row-label wording. The
PT-expression rule of the notebook-62 inventory was also reproduced on notebook 62's own units.

**Human PT-S2 is small.** The package labels human PT-S2 for 373 and 259 units, against 908 and 905 for the old
reviewed labels; about half of the old human S2 units are now S1. Every inherited rule was kept; the consequences
are reported here, not corrected.
- **S2 − S1 depends on human S2.**
  - Classified genes fall from 9,013 to 6,269, and the human ÷ mouse amplitude from 0.58 to 0.43.
  - The S2 − S1 union counts fall from 109/37/31/58 to 47/4/3/17 (conserved/reversal/mouse-only/human-only).
  - Human S2 − S1 gradients correlate only 0.63 with the old ones; mouse gradients correlate 0.97, and human
    S3 − early 0.94.
- **Undefined, as before.** The S2 − S1 index below the 80th strength percentile is undefined, as in notebook 49:
  the half-reference human ceiling is not estimable in 2 of the 6 bins.
- **Newly undefined.** S2 − S1 direction agreement is now undefined below the 80th percentile, because no gene
  there is zonated in both species.

**Material-change rules (primary arm, mt-free library).** Seven of the nine rules do not fire.
- **M1–M6 and M9 do not fire.**
  - **Human labels (M1).** Reference support is still weak, the same reading as before: human 0.720 (κ 0.51)
    against 0.712; mouse microdissection 0.851 against 0.868. On the same matched units the transfer agrees with
    the old reviewed labels (0.722) and with the package labels (0.719) about equally.
  - **Anatomy (M2).** Label order still follows glomerulus distance: ρ 0.63 in mouse and 0.24 in human. The S1 odds
    ratio at 10 µm is 4.47 in mouse and 3.49 in human (old 2.02).
  - **Index (M3, M4).** The index barely moves: 0.193, 0.271 and 0.291 against 0.170, 0.278 and 0.289. The D2
    wording ("largely different genes") and the D3 decision ("no consistent asymmetry") are unchanged.
  - **Strength (M5).** Rule (i) holds under both bootstraps and the verdict is unchanged. The top 1% index is
    0.61, 0.56 and 0.61.
  - **Sensitivities (M6, M9).** The claim stands with cross-fitted labels (maximum 0.28, upper bound 0.36). It also
    stands without the pass-1 Harmony features (0.24/0.31) and without those plus the panel genes (0.23/0.30). Our
    donor stays within the Lake normal range (percentile 0.71).
- **M7 fires.** The probe-balanced species-only naming changes.
  - The mouse-only Tyrosine metabolism enrichment (Comt, Fah, Maoa) is no longer called; no mouse-only pathway
    is named.
  - Human-only: Cell Junction Organization (F11r, Jup, Pard6b).
  - Direction agreement among genes zonated in both species is 0.786 (151/192; old 0.723). Transporters still
    agree more: 29/34 = 0.853.
- **M8 fires.** Swapping to the main PT paralogs now moves two small counts beyond the rule: S2 − S1 human-only
  17 → 21 and S3 − early mouse-only 8 → 5. Every index moves by ≤ 0.004.

**Secondary arm (mt in the library).**
- **Indices and part 49 do not depend on the library.** The indices, the transfer and part 49 are identical in
  both arms to three decimals: a library change shifts every gene's gradient by a constant, and correlations
  ignore it. Their changes from the old run therefore come from the units and labels.
- **Class counts do depend on the library.**
  - Union counts: 165/49/13/61 with mt in, against 151/41/27/44 mt-free.
  - The mt-free library moves the human S3 − early median gradient from 0.25 to 0.11, so the absolute effect
    floors call different genes.
  - Direction agreement moves mostly with the units and labels (0.723 → 0.771 with mt in) and slightly with the
    library (→ 0.786 mt-free).
  - The union species-only counts move with both: old 41/76, mt in 13/61, mt-free 27/44 (mouse-only/human-only).
  - Notebook 48 B's naming is not run in this arm, so M7's naming change is not attributed.

**How the protocol was read** (points it did not settle; each takes the reading closest to the source notebooks).
1. **Gene order.** The gene space is ordered as the ortholog map, as `rebuild_pt_expression` gave it to notebooks
   42–64. Notebook 48 B's seeded matched draws depend on gene order.
2. **Library-free parts.** Part 42's geometry, attachment and registration QC do not depend on the library, so
   they run once, in the primary bundle.
3. **Notebook 49 in the secondary arm.** It runs part A with the gene bootstrap: every scheme's curves, rules (i)
   and (ii), the bin classes and direction agreement. The external pair, the mouse-strength diagnostic and part B
   run in the primary arm only.
4. **Notebook 47 (f).** The full-gene transferred labels come from part 42's centred transfers and are evaluated
   with notebook 46's `evaluate`. The equivalence check reproduced the published reversal counts this way.
5. **Expression glomeruli.** The QC units of family Glomerulus (label Podocyte). The 2 px join covers every QC
   unit.
6. **Pass-2 Harmony features.** These come from `pass2.h5ad`'s `highly_variable_for_harmony`; the package writes
   no pass-2 table.
7. **Notebook 64's PT expression.** It is computed on every package PT unit, with no coordinate trim, against the
   native panel total without the mt-encoded genes.
8. **Lake and Census.** Lake is mapped from one pass over all genes, and Census includes notebook 64's paralog
   extension. Notebook 64's guard and the equivalence check show that this equals notebook 43's route.
9. **Notebook 44 B.** The two exports supply "ours" and "mouse-only". The primary universe drops the mt genes.
10. **Gene comparison.** Old gradients are rebuilt from the old inputs for S2 − S1 and S3 − early; S3c needs the
    old depth anchors. A gene enters a species if both specimens pass the ≥ 20-count filter in both runs.
11. **Rule M7's naming.** The rule compares the named pathways; the genes behind them are listed descriptively.
12. **Naming in notebook 47.** Its setting names keep "mouse reviewed"; on this run the mouse labels are the
    package labels.